# DigiHero: Regression analyses

This notebook contains the regression and sensitivity analyses for the DigiHero sample. Sections are organized by outcome (AI use, health-related use, side effects, trust) and by model specification (sociodemographic-only vs. extended models including clinical variables), followed by exploratory mediation and intensity/dose-response analyses

## Notebook roadmap

- Data preparation and missingness checks
- Main regressions (AI use; health-related use; side effects; trust)
- Sensitivity analyses (alternative codings/specifications)
- Exploratory Mediation analyses (where applicable)
- Exploratory Intensity (dose-response) analyses
- Exploratory loneliness models


In [1]:
import sys
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

# -----------------------------
# INPUT FILE
# -----------------------------
DATA_FILE = 'KIDaten_renamed.csv'  # change when needed

# Robust CSV read (tries common separators)
def read_table(path):
    path_l = str(path).lower()
    if path_l.endswith('.csv'):
        attempts = []
        trials = [(',', 'c'), (';', 'c'), ('	', 'c'), ('|', 'c'), (',', 'python'), (';', 'python')]
        for sep, eng in trials:
            try:
                dfx = pd.read_csv(path, sep=sep, engine=eng, low_memory=False)
                if dfx.shape[1] > 1:
                    print(f"Loaded CSV with sep={repr(sep)}, engine={eng}")
                    return dfx
                attempts.append(f"sep={repr(sep)} engine={eng}: only 1 column")
            except Exception as e:
                attempts.append(f"sep={repr(sep)} engine={eng}: {type(e).__name__}")
        raise ValueError("Could not parse CSV. Attempts: " + " | ".join(attempts))
    return pd.read_excel(path)

df = read_table(DATA_FILE)
print(f"Rows: {len(df):,}, Columns: {df.shape[1]:,}")


Loaded CSV with sep=';', engine=c
Rows: 29,571, Columns: 577


## Data preparation and missingness checks

This section prepares the regression-ready dataset and summarizes missingness for outcomes and predictors used across models.


In [2]:
# -----------------------------
# Variable names (survey export)
# -----------------------------
ki_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
gender_col = 'gender_filled'
age_col = 'birth_year_filled'
sozdem_employed_col = 'sozdem_sind_sie_zurzeit_erwerbstaetig'
edu_school_col = 'sozdem_welchen_hoechsten_allgemeinbildenden_schulabschl'
edu_coded_col = 'education_level_coded'

# -----------------------------
# Analysis sample restriction
# -----------------------------
# Keep participants with non-missing age and gender and a valid Yes/No response
# to the AI-use item.
yes_no = ['Ja', 'ja', 'Yes', 'yes', 'Nein', 'nein', 'No', 'no']

n_total = len(df)
miss_gender = df[gender_col].isna()
miss_age = df[age_col].isna()
has_demo = (~miss_gender) & (~miss_age)
other_ki = has_demo & ~df[ki_col].astype(str).str.strip().isin(yes_no)

n_miss_gender = int(miss_gender.sum())
n_miss_age = int(miss_age.sum())
n_other_ki = int(other_ki.sum())

mask_keep = has_demo & df[ki_col].astype(str).str.strip().isin(yes_no)
n_excluded = int(n_total - mask_keep.sum())

print('Exclusions (reasons can overlap):')
print(f'  Missing gender:               {n_miss_gender:>6}')
print(f'  Missing age:                  {n_miss_age:>6}')
print(f'  Non-Yes/No on KI question:    {n_other_ki:>6}')
print(f'  Total excluded:               {n_excluded:>6}')
print(f'  Analysis sample (kept):       {mask_keep.sum():>6}')

# Keep only rows with non-missing gender/age and Yes/No KI response
df_reg = df.loc[mask_keep].copy()

# -----------------------------
# 2) Build regression variables
# -----------------------------
# Outcome: AI use 1=Yes, 0=No (Ja/Yes vs Nein/No)
ai_s = df_reg[ki_col].astype(str).str.strip().str.lower()
df_reg['ai_use'] = np.where(ai_s.isin(['ja', 'yes']), 1, np.where(ai_s.isin(['nein', 'no']), 0, np.nan))

# Age (as in original analysis)
REFERENCE_YEAR = 2026
df_reg['age_2026'] = REFERENCE_YEAR - pd.to_numeric(df_reg[age_col], errors='coerce')

# Gender binary: include both male and female in model via indicator
# male=1, female=0; other categories become missing
gender_s = df_reg[gender_col].astype(str).str.strip().str.lower()
df_reg['male'] = np.where(
    gender_s.isin(['männlich', 'männlich', 'maennlich']),
    1,
    np.where(gender_s.eq('weiblich'), 0, np.nan)
)

# Employment coding
s_emp = df_reg[sozdem_employed_col].astype(str).str.strip()
emp_blank = df_reg[sozdem_employed_col].isna() | s_emp.eq('') | s_emp.str.lower().eq('nan')
emp_yes = s_emp.str.startswith('Ja', na=False) | s_emp.str.startswith('ja', na=False)
emp_no = (
    s_emp.str.startswith('Nein', na=False)
    | s_emp.str.startswith('nein', na=False)
    | s_emp.str.startswith('No', na=False)
    | s_emp.str.startswith('no', na=False)
)
emp_valid = emp_yes | emp_no
emp_other = (~emp_blank) & (~emp_valid)

df_reg['employed'] = np.where(emp_yes, 1, np.where(emp_no, 0, np.nan))

# --- Mapping: Schulabschluss → Bildungsjahre ---
school_years_map = {
    "Allgemeine oder fachgebundene Hochschulreife/Abitur (Gymnasium bzw. Abschluss der Erweiterten Oberschule (der DDR))": 13,
    "Fachhochschulreife, Abschluss einer Fachoberschule": 12,
    "Realschulabschluss (Mittlere Reife)": 10,
    "Polytechnische Oberschule der DDR mit Abschluss der 10. Klasse": 10,
    "Hauptschulabschluss (Volksschulabschluss)": 9,
    "Polytechnische Oberschule der DDR mit Abschluss der 8. oder 9. Klasse": 9,
    "von der Schule abgegangen ohne Hauptschulabschluss (Volksschulabschluss)": 7,
    "Einen anderen Schulabschluss": 10,
    "Schüler:in, besuche eine allgemeinbildende Schule (Vollzeit oder Teilzeit)": np.nan,
    "Weiß nicht": np.nan,
    "Keine Angabe": np.nan,
}

# --- Mapping: Berufsabschluss (education_level_coded) → Zusatzjahre ---
vocational_years_map = {
    "Betriebliche Berufsausbildung (Lehre) abgeschlossen": 1.5,
    "Schulische Berufsausbildung abgeschlossen (Berufsfachschule, Handelsschule. Vorbereitungsdienst für den m": 2,
    "Ausbildung an einer Fach-, Meister-, Technikerschule, Berufs- oder Fachakademie abgeschlossen": 3,
    "Ausbildung an einer Fachschule der DDR abgeschlossen": 3,
    "Bachelor an einer (Fach-) Hochschule abgeschlossen": 5,
    "Fachhochschulabschluss (z.B. Diplom, Master)": 5,
    "Universitätsabschluss (z.B. Diplom, Magister, Staatsexamen, Master)": 5,
    "Promotion": 8,
    "Keinen beruflichen Abschluss und nicht in beruflicher Ausbildung": 0,
    "Noch in beruflicher Ausbildung (Berufsvorbereitungsjahr, Ausbildung, Praktikum, Studium)": 0,
    "Schüler/in und besuche eine berufsorientierte Aufbau-, Fachschule o.ä.": 0,
    "Einen anderen beruflichen Abschluss": np.nan,
    "Weiß nicht": np.nan,
    "Keine Angabe": np.nan,
}

df_reg['edu_school_years'] = df_reg[edu_school_col].map(school_years_map)
voc_str = df_reg[edu_coded_col].astype(str).str.strip()
df_reg['edu_vocational_years'] = voc_str.map(vocational_years_map)
df_reg['edu_total_years'] = np.nan
mask_add = df_reg['edu_school_years'].notna() & (df_reg['edu_school_years'] >= 7)
df_reg.loc[mask_add, 'edu_total_years'] = (
    df_reg.loc[mask_add, 'edu_school_years'] +
    df_reg.loc[mask_add, 'edu_vocational_years']
)

# Save prepared base sample for diagnostics/regressions
df_base_predictors = df_reg.copy()


Exclusions (reasons can overlap):
  Missing gender:                  100
  Missing age:                     127
  Non-Yes/No on KI question:       524
  Total excluded:                  661
  Analysis sample (kept):        28910


In [3]:
df_diag = df_base_predictors.copy()

# -----------------------------
# 2b) Additional exclusion diagnostics requested
# -----------------------------
school_raw = df_diag[edu_school_col].astype(str).str.strip()
voc_raw = df_diag[edu_coded_col].astype(str).str.strip()

school_blank = df_diag[edu_school_col].isna() | school_raw.eq('') | school_raw.str.lower().eq('nan')
voc_blank = df_diag[edu_coded_col].isna() | voc_raw.eq('') | voc_raw.str.lower().eq('nan')

school_mapped = df_diag['edu_school_years'].notna()
voc_mapped = df_diag['edu_vocational_years'].notna()

school_other = (~school_blank) & (~school_mapped)
voc_other = (~voc_blank) & (~voc_mapped)

edu_missing = df_diag['edu_total_years'].isna()
edu_missing_no_answer = edu_missing & (school_blank | voc_blank)
edu_missing_other_values = edu_missing & (~edu_missing_no_answer) & (school_other | voc_other)

# Education capture diagnostics (same spirit as your original education block)
n_edu_total = len(df_diag)
n_school_na = int(df_diag['edu_school_years'].isna().sum())
n_voc_na = int(df_diag['edu_vocational_years'].isna().sum())
n_either_na = int((df_diag['edu_school_years'].isna() | df_diag['edu_vocational_years'].isna()).sum())
n_both_valid = int((df_diag['edu_school_years'].notna() & df_diag['edu_vocational_years'].notna()).sum())

print()
print('Education mapping capture (within kept KI+age+gender sample):')
print(f'  N in sample before complete-case step:        {n_edu_total:>6}')
print(f'  NA in edu_school_years:                        {n_school_na:>6}')
print(f'  NA in edu_vocational_years:                    {n_voc_na:>6}')
print(f'  NA in either school or vocational years:       {n_either_na:>6}')
print(f'  Valid in both (can form edu_total_years):      {n_both_valid:>6}')

print()
print('Additional missingness within kept KI+age+gender sample:')
print(f'  Employment missing (no answer/blank):          {int(emp_blank.sum()):>6}')
print(f'  Employment not in predefined categories:        {int(emp_other.sum()):>6}')
print(f'  Education missing (no answer in school/voc):   {int(edu_missing_no_answer.sum()):>6}')
print(f'  Education not in predefined categories:         {int(edu_missing_other_values.sum()):>6}')

if int(emp_other.sum()) > 0:
    print('  Employment examples not recognized:', s_emp[emp_other].value_counts().head(5).index.tolist())
if int(edu_missing_other_values.sum()) > 0:
    bad_edu = pd.concat([
        school_raw[school_other],
        voc_raw[voc_other],
    ])
    print('  Education examples not recognized:', bad_edu.value_counts().head(5).index.tolist())



Education mapping capture (within kept KI+age+gender sample):
  N in sample before complete-case step:         28910
  NA in edu_school_years:                           434
  NA in edu_vocational_years:                      2602
  NA in either school or vocational years:         2693
  Valid in both (can form edu_total_years):       26217

Additional missingness within kept KI+age+gender sample:
  Employment missing (no answer/blank):             356
  Employment not in predefined categories:             0
  Education missing (no answer in school/voc):      340
  Education not in predefined categories:           2353
  Education examples not recognized: ['Schulische Berufsausbildung abgeschlossen \r\n              (Berufsfachschule, Handelsschule. Vorbereitungsdienst für den m', 'Schulische Berufsausbildung abgeschlossen (Berufsfachschule, Handelsschule, Vorbereitungsdienst für den mittleren Dienst', 'Einen anderen beruflichen Abschluss', 'Keine Angabe', 'Schüler:in, besuche eine allg

## Main models: AI use (binary)

### Predictors: sociodemographics


In [4]:
df_reg = df_base_predictors.copy()

# -----------------------------
# 3) Strict complete-case regression sample
# -----------------------------
required = ['ai_use', 'age_2026', 'male', 'edu_total_years', 'employed']

# Diagnostic: cumulative row loss per variable
print('Diagnostic: row loss for AI-use regression')
print(f'  Starting N (base predictors):   {len(df_reg):>6}')
remaining = df_reg.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f'  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})')
print(f'  Final complete cases:           {len(remaining):>6}')
print()

df_reg = df_reg.dropna(subset=required).copy()

df_reg['ai_use'] = df_reg['ai_use'].astype(int)
df_reg['male'] = df_reg['male'].astype(int)
df_reg['employed'] = df_reg['employed'].astype(int)
df_reg['age_2026'] = pd.to_numeric(df_reg['age_2026'], errors='coerce')
df_reg['edu_total_years'] = pd.to_numeric(df_reg['edu_total_years'], errors='coerce')
df_reg = df_reg.dropna(subset=['age_2026', 'edu_total_years'])

print(f'  Analysis N (final complete cases): {len(df_reg):,}')

# -----------------------------
# 4) Logistic regression
# -----------------------------
formula = 'ai_use ~ age_2026 + male + edu_total_years + employed'
print()
print(f'Model: {formula}')

if len(df_reg) < 10:
    print('Too few complete cases to fit model safely.')
elif df_reg['ai_use'].nunique() < 2:
    print('Outcome has only one class after filtering; model not fit.')
else:
    preds = ['age_2026', 'male', 'edu_total_years', 'employed']
    keep_preds = [p for p in preds if df_reg[p].nunique(dropna=True) > 1]
    dropped = [p for p in preds if p not in keep_preds]
    if dropped:
        print('Dropped constant predictor(s): ' + ', '.join(dropped))
    if not keep_preds:
        print('All predictors are constant; model not fit.')
    else:
        formula_fit = 'ai_use ~ ' + ' + '.join(keep_preds)
        result = smf.logit(formula_fit, data=df_reg).fit(disp=0)
        print(result.summary())

        coef = result.params
        conf = result.conf_int()
        conf.columns = ['CI_low', 'CI_high']
        report = pd.DataFrame({
            'Coef': coef,
            'OR': np.exp(coef),
            'OR_95%_CI_low': np.exp(conf['CI_low']),
            'OR_95%_CI_high': np.exp(conf['CI_high']),
            'p': result.pvalues,
        })
        print()
        print('Odds ratios and 95% CI:')
        print(report.round(4).to_string())

Diagnostic: row loss for AI-use regression
  Starting N (base predictors):    28910
  ai_use                    valid:  28910  missing:      0  (lost      0)
  age_2026                  valid:  28910  missing:      0  (lost      0)
  male                      valid:  28836  missing:     74  (lost     74)
  edu_total_years           valid:  26152  missing:   2684  (lost   2684)
  employed                  valid:  26097  missing:     55  (lost     55)
  Final complete cases:            26097

  Analysis N (final complete cases): 26,097

Model: ai_use ~ age_2026 + male + edu_total_years + employed
                           Logit Regression Results                           
Dep. Variable:                 ai_use   No. Observations:                26097
Model:                          Logit   Df Residuals:                    26092
Method:                           MLE   Df Model:                            4
Date:                Wed, 15 Apr 2026   Pseudo R-squ.:                  0.1031
Tim

## Main models: health-related AI use among AI users

### Outcome: General health-related use (binary)
### Predictors: sociodemographics


In [5]:
# -----------------------------
# 5) Logistic regression among AI users: medical use (use-case columns 1-3)
# Uses same participant inclusion mechanism base as AI-use model:
# - non-missing age/gender
# - KI question in Yes/No
# Then restricted to AI users only.
# -----------------------------

purpose_cols_13 = [
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3',
]
purpose_cols_13 = [c for c in purpose_cols_13 if c in df_base_predictors.columns]

non_medical_answers = [
    'hierfür nutze ich keine ki',
    '(bisher) kein bedarf, würde ki aber dafür nutzen',
]

# Start from same base inclusion as AI-use model, then AI users only
df_use_ki = df_base_predictors.loc[df_base_predictors['ai_use'] == 1].copy()

# med_use coding requested:
# 1 = at least one of the 3 items is a substantive (not non-medical) answer
# 0 = answered items are only non-medical answers
# NaN = no answer in all 3 items

def norm_str(x):
    if pd.isna(x):
        return ''
    v = str(x).strip()
    if v.lower() == 'nan':
        return ''
    return v.lower()

non_med_set = {x.lower().strip() for x in non_medical_answers}

med_use_vals = []
for _, row in df_use_ki[purpose_cols_13].iterrows():
    vals = [norm_str(row[c]) for c in purpose_cols_13]
    answered = [v for v in vals if v != '']

    if len(answered) == 0:
        med_use_vals.append(np.nan)
    elif any(v not in non_med_set for v in answered):
        med_use_vals.append(1)
    else:
        med_use_vals.append(0)

df_use_ki['med_use'] = med_use_vals

cols_needed_med = ['med_use', 'age_2026', 'male', 'edu_total_years', 'employed']
df_reg_med = df_use_ki.dropna(subset=cols_needed_med).copy()

# Ensure numeric dtypes
df_reg_med['med_use'] = pd.to_numeric(df_reg_med['med_use'], errors='coerce')
df_reg_med['age_2026'] = pd.to_numeric(df_reg_med['age_2026'], errors='coerce')
df_reg_med['male'] = pd.to_numeric(df_reg_med['male'], errors='coerce')
df_reg_med['edu_total_years'] = pd.to_numeric(df_reg_med['edu_total_years'], errors='coerce')
df_reg_med['employed'] = pd.to_numeric(df_reg_med['employed'], errors='coerce')
df_reg_med = df_reg_med.dropna(subset=cols_needed_med)

df_reg_med['med_use'] = df_reg_med['med_use'].astype(int)
df_reg_med['male'] = df_reg_med['male'].astype(int)
df_reg_med['employed'] = df_reg_med['employed'].astype(int)

print()
print('Logistic regression among AI users: med_use ~ age_2026 + male + edu_total_years + employed')
print('  (Same base inclusion as AI-use model; then AI users only; med_use missing if all 3 use-items are blank.)')
print(f'  N: {len(df_reg_med)}')

if len(df_reg_med) < 10:
    print('  Too few complete cases to fit model safely.')
elif df_reg_med['med_use'].nunique() < 2:
    print('  med_use has only one class; model not fit.')
else:
    n_med = int(df_reg_med['med_use'].sum())
    n_non = int((1 - df_reg_med['med_use']).sum())
    print(f'  med_use: {n_med} use, {n_non} non-use')

    preds_med = ['age_2026', 'male', 'edu_total_years', 'employed']
    keep_med = [p for p in preds_med if df_reg_med[p].nunique(dropna=True) > 1]
    drop_med = [p for p in preds_med if p not in keep_med]
    if drop_med:
        print('  Dropped constant predictor(s): ' + ', '.join(drop_med))

    if not keep_med:
        print('  All predictors are constant; model not fit.')
    else:
        formula_med = 'med_use ~ ' + ' + '.join(keep_med)
        print(f'  Model: {formula_med}')
        result_med = smf.logit(formula_med, data=df_reg_med).fit(disp=0)
        print(result_med.summary())

        coef_m = result_med.params
        conf_m = result_med.conf_int()
        conf_m.columns = ['CI_low', 'CI_high']
        report_m = pd.DataFrame({
            'Coef': coef_m,
            'OR': np.exp(coef_m),
            'OR_95%_CI_low': np.exp(conf_m['CI_low']),
            'OR_95%_CI_high': np.exp(conf_m['CI_high']),
            'p': result_med.pvalues,
        })
        print()
        print('Odds ratios and 95% CI (med_use model):')
        print(report_m.round(4).to_string())



Logistic regression among AI users: med_use ~ age_2026 + male + edu_total_years + employed
  (Same base inclusion as AI-use model; then AI users only; med_use missing if all 3 use-items are blank.)
  N: 13653
  med_use: 7854 use, 5799 non-use
  Model: med_use ~ age_2026 + male + edu_total_years + employed
                           Logit Regression Results                           
Dep. Variable:                med_use   No. Observations:                13653
Model:                          Logit   Df Residuals:                    13648
Method:                           MLE   Df Model:                            4
Date:                Wed, 15 Apr 2026   Pseudo R-squ.:                 0.01346
Time:                        21:28:13   Log-Likelihood:                -9183.0
converged:                       True   LL-Null:                       -9308.3
Covariance Type:            nonrobust   LLR p-value:                 4.845e-53
                      coef    std err          z      P>|z| 

### Outcome: Mental health-related use (binary)
### Predictors: sociodemographics


In [6]:
# -----------------------------
# Logistic regression among AI users: psychological/emotional use
# Run TWICE with same predictors:
#   (A) emo_use_excl13: Q4.4–Q4.12 only
#   (B) emo_use_incl13: Q4.4–Q4.13 (incl. Sonstiges)
#
# Predictors stay EXACTLY the same as your original code:
#   age_2026 + male + edu_total_years + employed
# -----------------------------

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# -----------------------------
# 1) Purpose columns
# -----------------------------
purpose_cols_4_12 = [
    'ki_bei_aengsten_und_sorgen',
    'ki_bei_selbstzweifeln_oder_niedergeschlagenheit',
    'ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we',
    'ki_bei_zwischenmenschlichen_konflikten',
    'ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan',
    'ki_bei_trauer_oder_verlust',
    'ki_wenn_grosse_lebensveraenderungen_anstehen',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5',
]
col_13 = 'ki_fuer_sonstige_gesundheits_psychische_oder_emotionale'

purpose_cols_4_12 = [c for c in purpose_cols_4_12 if c in df_base_predictors.columns]
purpose_cols_4_13 = purpose_cols_4_12 + ([col_13] if col_13 in df_base_predictors.columns else [])

if len(purpose_cols_4_12) == 0:
    raise KeyError("No emotional use columns (Q4.4–Q4.12) found in df_base_predictors.")
if col_13 not in df_base_predictors.columns:
    print("Warning: Q4.13 column not found; incl13 model cannot differ from excl13 model.")

non_use_answers = [
    'hierfür nutze ich keine ki',
    '(bisher) kein bedarf, würde ki aber dafür nutzen',
]
non_use_set = {x.lower().strip() for x in non_use_answers}

# Start from same base inclusion as AI-use model, then AI users only
df_use_ki_emo = df_base_predictors.loc[df_base_predictors['ai_use'] == 1].copy()

# -----------------------------
# 2) emo_use coding helper
# -----------------------------
def norm_str(x):
    if pd.isna(x):
        return ''
    v = str(x).strip()
    if v.lower() == 'nan':
        return ''
    return v.lower()

def build_emo_use(df_in, cols):
    vals_out = []
    for _, row in df_in[cols].iterrows():
        vals = [norm_str(row[c]) for c in cols]
        answered = [v for v in vals if v != '']

        if len(answered) == 0:
            vals_out.append(np.nan)
        elif any(v not in non_use_set for v in answered):
            vals_out.append(1)
        else:
            vals_out.append(0)
    return vals_out

# Build both outcomes
df_use_ki_emo['emo_use_excl13'] = build_emo_use(df_use_ki_emo, purpose_cols_4_12)
df_use_ki_emo['emo_use_incl13'] = build_emo_use(df_use_ki_emo, purpose_cols_4_13)

# -----------------------------
# 3) Regression runner (same predictors as your original model)
# -----------------------------
def run_emo_model(df_in, outcome_col, label):
    cols_needed = [outcome_col, 'age_2026', 'male', 'edu_total_years', 'employed']
    dfx = df_in.dropna(subset=cols_needed).copy()

    for c in cols_needed:
        dfx[c] = pd.to_numeric(dfx[c], errors='coerce')
    dfx = dfx.dropna(subset=cols_needed)

    dfx[outcome_col] = dfx[outcome_col].astype(int)
    dfx['male'] = dfx['male'].astype(int)
    dfx['employed'] = dfx['employed'].astype(int)

    print()
    print("=" * 90)
    print(label)
    print('Logistic regression among AI users: '
          f'{outcome_col} ~ age_2026 + male + edu_total_years + employed')
    print('  (Same base inclusion as AI-use model; then AI users only; '
          f'{outcome_col} missing if all relevant use-items are blank.)')
    print(f'  N: {len(dfx)}')

    if len(dfx) < 10:
        print('  Too few complete cases to fit model safely.')
        return None, dfx
    if dfx[outcome_col].nunique() < 2:
        print(f'  {outcome_col} has only one class; model not fit.')
        return None, dfx

    n_use = int(dfx[outcome_col].sum())
    n_non = int((1 - dfx[outcome_col]).sum())
    print(f'  {outcome_col}: {n_use} use, {n_non} non-use')

    preds = ['age_2026', 'male', 'edu_total_years', 'employed']
    keep = [p for p in preds if dfx[p].nunique(dropna=True) > 1]
    dropped = [p for p in preds if p not in keep]
    if dropped:
        print('  Dropped constant predictor(s): ' + ', '.join(dropped))

    if not keep:
        print('  All predictors are constant; model not fit.')
        return None, dfx

    formula = f'{outcome_col} ~ ' + ' + '.join(keep)
    print(f'  Model: {formula}')
    result = smf.logit(formula, data=dfx).fit(disp=0)
    print(result.summary())

    coef = result.params
    conf = result.conf_int()
    conf.columns = ['CI_low', 'CI_high']
    report = pd.DataFrame({
        'Coef': coef,
        'OR': np.exp(coef),
        'OR_95%_CI_low': np.exp(conf['CI_low']),
        'OR_95%_CI_high': np.exp(conf['CI_high']),
        'p': result.pvalues,
    })

    print()
    print(f'Odds ratios and 95% CI ({outcome_col} model):')
    print(report.round(4).to_string())

    return result, dfx

# -----------------------------
# 4) Run both models
# -----------------------------
res_excl, df_excl = run_emo_model(
    df_use_ki_emo,
    outcome_col='emo_use_excl13',
    label='MODEL A: EXCLUDING Q4.13 (only Q4.4–Q4.12)'
)

res_incl, df_incl = run_emo_model(
    df_use_ki_emo,
    outcome_col='emo_use_incl13',
    label='MODEL B: INCLUDING Q4.13 (Q4.4–Q4.13)'
)

# -----------------------------
# 5) Compact comparison table
# -----------------------------
if res_excl is not None and res_incl is not None:
    comp_rows = []
    for p in ['age_2026', 'male', 'edu_total_years', 'employed']:
        b_ex = res_excl.params.get(p, np.nan)
        b_in = res_incl.params.get(p, np.nan)
        comp_rows.append({
            'predictor': p,
            'coef_excl13': b_ex,
            'OR_excl13': np.exp(b_ex),
            'p_excl13': res_excl.pvalues.get(p, np.nan),
            'coef_incl13': b_in,
            'OR_incl13': np.exp(b_in),
            'p_incl13': res_incl.pvalues.get(p, np.nan),
            'delta_coef_incl_minus_excl': b_in - b_ex
        })
    comp = pd.DataFrame(comp_rows)

    print()
    print("=" * 90)
    print("COMPARISON: WITH vs WITHOUT Q4.13 (same predictors)")
    print("=" * 90)
    print(comp.round(4).to_string(index=False))
    print(f"\nN excl13: {len(df_excl):,} | N incl13: {len(df_incl):,}")


MODEL A: EXCLUDING Q4.13 (only Q4.4–Q4.12)
Logistic regression among AI users: emo_use_excl13 ~ age_2026 + male + edu_total_years + employed
  (Same base inclusion as AI-use model; then AI users only; emo_use_excl13 missing if all relevant use-items are blank.)
  N: 13652
  emo_use_excl13: 3889 use, 9763 non-use
  Model: emo_use_excl13 ~ age_2026 + male + edu_total_years + employed
                           Logit Regression Results                           
Dep. Variable:         emo_use_excl13   No. Observations:                13652
Model:                          Logit   Df Residuals:                    13647
Method:                           MLE   Df Model:                            4
Date:                Wed, 15 Apr 2026   Pseudo R-squ.:                 0.04273
Time:                        21:28:18   Log-Likelihood:                -7808.4
converged:                       True   LL-Null:                       -8156.9
Covariance Type:            nonrobust   LLR p-value:         

### Sensitivity model: Mental health-related use with categorical education

This specification replaces the continuous education metric with a categorical education variable.


In [7]:
# -----------------------------
# Logistic regression among AI users: psychological/emotional use
# Run TWICE with same predictors:
#   (A) emo_use_excl13: Q4.4–Q4.12 only
#   (B) emo_use_incl13: Q4.4–Q4.13 (incl. Sonstiges)
#
# Predictors (same structure as original, but education recoded):
#   age_2026 + male + edu5 + employed
# where edu5 is an ordinal categorical with an explicit "unknown" level,
# and the reference level is set to "medium".
# -----------------------------

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# -----------------------------
# 0) Build edu5 on df_base_predictors (Option A + unknown)
# -----------------------------
edu_school_col = "sozdem_welchen_hoechsten_allgemeinbildenden_schulabschl"
edu_coded_col  = "education_level_coded"
UNKNOWN_TOKENS = {"Weiß nicht", "Keine Angabe"}

def _norm(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return None
    return s

def _school_level(val):
    s = _norm(val)
    if s is None or s in UNKNOWN_TOKENS:
        return None
    if s == "Schüler:in, besuche eine allgemeinbildende Schule (Vollzeit oder Teilzeit)":
        return None

    low = {
        "von der Schule abgegangen ohne Hauptschulabschluss (Volksschulabschluss)",
        "Hauptschulabschluss (Volksschulabschluss)",
        "Polytechnische Oberschule der DDR mit Abschluss der 8. oder 9. Klasse",
    }
    medium = {
        "Realschulabschluss (Mittlere Reife)",
        "Polytechnische Oberschule der DDR mit Abschluss der 10. Klasse",
    }
    high = {
        "Fachhochschulreife, Abschluss einer Fachoberschule",
        "Allgemeine oder fachgebundene Hochschulreife/Abitur (Gymnasium bzw. Abschluss der Erweiterten Oberschule (der DDR))",
    }

    if s in low:
        return 0
    if s in medium:
        return 1
    if s in high:
        return 2
    return None

def _voc_level(val):
    s = _norm(val)
    if s is None or s in UNKNOWN_TOKENS:
        return None

    in_training = {
        "Noch in beruflicher Ausbildung (Berufsvorbereitungsjahr, Ausbildung, Praktikum, Studium)",
        "Schüler/in und besuche eine berufsorientierte Aufbau-, Fachschule o.ä.",
    }
    none_or_basic = {
        "Keinen beruflichen Abschluss und nicht in beruflicher Ausbildung",
        "Betriebliche Berufsausbildung (Lehre) abgeschlossen",
        "Schulische Berufsausbildung abgeschlossen (Berufsfachschule, Handelsschule. Vorbereitungsdienst für den m",
    }
    adv_or_tertiary = {
        "Ausbildung an einer Fach-, Meister-, Technikerschule, Berufs- oder Fachakademie abgeschlossen",
        "Ausbildung an einer Fachschule der DDR abgeschlossen",
        "Bachelor an einer (Fach-) Hochschule abgeschlossen",
        "Fachhochschulabschluss (z.B. Diplom, Master)",
        "Universitätsabschluss (z.B. Diplom, Magister, Staatsexamen, Master)",
    }

    if s in in_training:
        return "in_training"
    if s in none_or_basic:
        return 0
    if s in adv_or_tertiary:
        return 1
    if s == "Promotion":
        return 2
    return None

def edu5_category(school_val, voc_val):
    s_lvl = _school_level(school_val)
    v_lvl = _voc_level(voc_val)

    if v_lvl == 2:
        return "doctorate"
    if v_lvl == 1:
        return "tertiary_adv"

    if v_lvl == "in_training":
        if s_lvl is None:
            return "unknown"
        return "low" if s_lvl == 0 else ("medium" if s_lvl == 1 else "high")

    if s_lvl is not None:
        if s_lvl == 0:
            return "medium" if v_lvl == 0 else "low"
        if s_lvl == 1:
            return "medium"
        if s_lvl == 2:
            return "high"

    if v_lvl == 0:
        return "medium"

    return "unknown"

edu5_order = ["low", "medium", "high", "tertiary_adv", "doctorate", "unknown"]

df_base_predictors = df_base_predictors.copy()
df_base_predictors["edu5"] = [
    edu5_category(s, v)
    for s, v in zip(df_base_predictors.get(edu_school_col), df_base_predictors.get(edu_coded_col))
]
df_base_predictors["edu5"] = pd.Categorical(df_base_predictors["edu5"], categories=edu5_order, ordered=True)

# -----------------------------
# 1) Purpose columns
# -----------------------------
purpose_cols_4_12 = [
    'ki_bei_aengsten_und_sorgen',
    'ki_bei_selbstzweifeln_oder_niedergeschlagenheit',
    'ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we',
    'ki_bei_zwischenmenschlichen_konflikten',
    'ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan',
    'ki_bei_trauer_oder_verlust',
    'ki_wenn_grosse_lebensveraenderungen_anstehen',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5',
]
col_13 = 'ki_fuer_sonstige_gesundheits_psychische_oder_emotionale'

purpose_cols_4_12 = [c for c in purpose_cols_4_12 if c in df_base_predictors.columns]
purpose_cols_4_13 = purpose_cols_4_12 + ([col_13] if col_13 in df_base_predictors.columns else [])

if len(purpose_cols_4_12) == 0:
    raise KeyError("No emotional use columns (Q4.4–Q4.12) found in df_base_predictors.")
if col_13 not in df_base_predictors.columns:
    print("Warning: Q4.13 column not found; incl13 model cannot differ from excl13 model.")

non_use_answers = [
    'hierfür nutze ich keine ki',
    '(bisher) kein bedarf, würde ki aber dafür nutzen',
]
non_use_set = {x.lower().strip() for x in non_use_answers}

# Start from same base inclusion as AI-use model, then AI users only
df_use_ki_emo = df_base_predictors.loc[df_base_predictors['ai_use'] == 1].copy()

# -----------------------------
# 2) emo_use coding helper
# -----------------------------
def norm_str(x):
    if pd.isna(x):
        return ''
    v = str(x).strip()
    if v.lower() == 'nan':
        return ''
    return v.lower()

def build_emo_use(df_in, cols):
    vals_out = []
    for _, row in df_in[cols].iterrows():
        vals = [norm_str(row[c]) for c in cols]
        answered = [v for v in vals if v != '']

        if len(answered) == 0:
            vals_out.append(np.nan)
        elif any(v not in non_use_set for v in answered):
            vals_out.append(1)
        else:
            vals_out.append(0)
    return vals_out

# Build both outcomes
df_use_ki_emo['emo_use_excl13'] = build_emo_use(df_use_ki_emo, purpose_cols_4_12)
df_use_ki_emo['emo_use_incl13'] = build_emo_use(df_use_ki_emo, purpose_cols_4_13)

# -----------------------------
# 3) Regression runner (same predictors, education updated)
# -----------------------------
EDU_TERM = "C(edu5, Treatment(reference='medium'))"

def run_emo_model(df_in, outcome_col, label):
    cols_needed = [outcome_col, 'age_2026', 'male', 'edu5', 'employed']
    dfx = df_in.dropna(subset=cols_needed).copy()

    # numeric conversions (NOT edu5)
    for c in [outcome_col, 'age_2026', 'male', 'employed']:
        dfx[c] = pd.to_numeric(dfx[c], errors='coerce')
    dfx = dfx.dropna(subset=cols_needed)

    dfx[outcome_col] = dfx[outcome_col].astype(int)
    dfx['male'] = dfx['male'].astype(int)
    dfx['employed'] = dfx['employed'].astype(int)

    print()
    print("=" * 90)
    print(label)
    print('Logistic regression among AI users: '
          f'{outcome_col} ~ age_2026 + male + edu5(ref=medium) + employed')
    print('  (Same base inclusion as AI-use model; then AI users only; '
          f'{outcome_col} missing if all relevant use-items are blank.)')
    print(f'  N: {len(dfx)}')

    if len(dfx) < 10:
        print('  Too few complete cases to fit model safely.')
        return None, dfx
    if dfx[outcome_col].nunique() < 2:
        print(f'  {outcome_col} has only one class; model not fit.')
        return None, dfx

    n_use = int(dfx[outcome_col].sum())
    n_non = int((1 - dfx[outcome_col]).sum())
    print(f'  {outcome_col}: {n_use} use, {n_non} non-use')

    # Only drop constant numeric predictors; always keep edu term
    preds_num = ['age_2026', 'male', 'employed']
    keep_num = [p for p in preds_num if dfx[p].nunique(dropna=True) > 1]
    dropped = [p for p in preds_num if p not in keep_num]
    if dropped:
        print('  Dropped constant numeric predictor(s): ' + ', '.join(dropped))

    rhs = keep_num + [EDU_TERM]
    formula = f'{outcome_col} ~ ' + ' + '.join(rhs)
    print(f'  Model: {formula}')
    result = smf.logit(formula, data=dfx).fit(disp=0)
    print(result.summary())

    coef = result.params
    conf = result.conf_int()
    conf.columns = ['CI_low', 'CI_high']
    report = pd.DataFrame({
        'Coef': coef,
        'OR': np.exp(coef),
        'OR_95%_CI_low': np.exp(conf['CI_low']),
        'OR_95%_CI_high': np.exp(conf['CI_high']),
        'p': result.pvalues,
    })

    print()
    print(f'Odds ratios and 95% CI ({outcome_col} model):')
    print(report.round(4).to_string())

    return result, dfx

# -----------------------------
# 4) Run both models
# -----------------------------
res_excl, df_excl = run_emo_model(
    df_use_ki_emo,
    outcome_col='emo_use_excl13',
    label='MODEL A: EXCLUDING Q4.13 (only Q4.4–Q4.12)'
)

res_incl, df_incl = run_emo_model(
    df_use_ki_emo,
    outcome_col='emo_use_incl13',
    label='MODEL B: INCLUDING Q4.13 (Q4.4–Q4.13)'
)

# -----------------------------
# 5) Compact comparison table (updated for edu5)
# -----------------------------
if res_excl is not None and res_incl is not None:
    comp_rows = []
    # numeric predictors (same idea as before)
    for p in ['age_2026', 'male', 'employed']:
        b_ex = res_excl.params.get(p, np.nan)
        b_in = res_incl.params.get(p, np.nan)
        comp_rows.append({
            'predictor': p,
            'coef_excl13': b_ex,
            'OR_excl13': np.exp(b_ex),
            'p_excl13': res_excl.pvalues.get(p, np.nan),
            'coef_incl13': b_in,
            'OR_incl13': np.exp(b_in),
            'p_incl13': res_incl.pvalues.get(p, np.nan),
            'delta_coef_incl_minus_excl': b_in - b_ex
        })

    # education contrasts: include all edu5 dummy terms present in BOTH models
    edu_terms = sorted({k for k in res_excl.params.index if k.startswith("C(edu5")})
    for k in edu_terms:
        b_ex = res_excl.params.get(k, np.nan)
        b_in = res_incl.params.get(k, np.nan)
        comp_rows.append({
            'predictor': k,
            'coef_excl13': b_ex,
            'OR_excl13': np.exp(b_ex) if pd.notna(b_ex) else np.nan,
            'p_excl13': res_excl.pvalues.get(k, np.nan),
            'coef_incl13': b_in,
            'OR_incl13': np.exp(b_in) if pd.notna(b_in) else np.nan,
            'p_incl13': res_incl.pvalues.get(k, np.nan),
            'delta_coef_incl_minus_excl': (b_in - b_ex) if (pd.notna(b_in) and pd.notna(b_ex)) else np.nan
        })

    comp = pd.DataFrame(comp_rows)

    print()
    print("=" * 90)
    print("COMPARISON: WITH vs WITHOUT Q4.13 (same predictors)")
    print("=" * 90)
    print(comp.round(4).to_string(index=False))
    print(f"\nN excl13: {len(df_excl):,} | N incl13: {len(df_incl):,}")


MODEL A: EXCLUDING Q4.13 (only Q4.4–Q4.12)
Logistic regression among AI users: emo_use_excl13 ~ age_2026 + male + edu5(ref=medium) + employed
  (Same base inclusion as AI-use model; then AI users only; emo_use_excl13 missing if all relevant use-items are blank.)
  N: 14821
  emo_use_excl13: 4243 use, 10578 non-use
  Model: emo_use_excl13 ~ age_2026 + male + employed + C(edu5, Treatment(reference='medium'))
                           Logit Regression Results                           
Dep. Variable:         emo_use_excl13   No. Observations:                14821
Model:                          Logit   Df Residuals:                    14812
Method:                           MLE   Df Model:                            8
Date:                Wed, 15 Apr 2026   Pseudo R-squ.:                 0.04510
Time:                        21:28:26   Log-Likelihood:                -8474.4
converged:                       True   LL-Null:                       -8874.7
Covariance Type:            nonrobus

### Extended model: AI use with sociodemographic and clinical predictors


In [8]:
df_reg = df_base_predictors.copy()

# -----------------------------
# Build additional predictors
# -----------------------------

# PHQ-4 sum score (recode 1-4 -> 0-3; only when all 4 items are non-missing/valid)
phq4_cols = ['PHQ4_1', 'PHQ4_2', 'PHQ4_3', 'PHQ4_4']
phq4_raw = df_reg[phq4_cols].apply(pd.to_numeric, errors='coerce')
phq4_recoded = phq4_raw.replace({1: 0, 2: 1, 3: 2, 4: 3})
phq4_recoded = phq4_recoded.where(phq4_raw.isin([1, 2, 3, 4]), np.nan)

mask_phq4_complete = phq4_recoded.notna().all(axis=1)
df_reg['PHQ4_sum'] = phq4_recoded.sum(axis=1)
df_reg.loc[~mask_phq4_complete, 'PHQ4_sum'] = np.nan

# Behandlung_dich (current treatment): a=1 (Ja), b=0 (Nein), else NaN
bh = df_reg['Behandlung_dich'].astype(str).str.strip().str.lower()
df_reg['behandlung'] = np.where(bh == 'a', 1, np.where(bh == 'b', 0, np.nan))

# Diagnose_selbst_dich (self-reported diagnosis): a=1 (Ja), b=0 (Nein), c/else = NaN
ds = df_reg['Diagnose_selbst_dich'].astype(str).str.strip().str.lower()
df_reg['diagnose_selbst'] = np.where(ds == 'a', 1, np.where(ds == 'b', 0, np.nan))

# EQ5D5L_3 (daily activities): ordinal a=1 ... e=5, else NaN
eq_map = {'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5}
eq_raw = df_reg['EQ5D5L_3'].astype(str).str.strip().str.lower()
df_reg['eq5d_daily'] = eq_raw.map(eq_map)

# -----------------------------
# Complete-case regression sample
# -----------------------------
required = [
    'ai_use', 'age_2026', 'male', 'edu_total_years', 'employed',
    'PHQ4_sum', 'behandlung', 'diagnose_selbst', 'eq5d_daily',
]

# Diagnostic: cumulative row loss per variable
print('Diagnostic: row loss for extended AI-use regression')
print(f'  Starting N (base predictors):   {len(df_reg):>6}')
remaining = df_reg.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f'  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})')
print(f'  Final complete cases:           {len(remaining):>6}')
print()

df_reg = df_reg.dropna(subset=required).copy()

df_reg['ai_use'] = df_reg['ai_use'].astype(int)
df_reg['male'] = df_reg['male'].astype(int)
df_reg['employed'] = df_reg['employed'].astype(int)
df_reg['behandlung'] = df_reg['behandlung'].astype(int)
df_reg['diagnose_selbst'] = df_reg['diagnose_selbst'].astype(int)
df_reg['age_2026'] = pd.to_numeric(df_reg['age_2026'], errors='coerce')
df_reg['edu_total_years'] = pd.to_numeric(df_reg['edu_total_years'], errors='coerce')
df_reg['PHQ4_sum'] = pd.to_numeric(df_reg['PHQ4_sum'], errors='coerce')
df_reg['eq5d_daily'] = pd.to_numeric(df_reg['eq5d_daily'], errors='coerce')
df_reg = df_reg.dropna(subset=required)

print(f'  Analysis N (final complete cases): {len(df_reg):,}')

# -----------------------------
# Logistic regression
# -----------------------------
preds = ['age_2026', 'male', 'edu_total_years', 'employed',
         'PHQ4_sum', 'behandlung', 'diagnose_selbst', 'eq5d_daily']
formula = 'ai_use ~ ' + ' + '.join(preds)
print()
print(f'Model: {formula}')

if len(df_reg) < 10:
    print('Too few complete cases to fit model safely.')
elif df_reg['ai_use'].nunique() < 2:
    print('Outcome has only one class after filtering; model not fit.')
else:
    keep_preds = [p for p in preds if df_reg[p].nunique(dropna=True) > 1]
    dropped = [p for p in preds if p not in keep_preds]
    if dropped:
        print('Dropped constant predictor(s): ' + ', '.join(dropped))
    if not keep_preds:
        print('All predictors are constant; model not fit.')
    else:
        formula_fit = 'ai_use ~ ' + ' + '.join(keep_preds)
        result = smf.logit(formula_fit, data=df_reg).fit(disp=0)
        print(result.summary())

        coef = result.params
        conf = result.conf_int()
        conf.columns = ['CI_low', 'CI_high']
        report = pd.DataFrame({
            'Coef': coef,
            'OR': np.exp(coef),
            'OR_95%_CI_low': np.exp(conf['CI_low']),
            'OR_95%_CI_high': np.exp(conf['CI_high']),
            'p': result.pvalues,
        })
        print()
        print('Odds ratios and 95% CI:')
        print(report.round(4).to_string())

Diagnostic: row loss for extended AI-use regression
  Starting N (base predictors):    28910
  ai_use                    valid:  28910  missing:      0  (lost      0)
  age_2026                  valid:  28910  missing:      0  (lost      0)
  male                      valid:  28836  missing:     74  (lost     74)
  edu_total_years           valid:  26152  missing:   2684  (lost   2684)
  employed                  valid:  26097  missing:     55  (lost     55)
  PHQ4_sum                  valid:  11745  missing:  14352  (lost  14352)
  behandlung                valid:  11672  missing:     73  (lost     73)
  diagnose_selbst           valid:  11067  missing:    605  (lost    605)
  eq5d_daily                valid:  11044  missing:     23  (lost     23)
  Final complete cases:            11044

  Analysis N (final complete cases): 11,044

Model: ai_use ~ age_2026 + male + edu_total_years + employed + PHQ4_sum + behandlung + diagnose_selbst + eq5d_daily
                           Logit Regre

### Extended model: General health-related use with sociodemographic and clinical predictors


In [9]:
# Start from same base inclusion, then AI users only
df_reg_med = df_base_predictors.loc[df_base_predictors['ai_use'] == 1].copy()

# -----------------------------
# Build outcome: med_use (Q1-Q3)
# -----------------------------
purpose_cols_13 = [
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3',
]
purpose_cols_13 = [c for c in purpose_cols_13 if c in df_reg_med.columns]

non_medical_answers = [
    'hierfür nutze ich keine ki',
    '(bisher) kein bedarf, würde ki aber dafür nutzen',
]

def norm_str(x):
    if pd.isna(x):
        return ''
    v = str(x).strip()
    if v.lower() == 'nan':
        return ''
    return v.lower()

non_med_set = {x.lower().strip() for x in non_medical_answers}

med_use_vals = []
for _, row in df_reg_med[purpose_cols_13].iterrows():
    vals = [norm_str(row[c]) for c in purpose_cols_13]
    answered = [v for v in vals if v != '']

    if len(answered) == 0:
        med_use_vals.append(np.nan)
    elif any(v not in non_med_set for v in answered):
        med_use_vals.append(1)
    else:
        med_use_vals.append(0)

df_reg_med['med_use'] = med_use_vals

# -----------------------------
# Build additional predictors
# -----------------------------
phq4_cols = ['PHQ4_1', 'PHQ4_2', 'PHQ4_3', 'PHQ4_4']
phq4_raw = df_reg_med[phq4_cols].apply(pd.to_numeric, errors='coerce')
phq4_recoded = phq4_raw.replace({1: 0, 2: 1, 3: 2, 4: 3})
phq4_recoded = phq4_recoded.where(phq4_raw.isin([1, 2, 3, 4]), np.nan)

mask_phq4_complete = phq4_recoded.notna().all(axis=1)
df_reg_med['PHQ4_sum'] = phq4_recoded.sum(axis=1)
df_reg_med.loc[~mask_phq4_complete, 'PHQ4_sum'] = np.nan

bh = df_reg_med['Behandlung_dich'].astype(str).str.strip().str.lower()
df_reg_med['behandlung'] = np.where(bh == 'a', 1, np.where(bh == 'b', 0, np.nan))

ds = df_reg_med['Diagnose_selbst_dich'].astype(str).str.strip().str.lower()
df_reg_med['diagnose_selbst'] = np.where(ds == 'a', 1, np.where(ds == 'b', 0, np.nan))

eq_map = {'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5}
eq_raw = df_reg_med['EQ5D5L_3'].astype(str).str.strip().str.lower()
df_reg_med['eq5d_daily'] = eq_raw.map(eq_map)

# -----------------------------
# Complete-case regression sample
# -----------------------------
required = [
    'med_use', 'age_2026', 'male', 'edu_total_years', 'employed',
    'PHQ4_sum', 'behandlung', 'diagnose_selbst', 'eq5d_daily',
]

print('Diagnostic: row loss for med_use extended regression (AI users only)')
print(f'  Starting N (AI users):          {len(df_reg_med):>6}')
remaining = df_reg_med.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f'  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})')
print(f'  Final complete cases:           {len(remaining):>6}')
print()

df_reg_med = df_reg_med.dropna(subset=required).copy()

df_reg_med['med_use'] = df_reg_med['med_use'].astype(int)
df_reg_med['male'] = df_reg_med['male'].astype(int)
df_reg_med['employed'] = df_reg_med['employed'].astype(int)
df_reg_med['behandlung'] = df_reg_med['behandlung'].astype(int)
df_reg_med['diagnose_selbst'] = df_reg_med['diagnose_selbst'].astype(int)
df_reg_med['age_2026'] = pd.to_numeric(df_reg_med['age_2026'], errors='coerce')
df_reg_med['edu_total_years'] = pd.to_numeric(df_reg_med['edu_total_years'], errors='coerce')
df_reg_med['PHQ4_sum'] = pd.to_numeric(df_reg_med['PHQ4_sum'], errors='coerce')
df_reg_med['eq5d_daily'] = pd.to_numeric(df_reg_med['eq5d_daily'], errors='coerce')
df_reg_med = df_reg_med.dropna(subset=required)

print(f'  Analysis N (final complete cases): {len(df_reg_med):,}')

# -----------------------------
# Logistic regression
# -----------------------------
preds = ['age_2026', 'male', 'edu_total_years', 'employed',
         'PHQ4_sum', 'behandlung', 'diagnose_selbst', 'eq5d_daily']
formula = 'med_use ~ ' + ' + '.join(preds)
print()
print(f'Model: {formula}')

if len(df_reg_med) < 10:
    print('Too few complete cases to fit model safely.')
elif df_reg_med['med_use'].nunique() < 2:
    print('med_use has only one class; model not fit.')
else:
    keep_preds = [p for p in preds if df_reg_med[p].nunique(dropna=True) > 1]
    dropped = [p for p in preds if p not in keep_preds]
    if dropped:
        print('Dropped constant predictor(s): ' + ', '.join(dropped))
    if not keep_preds:
        print('All predictors are constant; model not fit.')
    else:
        formula_fit = 'med_use ~ ' + ' + '.join(keep_preds)
        result_med = smf.logit(formula_fit, data=df_reg_med).fit(disp=0)
        print(result_med.summary())

        coef = result_med.params
        conf = result_med.conf_int()
        conf.columns = ['CI_low', 'CI_high']
        report = pd.DataFrame({
            'Coef': coef,
            'OR': np.exp(coef),
            'OR_95%_CI_low': np.exp(conf['CI_low']),
            'OR_95%_CI_high': np.exp(conf['CI_high']),
            'p': result_med.pvalues,
        })
        print()
        print('Odds ratios and 95% CI (med_use model):')
        print(report.round(4).to_string())

Diagnostic: row loss for med_use extended regression (AI users only)
  Starting N (AI users):           15058
  med_use                   valid:  15046  missing:     12  (lost     12)
  age_2026                  valid:  15046  missing:      0  (lost      0)
  male                      valid:  15005  missing:     41  (lost     41)
  edu_total_years           valid:  13673  missing:   1332  (lost   1332)
  employed                  valid:  13653  missing:     20  (lost     20)
  PHQ4_sum                  valid:   5414  missing:   8239  (lost   8239)
  behandlung                valid:   5378  missing:     36  (lost     36)
  diagnose_selbst           valid:   5123  missing:    255  (lost    255)
  eq5d_daily                valid:   5113  missing:     10  (lost     10)
  Final complete cases:             5113

  Analysis N (final complete cases): 5,113

Model: med_use ~ age_2026 + male + edu_total_years + employed + PHQ4_sum + behandlung + diagnose_selbst + eq5d_daily
                     

### Extended model: mental health-related use with sociodemographic and clinical predictors


In [10]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df_reg = df_base_predictors.copy()

# -----------------------------
# Build additional predictors (same as your code)
# -----------------------------
phq4_cols = ['PHQ4_1', 'PHQ4_2', 'PHQ4_3', 'PHQ4_4']
phq4_raw = df_reg[phq4_cols].apply(pd.to_numeric, errors='coerce')
phq4_recoded = phq4_raw.replace({1: 0, 2: 1, 3: 2, 4: 3})
phq4_recoded = phq4_recoded.where(phq4_raw.isin([1, 2, 3, 4]), np.nan)

mask_phq4_complete = phq4_recoded.notna().all(axis=1)
df_reg['PHQ4_sum'] = phq4_recoded.sum(axis=1)
df_reg.loc[~mask_phq4_complete, 'PHQ4_sum'] = np.nan

bh = df_reg['Behandlung_dich'].astype(str).str.strip().str.lower()
df_reg['behandlung'] = np.where(bh == 'a', 1, np.where(bh == 'b', 0, np.nan))

ds = df_reg['Diagnose_selbst_dich'].astype(str).str.strip().str.lower()
df_reg['diagnose_selbst'] = np.where(ds == 'a', 1, np.where(ds == 'b', 0, np.nan))

eq_map = {'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5}
eq_raw = df_reg['EQ5D5L_3'].astype(str).str.strip().str.lower()
df_reg['eq5d_daily'] = eq_raw.map(eq_map)

# -----------------------------
# Build emo_use outcomes: excl13 vs incl13
# -----------------------------
purpose_cols_4_12 = [
    'ki_bei_aengsten_und_sorgen',
    'ki_bei_selbstzweifeln_oder_niedergeschlagenheit',
    'ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we',
    'ki_bei_zwischenmenschlichen_konflikten',
    'ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan',
    'ki_bei_trauer_oder_verlust',
    'ki_wenn_grosse_lebensveraenderungen_anstehen',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5',
]
col_13 = 'ki_fuer_sonstige_gesundheits_psychische_oder_emotionale'

purpose_cols_4_12 = [c for c in purpose_cols_4_12 if c in df_reg.columns]
purpose_cols_4_13 = purpose_cols_4_12 + ([col_13] if col_13 in df_reg.columns else [])

non_use_answers = [
    'hierfür nutze ich keine ki',
    '(bisher) kein bedarf, würde ki aber dafür nutzen',
]
non_use_set = {x.lower().strip() for x in non_use_answers}

def norm_str(x):
    if pd.isna(x):
        return ''
    v = str(x).strip()
    if v.lower() == 'nan':
        return ''
    return v.lower()

def build_emo_use(df_in, cols):
    out = []
    for _, row in df_in[cols].iterrows():
        vals = [norm_str(row[c]) for c in cols]
        answered = [v for v in vals if v != '']
        if len(answered) == 0:
            out.append(np.nan)
        elif any(v not in non_use_set for v in answered):
            out.append(1)
        else:
            out.append(0)
    return out

df_reg['emo_use_excl13'] = build_emo_use(df_reg, purpose_cols_4_12)
df_reg['emo_use_incl13'] = build_emo_use(df_reg, purpose_cols_4_13)

# -----------------------------
# Function to run extended model (same predictors as your AI-use model)
# -----------------------------
def run_extended_emo(outcome_col, label):
    required = [
        outcome_col, 'age_2026', 'male', 'edu_total_years', 'employed',
        'PHQ4_sum', 'behandlung', 'diagnose_selbst', 'eq5d_daily'
    ]

    print("\n" + "=" * 90)
    print(label)
    print("=" * 90)
    print(f"Diagnostic: row loss for extended model ({outcome_col})")
    print(f"  Starting N (base predictors):   {len(df_reg):>6}")

    remaining = df_reg.copy()
    for c in required:
        n_valid = int(remaining[c].notna().sum())
        n_miss = int(remaining[c].isna().sum())
        before = len(remaining)
        remaining = remaining.dropna(subset=[c])
        lost = before - len(remaining)
        print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")
    print(f"  Final complete cases:           {len(remaining):>6}")

    dfx = df_reg.dropna(subset=required).copy()

    dfx[outcome_col] = pd.to_numeric(dfx[outcome_col], errors='coerce').astype(int)
    dfx['male'] = pd.to_numeric(dfx['male'], errors='coerce').astype(int)
    dfx['employed'] = pd.to_numeric(dfx['employed'], errors='coerce').astype(int)
    dfx['behandlung'] = pd.to_numeric(dfx['behandlung'], errors='coerce').astype(int)
    dfx['diagnose_selbst'] = pd.to_numeric(dfx['diagnose_selbst'], errors='coerce').astype(int)
    dfx['age_2026'] = pd.to_numeric(dfx['age_2026'], errors='coerce')
    dfx['edu_total_years'] = pd.to_numeric(dfx['edu_total_years'], errors='coerce')
    dfx['PHQ4_sum'] = pd.to_numeric(dfx['PHQ4_sum'], errors='coerce')
    dfx['eq5d_daily'] = pd.to_numeric(dfx['eq5d_daily'], errors='coerce')
    dfx = dfx.dropna(subset=required)

    print(f"\n  Analysis N (final complete cases): {len(dfx):,}")

    preds = ['age_2026', 'male', 'edu_total_years', 'employed',
             'PHQ4_sum', 'behandlung', 'diagnose_selbst', 'eq5d_daily']
    formula = outcome_col + ' ~ ' + ' + '.join(preds)
    print(f"  Model: {formula}")

    if len(dfx) < 10:
        print('Too few complete cases to fit model safely.')
        return None, dfx
    if dfx[outcome_col].nunique() < 2:
        print('Outcome has only one class after filtering; model not fit.')
        return None, dfx

    keep_preds = [p for p in preds if dfx[p].nunique(dropna=True) > 1]
    dropped = [p for p in preds if p not in keep_preds]
    if dropped:
        print('Dropped constant predictor(s): ' + ', '.join(dropped))
    if not keep_preds:
        print('All predictors are constant; model not fit.')
        return None, dfx

    formula_fit = outcome_col + ' ~ ' + ' + '.join(keep_preds)
    result = smf.logit(formula_fit, data=dfx).fit(disp=0)
    print(result.summary())

    coef = result.params
    conf = result.conf_int()
    conf.columns = ['CI_low', 'CI_high']
    report = pd.DataFrame({
        'Coef': coef,
        'OR': np.exp(coef),
        'OR_95%_CI_low': np.exp(conf['CI_low']),
        'OR_95%_CI_high': np.exp(conf['CI_high']),
        'p': result.pvalues,
    })
    print('\nOdds ratios and 95% CI:')
    print(report.round(4).to_string())

    return result, dfx

# Run both
res_excl13, dfx_excl13 = run_extended_emo(
    outcome_col='emo_use_excl13',
    label='MODEL A: EXCLUDING Q4.13 (Q4.4–Q4.12)'
)
res_incl13, dfx_incl13 = run_extended_emo(
    outcome_col='emo_use_incl13',
    label='MODEL B: INCLUDING Q4.13 (Q4.4–Q4.13)'
)


MODEL A: EXCLUDING Q4.13 (Q4.4–Q4.12)
Diagnostic: row loss for extended model (emo_use_excl13)
  Starting N (base predictors):    28910
  emo_use_excl13            valid:  15045  missing:  13865  (lost  13865)
  age_2026                  valid:  15045  missing:      0  (lost      0)
  male                      valid:  15004  missing:     41  (lost     41)
  edu_total_years           valid:  13672  missing:   1332  (lost   1332)
  employed                  valid:  13652  missing:     20  (lost     20)
  PHQ4_sum                  valid:   5414  missing:   8238  (lost   8238)
  behandlung                valid:   5378  missing:     36  (lost     36)
  diagnose_selbst           valid:   5123  missing:    255  (lost    255)
  eq5d_daily                valid:   5113  missing:     10  (lost     10)
  Final complete cases:             5113

  Analysis N (final complete cases): 5,113
  Model: emo_use_excl13 ~ age_2026 + male + edu_total_years + employed + PHQ4_sum + behandlung + diagnose_selbst

### Sensitivity model: mental health-related use (HiTOP instead of PHQ-4)

This specification tests robustness to the choice of symptom burden indicator.


In [11]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df_reg = df_base_predictors.copy()

# -----------------------------
# Build additional predictors (replace PHQ4 with HiTOP_GS_sum)
# -----------------------------

# HiTOP GS sum (HiTOP_1–HiTOP_8), a=1..e=5, complete-case across items
letter_to_num = {"a": 1, "b": 2, "c": 3, "d": 4, "e": 5}

hitop_gs_cols = [f"HiTOP_{i}" for i in range(1, 9) if f"HiTOP_{i}" in df_reg.columns]
if len(hitop_gs_cols) != 8:
    print(f"Warning: expected 8 HiTOP columns (HiTOP_1..HiTOP_8), found {len(hitop_gs_cols)}")

hitop_gs_numeric = df_reg[hitop_gs_cols].apply(
    lambda col: col.astype(str).str.strip().str.lower().map(letter_to_num)
)
df_reg["HiTOP_GS_sum"] = hitop_gs_numeric.sum(axis=1)
mask_hitop_complete = hitop_gs_numeric.notna().all(axis=1)
df_reg.loc[~mask_hitop_complete, "HiTOP_GS_sum"] = np.nan

# Treatment
bh = df_reg["Behandlung_dich"].astype(str).str.strip().str.lower()
df_reg["behandlung"] = np.where(bh == "a", 1, np.where(bh == "b", 0, np.nan))

# Diagnosis
ds = df_reg["Diagnose_selbst_dich"].astype(str).str.strip().str.lower()
df_reg["diagnose_selbst"] = np.where(ds == "a", 1, np.where(ds == "b", 0, np.nan))

# EQ5D daily functioning
eq_map = {"a": 1, "b": 2, "c": 3, "d": 4, "e": 5}
eq_raw = df_reg["EQ5D5L_3"].astype(str).str.strip().str.lower()
df_reg["eq5d_daily"] = eq_raw.map(eq_map)

# -----------------------------
# Build emo_use outcomes: excl13 vs incl13
# -----------------------------
purpose_cols_4_12 = [
    "ki_bei_aengsten_und_sorgen",
    "ki_bei_selbstzweifeln_oder_niedergeschlagenheit",
    "ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we",
    "ki_bei_zwischenmenschlichen_konflikten",
    "ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan",
    "ki_bei_trauer_oder_verlust",
    "ki_wenn_grosse_lebensveraenderungen_anstehen",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5",
]
col_13 = "ki_fuer_sonstige_gesundheits_psychische_oder_emotionale"

purpose_cols_4_12 = [c for c in purpose_cols_4_12 if c in df_reg.columns]
purpose_cols_4_13 = purpose_cols_4_12 + ([col_13] if col_13 in df_reg.columns else [])

non_use_answers = [
    "hierfür nutze ich keine ki",
    "(bisher) kein bedarf, würde ki aber dafür nutzen",
]
non_use_set = {x.lower().strip() for x in non_use_answers}

def norm_str(x):
    if pd.isna(x):
        return ""
    v = str(x).strip()
    if v.lower() == "nan":
        return ""
    return v.lower()

def build_emo_use(df_in, cols):
    out = []
    for _, row in df_in[cols].iterrows():
        vals = [norm_str(row[c]) for c in cols]
        answered = [v for v in vals if v != ""]
        if len(answered) == 0:
            out.append(np.nan)
        elif any(v not in non_use_set for v in answered):
            out.append(1)
        else:
            out.append(0)
    return out

df_reg["emo_use_excl13"] = build_emo_use(df_reg, purpose_cols_4_12)
df_reg["emo_use_incl13"] = build_emo_use(df_reg, purpose_cols_4_13)

# -----------------------------
# Function to run extended model (replace PHQ4_sum with HiTOP_GS_sum)
# -----------------------------
def run_extended_emo(outcome_col, label):
    required = [
        outcome_col, "age_2026", "male", "edu_total_years", "employed",
        "HiTOP_GS_sum", "behandlung", "diagnose_selbst", "eq5d_daily"
    ]

    print("\n" + "=" * 90)
    print(label)
    print("=" * 90)
    print(f"Diagnostic: row loss for extended model ({outcome_col})")
    print(f"  Starting N (base predictors):   {len(df_reg):>6}")

    remaining = df_reg.copy()
    for c in required:
        n_valid = int(remaining[c].notna().sum())
        n_miss = int(remaining[c].isna().sum())
        before = len(remaining)
        remaining = remaining.dropna(subset=[c])
        lost = before - len(remaining)
        print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")
    print(f"  Final complete cases:           {len(remaining):>6}")

    dfx = df_reg.dropna(subset=required).copy()

    dfx[outcome_col] = pd.to_numeric(dfx[outcome_col], errors="coerce").astype(int)
    dfx["male"] = pd.to_numeric(dfx["male"], errors="coerce").astype(int)
    dfx["employed"] = pd.to_numeric(dfx["employed"], errors="coerce").astype(int)
    dfx["behandlung"] = pd.to_numeric(dfx["behandlung"], errors="coerce").astype(int)
    dfx["diagnose_selbst"] = pd.to_numeric(dfx["diagnose_selbst"], errors="coerce").astype(int)

    dfx["age_2026"] = pd.to_numeric(dfx["age_2026"], errors="coerce")
    dfx["edu_total_years"] = pd.to_numeric(dfx["edu_total_years"], errors="coerce")
    dfx["HiTOP_GS_sum"] = pd.to_numeric(dfx["HiTOP_GS_sum"], errors="coerce")
    dfx["eq5d_daily"] = pd.to_numeric(dfx["eq5d_daily"], errors="coerce")
    dfx = dfx.dropna(subset=required)

    print(f"\n  Analysis N (final complete cases): {len(dfx):,}")

    preds = [
        "age_2026", "male", "edu_total_years", "employed",
        "HiTOP_GS_sum", "behandlung", "diagnose_selbst", "eq5d_daily"
    ]
    formula = outcome_col + " ~ " + " + ".join(preds)
    print(f"  Model: {formula}")

    if len(dfx) < 10:
        print("Too few complete cases to fit model safely.")
        return None, dfx
    if dfx[outcome_col].nunique() < 2:
        print("Outcome has only one class after filtering; model not fit.")
        return None, dfx

    keep_preds = [p for p in preds if dfx[p].nunique(dropna=True) > 1]
    dropped = [p for p in preds if p not in keep_preds]
    if dropped:
        print("Dropped constant predictor(s): " + ", ".join(dropped))
    if not keep_preds:
        print("All predictors are constant; model not fit.")
        return None, dfx

    formula_fit = outcome_col + " ~ " + " + ".join(keep_preds)
    result = smf.logit(formula_fit, data=dfx).fit(disp=0)
    print(result.summary())

    coef = result.params
    conf = result.conf_int()
    conf.columns = ["CI_low", "CI_high"]
    report = pd.DataFrame({
        "Coef": coef,
        "OR": np.exp(coef),
        "OR_95%_CI_low": np.exp(conf["CI_low"]),
        "OR_95%_CI_high": np.exp(conf["CI_high"]),
        "p": result.pvalues,
    })
    print("\nOdds ratios and 95% CI:")
    print(report.round(4).to_string())

    return result, dfx

# Run both
res_excl13, dfx_excl13 = run_extended_emo(
    outcome_col="emo_use_excl13",
    label="MODEL A: EXCLUDING Q4.13 (Q4.4–Q4.12)"
)
res_incl13, dfx_incl13 = run_extended_emo(
    outcome_col="emo_use_incl13",
    label="MODEL B: INCLUDING Q4.13 (Q4.4–Q4.13)"
)


MODEL A: EXCLUDING Q4.13 (Q4.4–Q4.12)
Diagnostic: row loss for extended model (emo_use_excl13)
  Starting N (base predictors):    28910
  emo_use_excl13            valid:  15045  missing:  13865  (lost  13865)
  age_2026                  valid:  15045  missing:      0  (lost      0)
  male                      valid:  15004  missing:     41  (lost     41)
  edu_total_years           valid:  13672  missing:   1332  (lost   1332)
  employed                  valid:  13652  missing:     20  (lost     20)
  HiTOP_GS_sum              valid:   5429  missing:   8223  (lost   8223)
  behandlung                valid:   5394  missing:     35  (lost     35)
  diagnose_selbst           valid:   5141  missing:    253  (lost    253)
  eq5d_daily                valid:   5131  missing:     10  (lost     10)
  Final complete cases:             5131

  Analysis N (final complete cases): 5,131
  Model: emo_use_excl13 ~ age_2026 + male + edu_total_years + employed + HiTOP_GS_sum + behandlung + diagnose_se

### Sensitivity model: mental health-related support use (number of diagnoses)

This specification replaces the binary diagnosis indicator with a diagnosis count.


In [12]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df_reg = df_base_predictors.copy()

# -----------------------------
# Build additional predictors (keep PHQ4, keep treatment, keep EQ5D)
# -----------------------------
# PHQ-4: RECODE 1-4 -> 0-3, then sum only if all 4 items are present/valid
phq4_cols = ["PHQ4_1", "PHQ4_2", "PHQ4_3", "PHQ4_4"]
phq4_raw = df_reg[phq4_cols].apply(pd.to_numeric, errors="coerce")
phq4_recoded = phq4_raw.replace({1: 0, 2: 1, 3: 2, 4: 3})
phq4_recoded = phq4_recoded.where(phq4_raw.isin([1, 2, 3, 4]), np.nan)

mask_phq4_complete = phq4_recoded.notna().all(axis=1)
df_reg["PHQ4_sum"] = phq4_recoded.sum(axis=1)
df_reg.loc[~mask_phq4_complete, "PHQ4_sum"] = np.nan

# Behandlung_dich: a=1, b=0, else NaN
bh = df_reg["Behandlung_dich"].astype(str).str.strip().str.lower()
df_reg["behandlung"] = np.where(bh == "a", 1, np.where(bh == "b", 0, np.nan))

# EQ5D5L_3: a=1 ... e=5, else NaN
eq_map = {"a": 1, "b": 2, "c": 3, "d": 4, "e": 5}
eq_raw = df_reg["EQ5D5L_3"].astype(str).str.strip().str.lower()
df_reg["eq5d_daily"] = eq_raw.map(eq_map)

# -----------------------------
# NEW: n_diagnoses (ordinal count of psychiatric diagnoses)
# Rules:
# - Diagnose_selbst_dich == 'a' (Yes) -> sum Diagnose_* present==1
# - answered but NOT 'a' (e.g., 'b' No, 'c' don't know, other non-blank) -> 0
# - blank -> NaN
# -----------------------------
diagnose_selbst_col = "Diagnose_selbst_dich"

diagnose_cols = [
    "Diagnose_PD", "Diagnose_DD", "Diagnose_BD", "Diagnose_AD", "Diagnose_OCD",
    "Diagnose_PTSD", "Diagnose_SD", "Diagnose_ED", "Diagnose_PBD", "Diagnose_DMD",
    "Diagnose_ADHD", "Diagnose_ASD", "Diagnose_SUD", "Diagnose_D", "Diagnose_andere",
]

ds_self = df_reg[diagnose_selbst_col]
ds_stripped = ds_self.astype(str).str.strip()
ds_lower = ds_stripped.str.lower()

is_blank = ds_self.isna() | (ds_stripped == "") | ds_lower.isin(["nan", "none"])
mask_yes = ds_lower == "a"
mask_other_answered = (~is_blank) & (~mask_yes)

df_reg["n_diagnoses"] = np.nan
df_reg.loc[mask_other_answered, "n_diagnoses"] = 0

cols_ok = [c for c in diagnose_cols if c in df_reg.columns]
missing_diag_cols = [c for c in diagnose_cols if c not in df_reg.columns]
if missing_diag_cols:
    print(f"Note: missing Diagnose_* columns ignored for n_diagnoses: {missing_diag_cols}")

if cols_ok and mask_yes.any():
    blk = df_reg.loc[mask_yes, cols_ok]

    present = pd.DataFrame(
        {
            c: (pd.to_numeric(blk[c], errors="coerce") == 1)
            | (blk[c].astype(str).str.strip() == "1")
            for c in cols_ok
        },
        index=blk.index,
    )
    df_reg.loc[mask_yes, "n_diagnoses"] = present.sum(axis=1)

# Optional quick check
print("\nn_diagnoses distribution (including missing):")
print(df_reg["n_diagnoses"].value_counts(dropna=False).sort_index().to_string())

# -----------------------------
# Build emo_use outcomes: excl13 vs incl13
# -----------------------------
purpose_cols_4_12 = [
    "ki_bei_aengsten_und_sorgen",
    "ki_bei_selbstzweifeln_oder_niedergeschlagenheit",
    "ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we",
    "ki_bei_zwischenmenschlichen_konflikten",
    "ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan",
    "ki_bei_trauer_oder_verlust",
    "ki_wenn_grosse_lebensveraenderungen_anstehen",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5",
]
col_13 = "ki_fuer_sonstige_gesundheits_psychische_oder_emotionale"

purpose_cols_4_12 = [c for c in purpose_cols_4_12 if c in df_reg.columns]
purpose_cols_4_13 = purpose_cols_4_12 + ([col_13] if col_13 in df_reg.columns else [])

non_use_answers = [
    "hierfür nutze ich keine ki",
    "(bisher) kein bedarf, würde ki aber dafür nutzen",
]
non_use_set = {x.lower().strip() for x in non_use_answers}

def norm_str(x):
    if pd.isna(x):
        return ""
    v = str(x).strip()
    if v.lower() == "nan":
        return ""
    return v.lower()

def build_emo_use(df_in, cols):
    out = []
    for _, row in df_in[cols].iterrows():
        vals = [norm_str(row[c]) for c in cols]
        answered = [v for v in vals if v != ""]
        if len(answered) == 0:
            out.append(np.nan)
        elif any(v not in non_use_set for v in answered):
            out.append(1)
        else:
            out.append(0)
    return out

df_reg["emo_use_excl13"] = build_emo_use(df_reg, purpose_cols_4_12)
df_reg["emo_use_incl13"] = build_emo_use(df_reg, purpose_cols_4_13)

# -----------------------------
# Function to run extended model (PHQ4 kept; diagnosis replaced by n_diagnoses)
# -----------------------------
def run_extended_emo(outcome_col, label):
    required = [
        outcome_col, "age_2026", "male", "edu_total_years", "employed",
        "PHQ4_sum", "behandlung", "n_diagnoses", "eq5d_daily"
    ]

    print("\n" + "=" * 90)
    print(label)
    print("=" * 90)
    print(f"Diagnostic: row loss for extended model ({outcome_col})")
    print(f"  Starting N (base predictors):   {len(df_reg):>6}")

    remaining = df_reg.copy()
    for c in required:
        n_valid = int(remaining[c].notna().sum())
        n_miss = int(remaining[c].isna().sum())
        before = len(remaining)
        remaining = remaining.dropna(subset=[c])
        lost = before - len(remaining)
        print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")
    print(f"  Final complete cases:           {len(remaining):>6}")

    dfx = df_reg.dropna(subset=required).copy()

    # types
    dfx[outcome_col] = pd.to_numeric(dfx[outcome_col], errors="coerce").astype(int)
    dfx["male"] = pd.to_numeric(dfx["male"], errors="coerce").astype(int)
    dfx["employed"] = pd.to_numeric(dfx["employed"], errors="coerce").astype(int)
    dfx["behandlung"] = pd.to_numeric(dfx["behandlung"], errors="coerce").astype(int)

    dfx["age_2026"] = pd.to_numeric(dfx["age_2026"], errors="coerce")
    dfx["edu_total_years"] = pd.to_numeric(dfx["edu_total_years"], errors="coerce")
    dfx["PHQ4_sum"] = pd.to_numeric(dfx["PHQ4_sum"], errors="coerce")
    dfx["n_diagnoses"] = pd.to_numeric(dfx["n_diagnoses"], errors="coerce")
    dfx["eq5d_daily"] = pd.to_numeric(dfx["eq5d_daily"], errors="coerce")
    dfx = dfx.dropna(subset=required)

    print(f"\n  Analysis N (final complete cases): {len(dfx):,}")

    preds = [
        "age_2026", "male", "edu_total_years", "employed",
        "PHQ4_sum", "behandlung", "n_diagnoses", "eq5d_daily"
    ]
    formula = outcome_col + " ~ " + " + ".join(preds)
    print(f"  Model: {formula}")

    if len(dfx) < 10:
        print("Too few complete cases to fit model safely.")
        return None, dfx
    if dfx[outcome_col].nunique() < 2:
        print("Outcome has only one class after filtering; model not fit.")
        return None, dfx

    keep_preds = [p for p in preds if dfx[p].nunique(dropna=True) > 1]
    dropped = [p for p in preds if p not in keep_preds]
    if dropped:
        print("Dropped constant predictor(s): " + ", ".join(dropped))
    if not keep_preds:
        print("All predictors are constant; model not fit.")
        return None, dfx

    formula_fit = outcome_col + " ~ " + " + ".join(keep_preds)
    result = smf.logit(formula_fit, data=dfx).fit(disp=0)
    print(result.summary())

    coef = result.params
    conf = result.conf_int()
    conf.columns = ["CI_low", "CI_high"]
    report = pd.DataFrame({
        "Coef": coef,
        "OR": np.exp(coef),
        "OR_95%_CI_low": np.exp(conf["CI_low"]),
        "OR_95%_CI_high": np.exp(conf["CI_high"]),
        "p": result.pvalues,
    })
    print("\nOdds ratios and 95% CI:")
    print(report.round(4).to_string())

    return result, dfx

# Run both
res_excl13, dfx_excl13 = run_extended_emo(
    outcome_col="emo_use_excl13",
    label="MODEL A: EXCLUDING Q4.13 (Q4.4–Q4.12)"
)
res_incl13, dfx_incl13 = run_extended_emo(
    outcome_col="emo_use_incl13",
    label="MODEL B: INCLUDING Q4.13 (Q4.4–Q4.13)"
)


n_diagnoses distribution (including missing):
n_diagnoses
0.0    10961
1.0     1024
2.0      574
3.0      243
4.0       98
5.0       26
6.0       11
7.0        3
8.0        4
9.0        1
NaN    15965

MODEL A: EXCLUDING Q4.13 (Q4.4–Q4.12)
Diagnostic: row loss for extended model (emo_use_excl13)
  Starting N (base predictors):    28910
  emo_use_excl13            valid:  15045  missing:  13865  (lost  13865)
  age_2026                  valid:  15045  missing:      0  (lost      0)
  male                      valid:  15004  missing:     41  (lost     41)
  edu_total_years           valid:  13672  missing:   1332  (lost   1332)
  employed                  valid:  13652  missing:     20  (lost     20)
  PHQ4_sum                  valid:   5414  missing:   8238  (lost   8238)
  behandlung                valid:   5378  missing:     36  (lost     36)
  n_diagnoses               valid:   5363  missing:     15  (lost     15)
  eq5d_daily                valid:   5352  missing:     11  (lost    

## Main models: side effects of AI use

### Predictors: sociodemographics


In [13]:
# Regression: sociodemographic predictors -> side effects (binary)
# Sample: AI users (ai_use == 1) from the prepared base dataset.
# Note: This model is not restricted to health-related AI users; it targets side
# effects among AI users in general.

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# -----------------------------
# 1) Base sample
# -----------------------------
d_sub = df_base_predictors.loc[df_base_predictors["ai_use"] == 1].copy()

# -----------------------------
# 2) Side-effect coding
# -----------------------------
side_effect_cols = {
    "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner": "G2Q5.1 Social alienation",
    "ki_ai_dependency": "G2Q6 AI dependency",
    "ki_decision_difficulty": "G2Q7 Decision difficulty",
}

for c in side_effect_cols.keys():
    if c not in d_sub.columns:
        print(f"Warning: missing side-effect column: {c}")

SOCIAL_YES_NUM = {-1, -2, -3}
YES_VALUES_TEXT = {
    "trifft ein wenig zu",
    "trifft teilweise zu",
    "trifft völlig zu",
    "trifft voellig zu",
}

def to_yes_no(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    if col == "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner":
        num = pd.to_numeric(s, errors="coerce")
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0

    if s.lower() in YES_VALUES_TEXT:
        return 1
    return 0

for col in side_effect_cols:
    ycol = f"{col}_yes"
    if col in d_sub.columns:
        d_sub[ycol] = d_sub[col].apply(lambda x: to_yes_no(x, col))
    else:
        d_sub[ycol] = np.nan

# -----------------------------
# 3) Quick counts per side effect
# -----------------------------
print("Per-side-effect Yes/No/Missing counts:")
for col, label in side_effect_cols.items():
    ycol = f"{col}_yes"
    n_yes = int((d_sub[ycol] == 1).sum())
    n_no = int((d_sub[ycol] == 0).sum())
    n_miss = int(d_sub[ycol].isna().sum())
    n_ans = n_yes + n_no
    pct_yes = (100 * n_yes / n_ans) if n_ans > 0 else np.nan
    print(f"  {label:<35} Yes={n_yes:>6}  No={n_no:>6}  Missing={n_miss:>6}  Yes%={pct_yes:.1f}" if n_ans > 0
          else f"  {label:<35} Yes={n_yes:>6}  No={n_no:>6}  Missing={n_miss:>6}  Yes%=n/a")

# -----------------------------
# 4) Person-level outcome: any side effect
# -----------------------------
ycols = [f"{c}_yes" for c in side_effect_cols.keys()]
answered_any = d_sub[ycols].notna().any(axis=1)
any_yes = d_sub[ycols].eq(1).any(axis=1)

# 1 = at least one side effect yes
# 0 = answered side-effect items but none yes
# NaN = all three missing
d_sub["side_effect_any"] = np.where(~answered_any, np.nan, np.where(any_yes, 1, 0))

print("\nPerson-level outcome side_effect_any:")
print(d_sub["side_effect_any"].value_counts(dropna=False).to_string())

# -----------------------------
# 5) Complete-case regression sample
# -----------------------------
required = ["side_effect_any", "age_2026", "male", "edu_total_years", "employed"]

print("\nDiagnostic: row loss for side_effect_any regression")
print(f"  Starting N (AI users):          {len(d_sub):>6}")
remaining = d_sub.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")
print(f"  Final complete cases:           {len(remaining):>6}")
print()

df_reg_se = d_sub.dropna(subset=required).copy()

# enforce types
df_reg_se["side_effect_any"] = df_reg_se["side_effect_any"].astype(int)
df_reg_se["male"] = df_reg_se["male"].astype(int)
df_reg_se["employed"] = df_reg_se["employed"].astype(int)
df_reg_se["age_2026"] = pd.to_numeric(df_reg_se["age_2026"], errors="coerce")
df_reg_se["edu_total_years"] = pd.to_numeric(df_reg_se["edu_total_years"], errors="coerce")
df_reg_se = df_reg_se.dropna(subset=["age_2026", "edu_total_years"])

print(f"  Analysis N (final complete cases): {len(df_reg_se):,}")

# -----------------------------
# 6) Logistic regression
# -----------------------------
preds = ["age_2026", "male", "edu_total_years", "employed"]
formula = "side_effect_any ~ " + " + ".join(preds)

print()
print(f"Model: {formula}")

if len(df_reg_se) < 10:
    print("Too few complete cases to fit model safely.")
elif df_reg_se["side_effect_any"].nunique() < 2:
    print("Outcome has only one class after filtering; model not fit.")
else:
    keep_preds = [p for p in preds if df_reg_se[p].nunique(dropna=True) > 1]
    dropped = [p for p in preds if p not in keep_preds]
    if dropped:
        print("Dropped constant predictor(s): " + ", ".join(dropped))
    if not keep_preds:
        print("All predictors are constant; model not fit.")
    else:
        formula_fit = "side_effect_any ~ " + " + ".join(keep_preds)
        try:
            result_se = smf.logit(formula_fit, data=df_reg_se).fit(disp=0)
            print(result_se.summary())

            coef = result_se.params
            conf = result_se.conf_int()
            conf.columns = ["CI_low", "CI_high"]
            report = pd.DataFrame({
                "Coef": coef,
                "OR": np.exp(coef),
                "OR_95%_CI_low": np.exp(conf["CI_low"]),
                "OR_95%_CI_high": np.exp(conf["CI_high"]),
                "p": result_se.pvalues,
            })
            print()
            print("Odds ratios and 95% CI (side_effect_any model):")
            print(report.round(4).to_string())
        except Exception as e:
            print(f"Model fit failed: {e}")

Per-side-effect Yes/No/Missing counts:
  G2Q5.1 Social alienation            Yes=    43  No= 14545  Missing=   470  Yes%=0.3
  G2Q6 AI dependency                  Yes=  1499  No= 13391  Missing=   168  Yes%=10.1
  G2Q7 Decision difficulty            Yes=   696  No= 14195  Missing=   167  Yes%=4.7

Person-level outcome side_effect_any:
side_effect_any
0.0    13087
1.0     1822
NaN      149

Diagnostic: row loss for side_effect_any regression
  Starting N (AI users):           15058
  side_effect_any           valid:  14909  missing:    149  (lost    149)
  age_2026                  valid:  14909  missing:      0  (lost      0)
  male                      valid:  14868  missing:     41  (lost     41)
  edu_total_years           valid:  13554  missing:   1314  (lost   1314)
  employed                  valid:  13534  missing:     20  (lost     20)
  Final complete cases:            13534

  Analysis N (final complete cases): 13,534

Model: side_effect_any ~ age_2026 + male + edu_total_year

### Sensitivity model: side effects with categorical education


In [14]:
# Regression: sociodemographic predictors -> side effects (Yes/No)
# Updated: uses categorical education edu5 (ref = "medium") instead of edu_total_years

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# -----------------------------
# 0) Build edu5 on df_base_predictors (Option A + unknown)
# -----------------------------
edu_school_col = "sozdem_welchen_hoechsten_allgemeinbildenden_schulabschl"
edu_coded_col  = "education_level_coded"
UNKNOWN_TOKENS = {"Weiß nicht", "Keine Angabe"}

def _norm(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return None
    return s

def _school_level(val):
    s = _norm(val)
    if s is None or s in UNKNOWN_TOKENS:
        return None
    if s == "Schüler:in, besuche eine allgemeinbildende Schule (Vollzeit oder Teilzeit)":
        return None

    low = {
        "von der Schule abgegangen ohne Hauptschulabschluss (Volksschulabschluss)",
        "Hauptschulabschluss (Volksschulabschluss)",
        "Polytechnische Oberschule der DDR mit Abschluss der 8. oder 9. Klasse",
    }
    medium = {
        "Realschulabschluss (Mittlere Reife)",
        "Polytechnische Oberschule der DDR mit Abschluss der 10. Klasse",
    }
    high = {
        "Fachhochschulreife, Abschluss einer Fachoberschule",
        "Allgemeine oder fachgebundene Hochschulreife/Abitur (Gymnasium bzw. Abschluss der Erweiterten Oberschule (der DDR))",
    }

    if s in low:
        return 0
    if s in medium:
        return 1
    if s in high:
        return 2
    return None

def _voc_level(val):
    s = _norm(val)
    if s is None or s in UNKNOWN_TOKENS:
        return None

    in_training = {
        "Noch in beruflicher Ausbildung (Berufsvorbereitungsjahr, Ausbildung, Praktikum, Studium)",
        "Schüler/in und besuche eine berufsorientierte Aufbau-, Fachschule o.ä.",
    }
    none_or_basic = {
        "Keinen beruflichen Abschluss und nicht in beruflicher Ausbildung",
        "Betriebliche Berufsausbildung (Lehre) abgeschlossen",
        "Schulische Berufsausbildung abgeschlossen (Berufsfachschule, Handelsschule. Vorbereitungsdienst für den m",
    }
    adv_or_tertiary = {
        "Ausbildung an einer Fach-, Meister-, Technikerschule, Berufs- oder Fachakademie abgeschlossen",
        "Ausbildung an einer Fachschule der DDR abgeschlossen",
        "Bachelor an einer (Fach-) Hochschule abgeschlossen",
        "Fachhochschulabschluss (z.B. Diplom, Master)",
        "Universitätsabschluss (z.B. Diplom, Magister, Staatsexamen, Master)",
    }

    if s in in_training:
        return "in_training"
    if s in none_or_basic:
        return 0
    if s in adv_or_tertiary:
        return 1
    if s == "Promotion":
        return 2
    return None

def edu5_category(school_val, voc_val):
    s_lvl = _school_level(school_val)
    v_lvl = _voc_level(voc_val)

    if v_lvl == 2:
        return "doctorate"
    if v_lvl == 1:
        return "tertiary_adv"

    if v_lvl == "in_training":
        if s_lvl is None:
            return "unknown"
        return "low" if s_lvl == 0 else ("medium" if s_lvl == 1 else "high")

    if s_lvl is not None:
        if s_lvl == 0:
            return "medium" if v_lvl == 0 else "low"
        if s_lvl == 1:
            return "medium"
        if s_lvl == 2:
            return "high"

    if v_lvl == 0:
        return "medium"

    return "unknown"

edu5_order = ["low", "medium", "high", "tertiary_adv", "doctorate", "unknown"]

df_base_predictors = df_base_predictors.copy()
df_base_predictors["edu5"] = [
    edu5_category(s, v)
    for s, v in zip(df_base_predictors.get(edu_school_col), df_base_predictors.get(edu_coded_col))
]
df_base_predictors["edu5"] = pd.Categorical(df_base_predictors["edu5"], categories=edu5_order, ordered=True)

# -----------------------------
# 1) Base sample
# -----------------------------
d_sub = df_base_predictors.loc[df_base_predictors["ai_use"] == 1].copy()

# -----------------------------
# 2) Side-effect coding
# -----------------------------
side_effect_cols = {
    "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner": "G2Q5.1 Social alienation",
    "ki_ai_dependency": "G2Q6 AI dependency",
    "ki_decision_difficulty": "G2Q7 Decision difficulty",
}

for c in side_effect_cols.keys():
    if c not in d_sub.columns:
        print(f"Warning: missing side-effect column: {c}")

SOCIAL_YES_NUM = {-1, -2, -3}
YES_VALUES_TEXT = {
    "trifft ein wenig zu",
    "trifft teilweise zu",
    "trifft völlig zu",
    "trifft voellig zu",
}

def to_yes_no(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    if col == "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner":
        num = pd.to_numeric(s, errors="coerce")
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0

    if s.lower() in YES_VALUES_TEXT:
        return 1
    return 0

for col in side_effect_cols:
    ycol = f"{col}_yes"
    if col in d_sub.columns:
        d_sub[ycol] = d_sub[col].apply(lambda x: to_yes_no(x, col))
    else:
        d_sub[ycol] = np.nan

# -----------------------------
# 3) Quick counts per side effect
# -----------------------------
print("Per-side-effect Yes/No/Missing counts:")
for col, label in side_effect_cols.items():
    ycol = f"{col}_yes"
    n_yes = int((d_sub[ycol] == 1).sum())
    n_no = int((d_sub[ycol] == 0).sum())
    n_miss = int(d_sub[ycol].isna().sum())
    n_ans = n_yes + n_no
    pct_yes = (100 * n_yes / n_ans) if n_ans > 0 else np.nan
    print(f"  {label:<35} Yes={n_yes:>6}  No={n_no:>6}  Missing={n_miss:>6}  Yes%={pct_yes:.1f}" if n_ans > 0
          else f"  {label:<35} Yes={n_yes:>6}  No={n_no:>6}  Missing={n_miss:>6}  Yes%=n/a")

# -----------------------------
# 4) Person-level outcome: any side effect
# -----------------------------
ycols = [f"{c}_yes" for c in side_effect_cols.keys()]
answered_any = d_sub[ycols].notna().any(axis=1)
any_yes = d_sub[ycols].eq(1).any(axis=1)

d_sub["side_effect_any"] = np.where(~answered_any, np.nan, np.where(any_yes, 1, 0))

print("\nPerson-level outcome side_effect_any:")
print(d_sub["side_effect_any"].value_counts(dropna=False).to_string())

# -----------------------------
# 5) Complete-case regression sample
# -----------------------------
required = ["side_effect_any", "age_2026", "male", "edu5", "employed"]

print("\nDiagnostic: row loss for side_effect_any regression")
print(f"  Starting N (AI users):          {len(d_sub):>6}")
remaining = d_sub.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")
print(f"  Final complete cases:           {len(remaining):>6}")
print()

df_reg_se = d_sub.dropna(subset=required).copy()

# enforce types (edu5 stays categorical)
df_reg_se["side_effect_any"] = df_reg_se["side_effect_any"].astype(int)
df_reg_se["male"] = pd.to_numeric(df_reg_se["male"], errors="coerce").astype(int)
df_reg_se["employed"] = pd.to_numeric(df_reg_se["employed"], errors="coerce").astype(int)
df_reg_se["age_2026"] = pd.to_numeric(df_reg_se["age_2026"], errors="coerce")
df_reg_se = df_reg_se.dropna(subset=["age_2026"])

print(f"  Analysis N (final complete cases): {len(df_reg_se):,}")

# -----------------------------
# 6) Logistic regression
# -----------------------------
edu_term = "C(edu5, Treatment(reference='medium'))"
preds_num = ["age_2026", "male", "employed"]
formula = "side_effect_any ~ " + " + ".join(preds_num + [edu_term])

print()
print(f"Model: {formula}")

if len(df_reg_se) < 10:
    print("Too few complete cases to fit model safely.")
elif df_reg_se["side_effect_any"].nunique() < 2:
    print("Outcome has only one class after filtering; model not fit.")
else:
    keep_num = [p for p in preds_num if df_reg_se[p].nunique(dropna=True) > 1]
    dropped = [p for p in preds_num if p not in keep_num]
    if dropped:
        print("Dropped constant numeric predictor(s): " + ", ".join(dropped))

    rhs = keep_num + [edu_term]
    formula_fit = "side_effect_any ~ " + " + ".join(rhs)

    try:
        result_se = smf.logit(formula_fit, data=df_reg_se).fit(disp=0)
        print(result_se.summary())

        coef = result_se.params
        conf = result_se.conf_int()
        conf.columns = ["CI_low", "CI_high"]
        report = pd.DataFrame({
            "Coef": coef,
            "OR": np.exp(coef),
            "OR_95%_CI_low": np.exp(conf["CI_low"]),
            "OR_95%_CI_high": np.exp(conf["CI_high"]),
            "p": result_se.pvalues,
        })
        print()
        print("Odds ratios and 95% CI (side_effect_any model):")
        print(report.round(4).to_string())
    except Exception as e:
        print(f"Model fit failed: {e}")

Per-side-effect Yes/No/Missing counts:
  G2Q5.1 Social alienation            Yes=    43  No= 14545  Missing=   470  Yes%=0.3
  G2Q6 AI dependency                  Yes=  1499  No= 13391  Missing=   168  Yes%=10.1
  G2Q7 Decision difficulty            Yes=   696  No= 14195  Missing=   167  Yes%=4.7

Person-level outcome side_effect_any:
side_effect_any
0.0    13087
1.0     1822
NaN      149

Diagnostic: row loss for side_effect_any regression
  Starting N (AI users):           15058
  side_effect_any           valid:  14909  missing:    149  (lost    149)
  age_2026                  valid:  14909  missing:      0  (lost      0)
  male                      valid:  14868  missing:     41  (lost     41)
  edu5                      valid:  14868  missing:      0  (lost      0)
  employed                  valid:  14688  missing:    180  (lost    180)
  Final complete cases:            14688

  Analysis N (final complete cases): 14,688

Model: side_effect_any ~ age_2026 + male + employed + C(e

### Extended model: side effects with sociodemographic and clinical predictors


In [15]:
# Extended regression: sociodemographic + clinical predictors -> side effects (Yes/No)
# Base sample here: AI users only (ai_use == 1)

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# -----------------------------
# 1) Base sample
# -----------------------------
d_sub = df_base_predictors.loc[df_base_predictors["ai_use"] == 1].copy()

# -----------------------------
# 2) Side-effect coding (your exact logic)
# -----------------------------
side_effect_cols = {
    "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner": "G2Q5.1 Social alienation",
    "ki_ai_dependency": "G2Q6 AI dependency",
    "ki_decision_difficulty": "G2Q7 Decision difficulty",
}

for c in side_effect_cols:
    if c not in d_sub.columns:
        print(f"Warning: missing side-effect column: {c}")

SOCIAL_YES_NUM = {-1, -2, -3}
YES_VALUES_TEXT = {
    "trifft ein wenig zu",
    "trifft teilweise zu",
    "trifft völlig zu",
    "trifft voellig zu",
}

def to_yes_no(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    if col == "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner":
        num = pd.to_numeric(s, errors="coerce")
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0

    if s.lower() in YES_VALUES_TEXT:
        return 1
    return 0

for col in side_effect_cols:
    ycol = f"{col}_yes"
    if col in d_sub.columns:
        d_sub[ycol] = d_sub[col].apply(lambda x: to_yes_no(x, col))
    else:
        d_sub[ycol] = np.nan

# Counts per side-effect item
print("Per-side-effect Yes/No/Missing counts:")
for col, label in side_effect_cols.items():
    ycol = f"{col}_yes"
    n_yes = int((d_sub[ycol] == 1).sum())
    n_no = int((d_sub[ycol] == 0).sum())
    n_miss = int(d_sub[ycol].isna().sum())
    n_ans = n_yes + n_no
    pct_yes = (100 * n_yes / n_ans) if n_ans > 0 else np.nan
    print(f"  {label:<35} Yes={n_yes:>6}  No={n_no:>6}  Missing={n_miss:>6}  Yes%={pct_yes:.1f}" if n_ans > 0
          else f"  {label:<35} Yes={n_yes:>6}  No={n_no:>6}  Missing={n_miss:>6}  Yes%=n/a")

# Person-level outcome: any side effect
ycols = [f"{c}_yes" for c in side_effect_cols]
answered_any = d_sub[ycols].notna().any(axis=1)
any_yes = d_sub[ycols].eq(1).any(axis=1)
d_sub["side_effect_any"] = np.where(~answered_any, np.nan, np.where(any_yes, 1, 0))

print("\nPerson-level outcome side_effect_any:")
print(d_sub["side_effect_any"].value_counts(dropna=False).to_string())

# -----------------------------
# 3) Additional predictors
# -----------------------------
# PHQ-4: RECODE 1-4 -> 0-3, then sum only if all 4 items are present/valid
phq4_cols = ["PHQ4_1", "PHQ4_2", "PHQ4_3", "PHQ4_4"]
phq4_raw = d_sub[phq4_cols].apply(pd.to_numeric, errors="coerce")
phq4_recoded = phq4_raw.replace({1: 0, 2: 1, 3: 2, 4: 3})
phq4_recoded = phq4_recoded.where(phq4_raw.isin([1, 2, 3, 4]), np.nan)

mask_phq4_complete = phq4_recoded.notna().all(axis=1)
d_sub["PHQ4_sum"] = phq4_recoded.sum(axis=1)
d_sub.loc[~mask_phq4_complete, "PHQ4_sum"] = np.nan

# Behandlung_dich: a=1, b=0, else NaN
bh = d_sub["Behandlung_dich"].astype(str).str.strip().str.lower()
d_sub["behandlung"] = np.where(bh == "a", 1, np.where(bh == "b", 0, np.nan))

# Diagnose_selbst_dich: a=1, b=0, c/else=NaN
ds = d_sub["Diagnose_selbst_dich"].astype(str).str.strip().str.lower()
d_sub["diagnose_selbst"] = np.where(ds == "a", 1, np.where(ds == "b", 0, np.nan))

# EQ5D5L_3: a=1 ... e=5, else NaN
eq_map = {"a": 1, "b": 2, "c": 3, "d": 4, "e": 5}
eq_raw = d_sub["EQ5D5L_3"].astype(str).str.strip().str.lower()
d_sub["eq5d_daily"] = eq_raw.map(eq_map)

# -----------------------------
# 4) Complete-case regression sample
# -----------------------------
required = [
    "side_effect_any",
    "age_2026", "male", "edu_total_years", "employed",
    "PHQ4_sum", "behandlung", "diagnose_selbst", "eq5d_daily",
]

print("\nDiagnostic: row loss for extended side_effect_any regression")
print(f"  Starting N (AI users):          {len(d_sub):>6}")
remaining = d_sub.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")
print(f"  Final complete cases:           {len(remaining):>6}")
print()

df_reg_se = d_sub.dropna(subset=required).copy()

# type safety
df_reg_se["side_effect_any"] = df_reg_se["side_effect_any"].astype(int)
df_reg_se["male"] = df_reg_se["male"].astype(int)
df_reg_se["employed"] = df_reg_se["employed"].astype(int)
df_reg_se["behandlung"] = df_reg_se["behandlung"].astype(int)
df_reg_se["diagnose_selbst"] = df_reg_se["diagnose_selbst"].astype(int)

df_reg_se["age_2026"] = pd.to_numeric(df_reg_se["age_2026"], errors="coerce")
df_reg_se["edu_total_years"] = pd.to_numeric(df_reg_se["edu_total_years"], errors="coerce")
df_reg_se["PHQ4_sum"] = pd.to_numeric(df_reg_se["PHQ4_sum"], errors="coerce")
df_reg_se["eq5d_daily"] = pd.to_numeric(df_reg_se["eq5d_daily"], errors="coerce")
df_reg_se = df_reg_se.dropna(subset=required)

print(f"  Analysis N (final complete cases): {len(df_reg_se):,}")

# -----------------------------
# 5) Logistic regression
# -----------------------------
preds = [
    "age_2026", "male", "edu_total_years", "employed",
    "PHQ4_sum", "behandlung", "diagnose_selbst", "eq5d_daily",
]
formula = "side_effect_any ~ " + " + ".join(preds)
print()
print(f"Model: {formula}")

if len(df_reg_se) < 10:
    print("Too few complete cases to fit model safely.")
elif df_reg_se["side_effect_any"].nunique() < 2:
    print("Outcome has only one class after filtering; model not fit.")
else:
    keep_preds = [p for p in preds if df_reg_se[p].nunique(dropna=True) > 1]
    dropped = [p for p in preds if p not in keep_preds]
    if dropped:
        print("Dropped constant predictor(s): " + ", ".join(dropped))
    if not keep_preds:
        print("All predictors are constant; model not fit.")
    else:
        formula_fit = "side_effect_any ~ " + " + ".join(keep_preds)
        try:
            result_se = smf.logit(formula_fit, data=df_reg_se).fit(disp=0)
            print(result_se.summary())

            coef = result_se.params
            conf = result_se.conf_int()
            conf.columns = ["CI_low", "CI_high"]
            report = pd.DataFrame({
                "Coef": coef,
                "OR": np.exp(coef),
                "OR_95%_CI_low": np.exp(conf["CI_low"]),
                "OR_95%_CI_high": np.exp(conf["CI_high"]),
                "p": result_se.pvalues,
            })
            print()
            print("Odds ratios and 95% CI (side_effect_any model):")
            print(report.round(4).to_string())
        except Exception as e:
            print(f"Model fit failed: {e}")

Per-side-effect Yes/No/Missing counts:
  G2Q5.1 Social alienation            Yes=    43  No= 14545  Missing=   470  Yes%=0.3
  G2Q6 AI dependency                  Yes=  1499  No= 13391  Missing=   168  Yes%=10.1
  G2Q7 Decision difficulty            Yes=   696  No= 14195  Missing=   167  Yes%=4.7

Person-level outcome side_effect_any:
side_effect_any
0.0    13087
1.0     1822
NaN      149

Diagnostic: row loss for extended side_effect_any regression
  Starting N (AI users):           15058
  side_effect_any           valid:  14909  missing:    149  (lost    149)
  age_2026                  valid:  14909  missing:      0  (lost      0)
  male                      valid:  14868  missing:     41  (lost     41)
  edu_total_years           valid:  13554  missing:   1314  (lost   1314)
  employed                  valid:  13534  missing:     20  (lost     20)
  PHQ4_sum                  valid:   5377  missing:   8157  (lost   8157)
  behandlung                valid:   5341  missing:     36  (

### Sensitivity model: side effects (HiTOP instead of PHQ-4)


In [16]:
# Extended regression: sociodemographic + clinical predictors -> side effects (Yes/No)
# CHANGE: PHQ4_sum removed and replaced by HiTOP_GS_sum (HiTOP_1–HiTOP_8 sum; complete-case across items)
# Base sample: AI users only (ai_use == 1)

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# -----------------------------
# 1) Base sample
# -----------------------------
d_sub = df_base_predictors.loc[df_base_predictors["ai_use"] == 1].copy()

# -----------------------------
# 2) Side-effect coding (your exact logic)
# -----------------------------
side_effect_cols = {
    "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner": "G2Q5.1 Social alienation",
    "ki_ai_dependency": "G2Q6 AI dependency",
    "ki_decision_difficulty": "G2Q7 Decision difficulty",
}

for c in side_effect_cols:
    if c not in d_sub.columns:
        print(f"Warning: missing side-effect column: {c}")

SOCIAL_YES_NUM = {-1, -2, -3}
YES_VALUES_TEXT = {
    "trifft ein wenig zu",
    "trifft teilweise zu",
    "trifft völlig zu",
    "trifft voellig zu",
}

def to_yes_no(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    if col == "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner":
        num = pd.to_numeric(s, errors="coerce")
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0

    if s.lower() in YES_VALUES_TEXT:
        return 1
    return 0

for col in side_effect_cols:
    ycol = f"{col}_yes"
    if col in d_sub.columns:
        d_sub[ycol] = d_sub[col].apply(lambda x: to_yes_no(x, col))
    else:
        d_sub[ycol] = np.nan

print("Per-side-effect Yes/No/Missing counts:")
for col, label in side_effect_cols.items():
    ycol = f"{col}_yes"
    n_yes = int((d_sub[ycol] == 1).sum())
    n_no = int((d_sub[ycol] == 0).sum())
    n_miss = int(d_sub[ycol].isna().sum())
    n_ans = n_yes + n_no
    pct_yes = (100 * n_yes / n_ans) if n_ans > 0 else np.nan
    print(f"  {label:<35} Yes={n_yes:>6}  No={n_no:>6}  Missing={n_miss:>6}  Yes%={pct_yes:.1f}" if n_ans > 0
          else f"  {label:<35} Yes={n_yes:>6}  No={n_no:>6}  Missing={n_miss:>6}  Yes%=n/a")

ycols = [f"{c}_yes" for c in side_effect_cols]
answered_any = d_sub[ycols].notna().any(axis=1)
any_yes = d_sub[ycols].eq(1).any(axis=1)
d_sub["side_effect_any"] = np.where(~answered_any, np.nan, np.where(any_yes, 1, 0))

print("\nPerson-level outcome side_effect_any:")
print(d_sub["side_effect_any"].value_counts(dropna=False).to_string())

# -----------------------------
# 3) Additional predictors (HiTOP instead of PHQ-4)
# -----------------------------
# HiTOP GS sum (HiTOP_1–HiTOP_8), a=1..e=5, complete-case across items
letter_to_num = {"a": 1, "b": 2, "c": 3, "d": 4, "e": 5}
hitop_gs_cols = [f"HiTOP_{i}" for i in range(1, 9) if f"HiTOP_{i}" in d_sub.columns]
if len(hitop_gs_cols) != 8:
    print(f"Warning: expected 8 HiTOP columns (HiTOP_1..HiTOP_8), found {len(hitop_gs_cols)}")

hitop_gs_numeric = d_sub[hitop_gs_cols].apply(
    lambda col: col.astype(str).str.strip().str.lower().map(letter_to_num)
)
d_sub["HiTOP_GS_sum"] = hitop_gs_numeric.sum(axis=1)
mask_hitop_complete = hitop_gs_numeric.notna().all(axis=1)
d_sub.loc[~mask_hitop_complete, "HiTOP_GS_sum"] = np.nan

# Behandlung_dich: a=1, b=0, else NaN
bh = d_sub["Behandlung_dich"].astype(str).str.strip().str.lower()
d_sub["behandlung"] = np.where(bh == "a", 1, np.where(bh == "b", 0, np.nan))

# Diagnose_selbst_dich: a=1, b=0, else NaN
ds = d_sub["Diagnose_selbst_dich"].astype(str).str.strip().str.lower()
d_sub["diagnose_selbst"] = np.where(ds == "a", 1, np.where(ds == "b", 0, np.nan))

# EQ5D5L_3: a=1 ... e=5, else NaN
eq_map = {"a": 1, "b": 2, "c": 3, "d": 4, "e": 5}
eq_raw = d_sub["EQ5D5L_3"].astype(str).str.strip().str.lower()
d_sub["eq5d_daily"] = eq_raw.map(eq_map)

# -----------------------------
# 4) Complete-case regression sample
# -----------------------------
required = [
    "side_effect_any",
    "age_2026", "male", "edu_total_years", "employed",
    "HiTOP_GS_sum", "behandlung", "diagnose_selbst", "eq5d_daily",
]

print("\nDiagnostic: row loss for extended side_effect_any regression (HiTOP)")
print(f"  Starting N (AI users):          {len(d_sub):>6}")
remaining = d_sub.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")
print(f"  Final complete cases:           {len(remaining):>6}")
print()

df_reg_se = d_sub.dropna(subset=required).copy()

# type safety
df_reg_se["side_effect_any"] = df_reg_se["side_effect_any"].astype(int)
df_reg_se["male"] = df_reg_se["male"].astype(int)
df_reg_se["employed"] = df_reg_se["employed"].astype(int)
df_reg_se["behandlung"] = df_reg_se["behandlung"].astype(int)
df_reg_se["diagnose_selbst"] = df_reg_se["diagnose_selbst"].astype(int)

df_reg_se["age_2026"] = pd.to_numeric(df_reg_se["age_2026"], errors="coerce")
df_reg_se["edu_total_years"] = pd.to_numeric(df_reg_se["edu_total_years"], errors="coerce")
df_reg_se["HiTOP_GS_sum"] = pd.to_numeric(df_reg_se["HiTOP_GS_sum"], errors="coerce")
df_reg_se["eq5d_daily"] = pd.to_numeric(df_reg_se["eq5d_daily"], errors="coerce")
df_reg_se = df_reg_se.dropna(subset=required)

print(f"  Analysis N (final complete cases): {len(df_reg_se):,}")

# -----------------------------
# 5) Logistic regression
# -----------------------------
preds = [
    "age_2026", "male", "edu_total_years", "employed",
    "HiTOP_GS_sum", "behandlung", "diagnose_selbst", "eq5d_daily",
]
formula = "side_effect_any ~ " + " + ".join(preds)
print()
print(f"Model: {formula}")

if len(df_reg_se) < 10:
    print("Too few complete cases to fit model safely.")
elif df_reg_se["side_effect_any"].nunique() < 2:
    print("Outcome has only one class after filtering; model not fit.")
else:
    keep_preds = [p for p in preds if df_reg_se[p].nunique(dropna=True) > 1]
    dropped = [p for p in preds if p not in keep_preds]
    if dropped:
        print("Dropped constant predictor(s): " + ", ".join(dropped))
    if not keep_preds:
        print("All predictors are constant; model not fit.")
    else:
        formula_fit = "side_effect_any ~ " + " + ".join(keep_preds)
        try:
            result_se = smf.logit(formula_fit, data=df_reg_se).fit(disp=0)
            print(result_se.summary())

            coef = result_se.params
            conf = result_se.conf_int()
            conf.columns = ["CI_low", "CI_high"]
            report = pd.DataFrame({
                "Coef": coef,
                "OR": np.exp(coef),
                "OR_95%_CI_low": np.exp(conf["CI_low"]),
                "OR_95%_CI_high": np.exp(conf["CI_high"]),
                "p": result_se.pvalues,
            })
            print()
            print("Odds ratios and 95% CI (side_effect_any model, HiTOP):")
            print(report.round(4).to_string())
        except Exception as e:
            print(f"Model fit failed: {e}")

Per-side-effect Yes/No/Missing counts:
  G2Q5.1 Social alienation            Yes=    43  No= 14545  Missing=   470  Yes%=0.3
  G2Q6 AI dependency                  Yes=  1499  No= 13391  Missing=   168  Yes%=10.1
  G2Q7 Decision difficulty            Yes=   696  No= 14195  Missing=   167  Yes%=4.7

Person-level outcome side_effect_any:
side_effect_any
0.0    13087
1.0     1822
NaN      149

Diagnostic: row loss for extended side_effect_any regression (HiTOP)
  Starting N (AI users):           15058
  side_effect_any           valid:  14909  missing:    149  (lost    149)
  age_2026                  valid:  14909  missing:      0  (lost      0)
  male                      valid:  14868  missing:     41  (lost     41)
  edu_total_years           valid:  13554  missing:   1314  (lost   1314)
  employed                  valid:  13534  missing:     20  (lost     20)
  HiTOP_GS_sum              valid:   5392  missing:   8142  (lost   8142)
  behandlung                valid:   5357  missing:  

### Sensitivity model: side effects (number of diagnoses)


In [17]:
# Extended regression: sociodemographic + clinical predictors -> side effects (Yes/No)
# CHANGE: binary diagnose_selbst replaced by n_diagnoses (count of Diagnose_* == 1 when self-report Ja)
# Base sample: AI users only (ai_use == 1)

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# -----------------------------
# 1) Base sample
# -----------------------------
d_sub = df_base_predictors.loc[df_base_predictors["ai_use"] == 1].copy()

# -----------------------------
# 2) Side-effect coding (your exact logic)
# -----------------------------
side_effect_cols = {
    "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner": "G2Q5.1 Social alienation",
    "ki_ai_dependency": "G2Q6 AI dependency",
    "ki_decision_difficulty": "G2Q7 Decision difficulty",
}

for c in side_effect_cols:
    if c not in d_sub.columns:
        print(f"Warning: missing side-effect column: {c}")

SOCIAL_YES_NUM = {-1, -2, -3}
YES_VALUES_TEXT = {
    "trifft ein wenig zu",
    "trifft teilweise zu",
    "trifft völlig zu",
    "trifft voellig zu",
}

def to_yes_no(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    if col == "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner":
        num = pd.to_numeric(s, errors="coerce")
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0

    if s.lower() in YES_VALUES_TEXT:
        return 1
    return 0

for col in side_effect_cols:
    ycol = f"{col}_yes"
    if col in d_sub.columns:
        d_sub[ycol] = d_sub[col].apply(lambda x: to_yes_no(x, col))
    else:
        d_sub[ycol] = np.nan

print("Per-side-effect Yes/No/Missing counts:")
for col, label in side_effect_cols.items():
    ycol = f"{col}_yes"
    n_yes = int((d_sub[ycol] == 1).sum())
    n_no = int((d_sub[ycol] == 0).sum())
    n_miss = int(d_sub[ycol].isna().sum())
    n_ans = n_yes + n_no
    pct_yes = (100 * n_yes / n_ans) if n_ans > 0 else np.nan
    print(f"  {label:<35} Yes={n_yes:>6}  No={n_no:>6}  Missing={n_miss:>6}  Yes%={pct_yes:.1f}" if n_ans > 0
          else f"  {label:<35} Yes={n_yes:>6}  No={n_no:>6}  Missing={n_miss:>6}  Yes%=n/a")

ycols = [f"{c}_yes" for c in side_effect_cols]
answered_any = d_sub[ycols].notna().any(axis=1)
any_yes = d_sub[ycols].eq(1).any(axis=1)
d_sub["side_effect_any"] = np.where(~answered_any, np.nan, np.where(any_yes, 1, 0))

print("\nPerson-level outcome side_effect_any:")
print(d_sub["side_effect_any"].value_counts(dropna=False).to_string())

# -----------------------------
# 3) Additional predictors (PHQ-4 unchanged; diagnosis -> n_diagnoses)
# -----------------------------
# PHQ-4: RECODE 1-4 -> 0-3, then sum only if all 4 items are present/valid
phq4_cols = ["PHQ4_1", "PHQ4_2", "PHQ4_3", "PHQ4_4"]
phq4_raw = d_sub[phq4_cols].apply(pd.to_numeric, errors="coerce")
phq4_recoded = phq4_raw.replace({1: 0, 2: 1, 3: 2, 4: 3})
phq4_recoded = phq4_recoded.where(phq4_raw.isin([1, 2, 3, 4]), np.nan)

mask_phq4_complete = phq4_recoded.notna().all(axis=1)
d_sub["PHQ4_sum"] = phq4_recoded.sum(axis=1)
d_sub.loc[~mask_phq4_complete, "PHQ4_sum"] = np.nan

# Behandlung_dich: a=1, b=0, else NaN
bh = d_sub["Behandlung_dich"].astype(str).str.strip().str.lower()
d_sub["behandlung"] = np.where(bh == "a", 1, np.where(bh == "b", 0, np.nan))

# n_diagnoses: Ja -> sum Diagnose_*==1; answered but not Ja -> 0; blank -> NaN
diagnose_selbst_col = "Diagnose_selbst_dich"
diagnose_cols = [
    "Diagnose_PD", "Diagnose_DD", "Diagnose_BD", "Diagnose_AD", "Diagnose_OCD",
    "Diagnose_PTSD", "Diagnose_SD", "Diagnose_ED", "Diagnose_PBD", "Diagnose_DMD",
    "Diagnose_ADHD", "Diagnose_ASD", "Diagnose_SUD", "Diagnose_D", "Diagnose_andere",
]

ds_self = d_sub[diagnose_selbst_col]
ds_stripped = ds_self.astype(str).str.strip()
ds_lower = ds_stripped.str.lower()
is_blank = ds_self.isna() | (ds_stripped == "") | ds_lower.isin(["nan", "none"])
mask_yes = ds_lower == "a"
mask_other_answered = (~is_blank) & (~mask_yes)

d_sub["n_diagnoses"] = np.nan
d_sub.loc[mask_other_answered, "n_diagnoses"] = 0

cols_ok = [c for c in diagnose_cols if c in d_sub.columns]
missing_diag_cols = [c for c in diagnose_cols if c not in d_sub.columns]
if missing_diag_cols:
    print(f"Note: missing Diagnose_* columns ignored for n_diagnoses: {missing_diag_cols}")

if cols_ok and mask_yes.any():
    blk = d_sub.loc[mask_yes, cols_ok]
    present = pd.DataFrame(
        {
            c: (pd.to_numeric(blk[c], errors="coerce") == 1)
            | (blk[c].astype(str).str.strip() == "1")
            for c in cols_ok
        },
        index=blk.index,
    )
    d_sub.loc[mask_yes, "n_diagnoses"] = present.sum(axis=1)

# EQ5D5L_3: a=1 ... e=5, else NaN
eq_map = {"a": 1, "b": 2, "c": 3, "d": 4, "e": 5}
eq_raw = d_sub["EQ5D5L_3"].astype(str).str.strip().str.lower()
d_sub["eq5d_daily"] = eq_raw.map(eq_map)

# -----------------------------
# 4) Complete-case regression sample
# -----------------------------
required = [
    "side_effect_any",
    "age_2026", "male", "edu_total_years", "employed",
    "PHQ4_sum", "behandlung", "n_diagnoses", "eq5d_daily",
]

print("\nDiagnostic: row loss for extended side_effect_any regression (n_diagnoses)")
print(f"  Starting N (AI users):          {len(d_sub):>6}")
remaining = d_sub.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")
print(f"  Final complete cases:           {len(remaining):>6}")
print()

df_reg_se = d_sub.dropna(subset=required).copy()

# type safety
df_reg_se["side_effect_any"] = df_reg_se["side_effect_any"].astype(int)
df_reg_se["male"] = df_reg_se["male"].astype(int)
df_reg_se["employed"] = df_reg_se["employed"].astype(int)
df_reg_se["behandlung"] = df_reg_se["behandlung"].astype(int)

df_reg_se["age_2026"] = pd.to_numeric(df_reg_se["age_2026"], errors="coerce")
df_reg_se["edu_total_years"] = pd.to_numeric(df_reg_se["edu_total_years"], errors="coerce")
df_reg_se["PHQ4_sum"] = pd.to_numeric(df_reg_se["PHQ4_sum"], errors="coerce")
df_reg_se["n_diagnoses"] = pd.to_numeric(df_reg_se["n_diagnoses"], errors="coerce")
df_reg_se["eq5d_daily"] = pd.to_numeric(df_reg_se["eq5d_daily"], errors="coerce")
df_reg_se = df_reg_se.dropna(subset=required)

print(f"  Analysis N (final complete cases): {len(df_reg_se):,}")

# -----------------------------
# 5) Logistic regression
# -----------------------------
preds = [
    "age_2026", "male", "edu_total_years", "employed",
    "PHQ4_sum", "behandlung", "n_diagnoses", "eq5d_daily",
]
formula = "side_effect_any ~ " + " + ".join(preds)
print()
print(f"Model: {formula}")

if len(df_reg_se) < 10:
    print("Too few complete cases to fit model safely.")
elif df_reg_se["side_effect_any"].nunique() < 2:
    print("Outcome has only one class after filtering; model not fit.")
else:
    keep_preds = [p for p in preds if df_reg_se[p].nunique(dropna=True) > 1]
    dropped = [p for p in preds if p not in keep_preds]
    if dropped:
        print("Dropped constant predictor(s): " + ", ".join(dropped))
    if not keep_preds:
        print("All predictors are constant; model not fit.")
    else:
        formula_fit = "side_effect_any ~ " + " + ".join(keep_preds)
        try:
            result_se = smf.logit(formula_fit, data=df_reg_se).fit(disp=0)
            print(result_se.summary())

            coef = result_se.params
            conf = result_se.conf_int()
            conf.columns = ["CI_low", "CI_high"]
            report = pd.DataFrame({
                "Coef": coef,
                "OR": np.exp(coef),
                "OR_95%_CI_low": np.exp(conf["CI_low"]),
                "OR_95%_CI_high": np.exp(conf["CI_high"]),
                "p": result_se.pvalues,
            })
            print()
            print("Odds ratios and 95% CI (side_effect_any model):")
            print(report.round(4).to_string())
        except Exception as e:
            print(f"Model fit failed: {e}")

Per-side-effect Yes/No/Missing counts:
  G2Q5.1 Social alienation            Yes=    43  No= 14545  Missing=   470  Yes%=0.3
  G2Q6 AI dependency                  Yes=  1499  No= 13391  Missing=   168  Yes%=10.1
  G2Q7 Decision difficulty            Yes=   696  No= 14195  Missing=   167  Yes%=4.7

Person-level outcome side_effect_any:
side_effect_any
0.0    13087
1.0     1822
NaN      149

Diagnostic: row loss for extended side_effect_any regression (n_diagnoses)
  Starting N (AI users):           15058
  side_effect_any           valid:  14909  missing:    149  (lost    149)
  age_2026                  valid:  14909  missing:      0  (lost      0)
  male                      valid:  14868  missing:     41  (lost     41)
  edu_total_years           valid:  13554  missing:   1314  (lost   1314)
  employed                  valid:  13534  missing:     20  (lost     20)
  PHQ4_sum                  valid:   5377  missing:   8157  (lost   8157)
  behandlung                valid:   5341  miss

## Mediation analyses (if applicable)

### Mental health-related use (mediator), sociodemographic model


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# =========================================================
# MEDIATION ANALYSIS (with bootstrap)
# X = age_2026, male, edu_total_years, employed
# M = emo_use (Q4.4–Q4.13, INCLUDING "Sonstiges" Q4.13)
# Y = side_effect_any
# Base: AI users only
# =========================================================

# -----------------------------
# 1) Base sample
# -----------------------------
d = df_base_predictors.loc[df_base_predictors["ai_use"] == 1].copy()

# -----------------------------
# 2) Build mediator: emo_use (Q4.4–Q4.13, incl. Q4.13)
# -----------------------------
purpose_cols_4_13 = [
    'ki_bei_aengsten_und_sorgen',                              # Q4.4
    'ki_bei_selbstzweifeln_oder_niedergeschlagenheit',        # Q4.5
    'ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we',# Q4.6
    'ki_bei_zwischenmenschlichen_konflikten',                 # Q4.7
    'ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan',# Q4.8
    'ki_bei_trauer_oder_verlust',                             # Q4.9
    'ki_wenn_grosse_lebensveraenderungen_anstehen',           # Q4.10
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4', # Q4.11
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5', # Q4.12
    'ki_fuer_sonstige_gesundheits_psychische_oder_emotionale',   # Q4.13 (Sonstiges)
]
purpose_cols_4_13 = [c for c in purpose_cols_4_13 if c in d.columns]

if len(purpose_cols_4_13) == 0:
    raise KeyError("No emotional-use columns found for mediator coding.")

non_use_answers = {
    'hierfür nutze ich keine ki',
    '(bisher) kein bedarf, würde ki aber dafür nutzen',
}

def norm_str(x):
    if pd.isna(x):
        return ''
    s = str(x).strip().lower()
    return '' if s in ['', 'nan'] else s

emo_vals = []
for _, row in d[purpose_cols_4_13].iterrows():
    vals = [norm_str(row[c]) for c in purpose_cols_4_13]
    answered = [v for v in vals if v != '']

    if len(answered) == 0:
        emo_vals.append(np.nan)
    elif any(v not in non_use_answers for v in answered):
        emo_vals.append(1)
    else:
        emo_vals.append(0)

d["emo_use"] = emo_vals

print("Mediator (emo_use) distribution:")
print(d["emo_use"].value_counts(dropna=False).to_string())

# -----------------------------
# 3) Build outcome: side_effect_any (exact prior coding)
# -----------------------------
side_effect_cols = {
    "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner": "G2Q5.1 Social alienation",
    "ki_ai_dependency": "G2Q6 AI dependency",
    "ki_decision_difficulty": "G2Q7 Decision difficulty",
}

SOCIAL_YES_NUM = {-1, -2, -3}
YES_VALUES_TEXT = {
    "trifft ein wenig zu",
    "trifft teilweise zu",
    "trifft völlig zu",
    "trifft voellig zu",
}

def to_yes_no(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    if col == "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner":
        num = pd.to_numeric(s, errors="coerce")
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0

    if s.lower() in YES_VALUES_TEXT:
        return 1
    return 0

for col in side_effect_cols:
    ycol = f"{col}_yes"
    if col in d.columns:
        d[ycol] = d[col].apply(lambda x: to_yes_no(x, col))
    else:
        d[ycol] = np.nan

ycols = [f"{c}_yes" for c in side_effect_cols]
answered_any = d[ycols].notna().any(axis=1)
any_yes = d[ycols].eq(1).any(axis=1)
d["side_effect_any"] = np.where(~answered_any, np.nan, np.where(any_yes, 1, 0))

print("\nPerson-level outcome side_effect_any:")
print(d["side_effect_any"].value_counts(dropna=False).to_string())

# -----------------------------
# 4) Common complete-case sample
# -----------------------------
X = ["age_2026", "male", "edu_total_years", "employed"]
needed = ["emo_use", "side_effect_any"] + X

dfm = d.dropna(subset=needed).copy()
dfm["emo_use"] = pd.to_numeric(dfm["emo_use"], errors="coerce").astype(int)
dfm["side_effect_any"] = pd.to_numeric(dfm["side_effect_any"], errors="coerce").astype(int)
dfm["male"] = pd.to_numeric(dfm["male"], errors="coerce").astype(int)
dfm["employed"] = pd.to_numeric(dfm["employed"], errors="coerce").astype(int)
dfm["age_2026"] = pd.to_numeric(dfm["age_2026"], errors="coerce")
dfm["edu_total_years"] = pd.to_numeric(dfm["edu_total_years"], errors="coerce")
dfm = dfm.dropna(subset=needed)

print(f"\nCommon complete-case N: {len(dfm):,}")

# -----------------------------
# 5) Main mediation models
# -----------------------------
model_a = smf.logit("emo_use ~ age_2026 + male + edu_total_years + employed", data=dfm).fit(disp=0)
model_c = smf.logit("side_effect_any ~ age_2026 + male + edu_total_years + employed", data=dfm).fit(disp=0)
model_cp = smf.logit("side_effect_any ~ age_2026 + male + edu_total_years + employed + emo_use", data=dfm).fit(disp=0)

print("\n=== Path a: emo_use ~ X ===")
print(model_a.summary())
print("\n=== Total effect c: side_effect_any ~ X ===")
print(model_c.summary())
print("\n=== Direct effect c' (+ b): side_effect_any ~ X + emo_use ===")
print(model_cp.summary())

# Comparison table c vs c'
rows = []
for p in X:
    c = model_c.params.get(p, np.nan)
    cp = model_cp.params.get(p, np.nan)
    rows.append({
        "predictor": p,
        "coef_c_total": c,
        "OR_c_total": np.exp(c),
        "p_c_total": model_c.pvalues.get(p, np.nan),
        "coef_cprime_direct": cp,
        "OR_cprime_direct": np.exp(cp),
        "p_cprime_direct": model_cp.pvalues.get(p, np.nan),
        "delta_coef_c_minus_cprime": c - cp
    })

rows.append({
    "predictor": "emo_use (b path)",
    "coef_c_total": np.nan,
    "OR_c_total": np.nan,
    "p_c_total": np.nan,
    "coef_cprime_direct": model_cp.params.get("emo_use", np.nan),
    "OR_cprime_direct": np.exp(model_cp.params.get("emo_use", np.nan)),
    "p_cprime_direct": model_cp.pvalues.get("emo_use", np.nan),
    "delta_coef_c_minus_cprime": np.nan
})

compare = pd.DataFrame(rows)
print("\n=== Mediation-style comparison ===")
print(compare.round(4).to_string(index=False))

# -----------------------------
# 6) Bootstrap indirect effects (a*b)
# -----------------------------
B = 2000
SEED = 42
rng = np.random.default_rng(SEED)

ab_store = {p: [] for p in X}
ok = 0
fail = 0

for _ in range(B):
    idx = rng.integers(0, len(dfm), len(dfm))
    bs = dfm.iloc[idx]

    try:
        a_fit = smf.logit("emo_use ~ age_2026 + male + edu_total_years + employed", data=bs).fit(disp=0)
        cp_fit = smf.logit("side_effect_any ~ age_2026 + male + edu_total_years + employed + emo_use", data=bs).fit(disp=0)

        b_coef = cp_fit.params.get("emo_use", np.nan)

        for p in X:
            a_coef = a_fit.params.get(p, np.nan)
            ab_store[p].append(a_coef * b_coef)

        ok += 1
    except Exception:
        for p in X:
            ab_store[p].append(np.nan)
        fail += 1

boot_rows = []
for p in X:
    vals = np.array(ab_store[p], dtype=float)
    vals = vals[~np.isnan(vals)]
    if len(vals) == 0:
        boot_rows.append({
            "predictor": p,
            "ab_mean": np.nan,
            "ab_median": np.nan,
            "ab_ci_2.5": np.nan,
            "ab_ci_97.5": np.nan,
            "n_boot_ok_for_predictor": 0
        })
    else:
        boot_rows.append({
            "predictor": p,
            "ab_mean": vals.mean(),
            "ab_median": np.median(vals),
            "ab_ci_2.5": np.quantile(vals, 0.025),
            "ab_ci_97.5": np.quantile(vals, 0.975),
            "n_boot_ok_for_predictor": len(vals)
        })

boot_table = pd.DataFrame(boot_rows)

print("\n=== Bootstrap summary ===")
print(f"Requested draws: {B}")
print(f"Successful draws: {ok}")
print(f"Failed draws: {fail}")

print("\n=== Indirect effects (a*b) with bootstrap 95% CI ===")
print(boot_table.round(4).to_string(index=False))
print("\nInterpretation hint: CI excluding 0 suggests an indirect effect for that predictor.")

### Mental health-related use (mediator), extended model with clinical predictors


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# =========================================================
# MEDIATION ANALYSIS (with bootstrap) -- INCLUDING Q4.13
# X = sociodemographic + clinical predictors
# M = emo_use (Q4.4-Q4.13, incl. "Sonstiges")
# Y = side_effect_any
# Base: AI users only
# =========================================================

# -----------------------------
# 1) Base sample
# -----------------------------
d = df_base_predictors.loc[df_base_predictors["ai_use"] == 1].copy()

# -----------------------------
# 2) Build mediator: emo_use (Q4.4-Q4.13, incl. Q4.13)
# -----------------------------
purpose_cols_4_13 = [
    "ki_bei_aengsten_und_sorgen",
    "ki_bei_selbstzweifeln_oder_niedergeschlagenheit",
    "ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we",
    "ki_bei_zwischenmenschlichen_konflikten",
    "ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan",
    "ki_bei_trauer_oder_verlust",
    "ki_wenn_grosse_lebensveraenderungen_anstehen",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5",
    "ki_fuer_sonstige_gesundheits_psychische_oder_emotionale",  # Q4.13 INCLUDED
]
purpose_cols_4_13 = [c for c in purpose_cols_4_13 if c in d.columns]

non_use_answers = {
    "hierfür nutze ich keine ki",
    "(bisher) kein bedarf, würde ki aber dafür nutzen",
}
non_use_answers = {x.strip().lower() for x in non_use_answers}

def norm_str(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    return "" if s in {"", "nan"} else s

emo_vals = []
for _, row in d[purpose_cols_4_13].iterrows():
    vals = [norm_str(row[c]) for c in purpose_cols_4_13]
    answered = [v for v in vals if v != ""]
    if len(answered) == 0:
        emo_vals.append(np.nan)
    elif any(v not in non_use_answers for v in answered):
        emo_vals.append(1)
    else:
        emo_vals.append(0)

d["emo_use"] = emo_vals

# -----------------------------
# 3) Build outcome: side_effect_any
# -----------------------------
side_effect_cols = {
    "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner": "G2Q5.1 Social alienation",
    "ki_ai_dependency": "G2Q6 AI dependency",
    "ki_decision_difficulty": "G2Q7 Decision difficulty",
}

SOCIAL_YES_NUM = {-1, -2, -3}
YES_VALUES_TEXT = {
    "trifft ein wenig zu",
    "trifft teilweise zu",
    "trifft völlig zu",
    "trifft voellig zu",
}

def to_yes_no(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    if col == "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner":
        num = pd.to_numeric(s, errors="coerce")
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0

    return 1 if s.lower() in YES_VALUES_TEXT else 0

for col in side_effect_cols:
    ycol = f"{col}_yes"
    d[ycol] = d[col].apply(lambda x: to_yes_no(x, col)) if col in d.columns else np.nan

ycols = [f"{c}_yes" for c in side_effect_cols]
answered_any = d[ycols].notna().any(axis=1)
any_yes = d[ycols].eq(1).any(axis=1)
d["side_effect_any"] = np.where(~answered_any, np.nan, np.where(any_yes, 1, 0))

# -----------------------------
# 4) Build additional predictors
# -----------------------------
phq4_cols = ["PHQ4_1", "PHQ4_2", "PHQ4_3", "PHQ4_4"]
phq4_raw = d[phq4_cols].apply(pd.to_numeric, errors="coerce")
phq4_recoded = phq4_raw.replace({1: 0, 2: 1, 3: 2, 4: 3})
phq4_recoded = phq4_recoded.where(phq4_raw.isin([1, 2, 3, 4]), np.nan)

mask_phq4_complete = phq4_recoded.notna().all(axis=1)
d["PHQ4_sum"] = phq4_recoded.sum(axis=1)
d.loc[~mask_phq4_complete, "PHQ4_sum"] = np.nan

bh = d["Behandlung_dich"].astype(str).str.strip().str.lower()
d["behandlung"] = np.where(bh == "a", 1, np.where(bh == "b", 0, np.nan))

ds = d["Diagnose_selbst_dich"].astype(str).str.strip().str.lower()
d["diagnose_selbst"] = np.where(ds == "a", 1, np.where(ds == "b", 0, np.nan))

eq_map = {"a": 1, "b": 2, "c": 3, "d": 4, "e": 5}
eq_raw = d["EQ5D5L_3"].astype(str).str.strip().str.lower()
d["eq5d_daily"] = eq_raw.map(eq_map)

# -----------------------------
# 5) Common complete-case sample
# -----------------------------
X = [
    "age_2026", "male", "edu_total_years", "employed",
    "PHQ4_sum", "behandlung", "diagnose_selbst", "eq5d_daily"
]
needed = ["emo_use", "side_effect_any"] + X

dfm = d.dropna(subset=needed).copy()

dfm["emo_use"] = dfm["emo_use"].astype(int)
dfm["side_effect_any"] = dfm["side_effect_any"].astype(int)
dfm["male"] = dfm["male"].astype(int)
dfm["employed"] = dfm["employed"].astype(int)
dfm["behandlung"] = dfm["behandlung"].astype(int)
dfm["diagnose_selbst"] = dfm["diagnose_selbst"].astype(int)

for c in ["age_2026", "edu_total_years", "PHQ4_sum", "eq5d_daily"]:
    dfm[c] = pd.to_numeric(dfm[c], errors="coerce")
dfm = dfm.dropna(subset=needed)

print(f"Common complete-case N: {len(dfm):,}")

# -----------------------------
# 6) Main mediation models
# -----------------------------
x_formula = " + ".join(X)

model_a = smf.logit(f"emo_use ~ {x_formula}", data=dfm).fit(disp=0)
model_c = smf.logit(f"side_effect_any ~ {x_formula}", data=dfm).fit(disp=0)
model_cp = smf.logit(f"side_effect_any ~ {x_formula} + emo_use", data=dfm).fit(disp=0)

print("\n=== Path a: emo_use ~ X ===")
print(model_a.summary())
print("\n=== Total effect c: side_effect_any ~ X ===")
print(model_c.summary())
print("\n=== Direct effect c' (+ b): side_effect_any ~ X + emo_use ===")
print(model_cp.summary())

rows = []
for p in X:
    c = model_c.params.get(p, np.nan)
    cp = model_cp.params.get(p, np.nan)
    rows.append({
        "predictor": p,
        "coef_c_total": c,
        "OR_c_total": np.exp(c),
        "p_c_total": model_c.pvalues.get(p, np.nan),
        "coef_cprime_direct": cp,
        "OR_cprime_direct": np.exp(cp),
        "p_cprime_direct": model_cp.pvalues.get(p, np.nan),
        "delta_coef_c_minus_cprime": c - cp
    })

rows.append({
    "predictor": "emo_use (b path)",
    "coef_c_total": np.nan,
    "OR_c_total": np.nan,
    "p_c_total": np.nan,
    "coef_cprime_direct": model_cp.params.get("emo_use", np.nan),
    "OR_cprime_direct": np.exp(model_cp.params.get("emo_use", np.nan)),
    "p_cprime_direct": model_cp.pvalues.get("emo_use", np.nan),
    "delta_coef_c_minus_cprime": np.nan
})

compare = pd.DataFrame(rows)
print("\n=== Mediation-style comparison ===")
print(compare.round(4).to_string(index=False))

# -----------------------------
# 7) Bootstrap indirect effects (a*b)
# -----------------------------
B = 2000
SEED = 42
rng = np.random.default_rng(SEED)

ab_store = {p: [] for p in X}
ok, fail = 0, 0

for _ in range(B):
    idx = rng.integers(0, len(dfm), len(dfm))
    bs = dfm.iloc[idx]

    try:
        a_fit = smf.logit(f"emo_use ~ {x_formula}", data=bs).fit(disp=0)
        cp_fit = smf.logit(f"side_effect_any ~ {x_formula} + emo_use", data=bs).fit(disp=0)

        b_coef = cp_fit.params.get("emo_use", np.nan)
        for p in X:
            a_coef = a_fit.params.get(p, np.nan)
            ab_store[p].append(a_coef * b_coef)

        ok += 1
    except Exception:
        for p in X:
            ab_store[p].append(np.nan)
        fail += 1

boot_rows = []
for p in X:
    vals = np.array(ab_store[p], dtype=float)
    vals = vals[~np.isnan(vals)]
    if len(vals) == 0:
        boot_rows.append({
            "predictor": p, "ab_mean": np.nan, "ab_median": np.nan,
            "ab_ci_2.5": np.nan, "ab_ci_97.5": np.nan, "n_boot_ok_for_predictor": 0
        })
    else:
        boot_rows.append({
            "predictor": p,
            "ab_mean": vals.mean(),
            "ab_median": np.median(vals),
            "ab_ci_2.5": np.quantile(vals, 0.025),
            "ab_ci_97.5": np.quantile(vals, 0.975),
            "n_boot_ok_for_predictor": len(vals)
        })

boot_table = pd.DataFrame(boot_rows)

print("\n=== Bootstrap summary ===")
print(f"Requested draws: {B}")
print(f"Successful draws: {ok}")
print(f"Failed draws: {fail}")

print("\n=== Indirect effects (a*b) with bootstrap 95% CI ===")
print(boot_table.round(4).to_string(index=False))
print("\nInterpretation hint: CI excluding 0 suggests an indirect effect for that predictor.")

## Intensity (dose–response) analyses

These models use intensity scores (e.g., summed/mean frequency across items) among relevant user subsamples.


### General health-related use intensity: sociodemographic predictors


In [20]:
# -----------------------------
# 6) Intensity analysis among medical AI users (items 1-3)
# Outcome: SUM intensity score across the 3 medical-use items
#   Selten=1, Manchmal=2, Oft=3, Immer=4
#   Non-medical / blank answers are coded as 0 for this score.
#   Score = (item1 + item2 + item3)
# Include only participants with at least one medical-use answer and complete predictors.
# -----------------------------

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

purpose_cols_13 = [
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3',
]
purpose_cols_13 = [c for c in purpose_cols_13 if c in df_base_predictors.columns]

if len(purpose_cols_13) != 3:
    print('Warning: expected 3 medical-use columns (items 1-3), found:', len(purpose_cols_13))

# Start from same base inclusion as your med-use model
# (same participant inclusion mechanism as AI-use model, then AI users only)
df_med_int = df_base_predictors.loc[df_base_predictors['ai_use'] == 1].copy()


def norm_txt(x):
    if pd.isna(x):
        return ''
    s = str(x).strip().lower()
    if s == 'nan':
        return ''
    s = s.replace('ä', 'ae').replace('ö', 'oe').replace('ü', 'ue').replace('ß', 'ss')
    return s


intensity_map = {
    'selten': 1,
    'manchmal': 2,
    'oft': 3,
    'immer': 4,
}

non_medical_answers = {
    'hierfuer nutze ich keine ki',
    '(bisher) kein bedarf, wuerde ki aber dafuer nutzen',
}


def map_intensity(v):
    s = norm_txt(v)
    if s == '':
        return 0
    if s in non_medical_answers:
        return 0
    if s in intensity_map:
        return intensity_map[s]

    # Slightly robust fallback if extra wording appears
    for k, val in intensity_map.items():
        if k in s:
            return val

    return np.nan


if len(purpose_cols_13) > 0:
    for c in purpose_cols_13:
        df_med_int[f'{c}_int'] = df_med_int[c].apply(map_intensity)

    int_cols = [f'{c}_int' for c in purpose_cols_13]

    # Medical users for this analysis = at least one medical-use intensity response (1..4)
    has_medical_use = df_med_int[int_cols].apply(lambda r: any(pd.notna(v) and v > 0 for v in r), axis=1)

    # Keep rows where all mapped intensity values are known (0..4), then compute score
    known_intensity = df_med_int[int_cols].notna().all(axis=1)
    df_med_int['med_use_intensity_sum'] = df_med_int[int_cols].sum(axis=1)
    # Use sum score directly (no division)
    df_med_int['med_use_intensity_score'] = df_med_int['med_use_intensity_sum']

    cols_needed = ['med_use_intensity_score', 'age_2026', 'male', 'edu_total_years', 'employed']
    df_reg_int = df_med_int.loc[has_medical_use & known_intensity].copy()
    df_reg_int = df_reg_int.dropna(subset=cols_needed)

    # Ensure numeric dtypes
    for c in cols_needed:
        df_reg_int[c] = pd.to_numeric(df_reg_int[c], errors='coerce')
    df_reg_int = df_reg_int.dropna(subset=cols_needed)

    df_reg_int['male'] = df_reg_int['male'].astype(int)
    df_reg_int['employed'] = df_reg_int['employed'].astype(int)

    print()
    print('Intensity regression among medical AI users: med_use_intensity_score ~ age_2026 + male + edu_total_years + employed')
    print('  Coding: Selten=1, Manchmal=2, Oft=3, Immer=4; Non-medical/blank=0; score=sum(items1-3)')

    # Overview of intensity categories before regression
    label_map = {1: 'Selten', 2: 'Manchmal', 3: 'Oft', 4: 'Immer'}

    # pooled across the 3 items, only among analysis sample
    pooled = df_reg_int[int_cols].stack()
    pooled = pooled[pooled.isin([1, 2, 3, 4])]
    counts = pooled.value_counts().sort_index()

    print('\nIntensity response frequencies (pooled across items 1-3):')
    for k in [1, 2, 3, 4]:
        print(f"  {label_map[k]}: {int(counts.get(k, 0))}")

    # optional: per-item counts
    print('\nPer-item intensity frequencies:')
    for c in int_cols:
        vc = df_reg_int[c].value_counts().sort_index()
        print(f'  {c}: ' + ', '.join([f"{label_map[k]}={int(vc.get(k, 0))}" for k in [1, 2, 3, 4]]))

    print(f'\n  N (complete cases): {len(df_reg_int)}')

    if len(df_reg_int) < 10:
        print('  Too few complete cases to fit model safely.')
    else:
        print('  Outcome descriptives:')
        print(df_reg_int['med_use_intensity_score'].describe().round(3).to_string())

        preds_int = ['age_2026', 'male', 'edu_total_years', 'employed']
        keep_int = [p for p in preds_int if df_reg_int[p].nunique(dropna=True) > 1]
        drop_int = [p for p in preds_int if p not in keep_int]

        if drop_int:
            print('  Dropped constant predictor(s): ' + ', '.join(drop_int))

        if not keep_int:
            print('  All predictors are constant; model not fit.')
        else:
            formula_int = 'med_use_intensity_score ~ ' + ' + '.join(keep_int)
            print(f'  Model: {formula_int}')
            try:
                # OLS for continuous intensity score
                result_int = smf.ols(formula_int, data=df_reg_int).fit(cov_type='HC3')
                print(result_int.summary())
            except Exception as e:
                print(f'  Model fit failed: {e}')
else:
    print('Medical-use columns not available; intensity analysis not run.')


Intensity regression among medical AI users: med_use_intensity_score ~ age_2026 + male + edu_total_years + employed
  Coding: Selten=1, Manchmal=2, Oft=3, Immer=4; Non-medical/blank=0; score=sum(items1-3)

Intensity response frequencies (pooled across items 1-3):
  Selten: 5836
  Manchmal: 5890
  Oft: 2571
  Immer: 677

Per-item intensity frequencies:
  ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_int: Selten=2398, Manchmal=2195, Oft=896, Immer=217
  ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2_int: Selten=1686, Manchmal=1972, Oft=892, Immer=309
  ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3_int: Selten=1752, Manchmal=1723, Oft=783, Immer=151

  N (complete cases): 7854
  Outcome descriptives:
count    7854.000
mean        3.570
std         2.433
min         1.000
25%         2.000
50%         3.000
75%         5.000
max        12.000
  Model: med_use_intensity_score ~ age_2026 + male + edu_total_years + employed
                               OLS 

### General health-related use intensity: sociodemographic + clinical predictors


In [21]:
# -----------------------------
# 7) Extended intensity regression: sociodemographic + clinical predictors
# Outcome: med_use_intensity_score (items 1-3, SUM score)
# -----------------------------

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# Medical-use items 1-3
purpose_cols_13 = [
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3',
]
purpose_cols_13 = [c for c in purpose_cols_13 if c in df_base_predictors.columns]

# Base sample: AI users only (same as prior analyses)
d_ext = df_base_predictors.loc[df_base_predictors['ai_use'] == 1].copy()


def norm_txt(x):
    if pd.isna(x):
        return ''
    s = str(x).strip().lower()
    if s == 'nan':
        return ''
    s = s.replace('ä', 'ae').replace('ö', 'oe').replace('ü', 'ue').replace('ß', 'ss')
    return s


intensity_map = {
    'selten': 1,
    'manchmal': 2,
    'oft': 3,
    'immer': 4,
}
non_medical_answers = {
    'hierfuer nutze ich keine ki',
    '(bisher) kein bedarf, wuerde ki aber dafuer nutzen',
}


def map_intensity(v):
    s = norm_txt(v)
    if s == '':
        return 0
    if s in non_medical_answers:
        return 0
    if s in intensity_map:
        return intensity_map[s]
    for k, val in intensity_map.items():
        if k in s:
            return val
    return np.nan


if len(purpose_cols_13) == 3:
    for c in purpose_cols_13:
        d_ext[f'{c}_int'] = d_ext[c].apply(map_intensity)

    int_cols = [f'{c}_int' for c in purpose_cols_13]

    # Include only medical users for this analysis (= at least one intensity 1..4)
    has_medical_use = d_ext[int_cols].apply(lambda r: any(pd.notna(v) and v > 0 for v in r), axis=1)

    # Keep only rows with known mapped intensity values on all 3 items
    known_intensity = d_ext[int_cols].notna().all(axis=1)

    d_ext['med_use_intensity_sum'] = d_ext[int_cols].sum(axis=1)
    # Use sum score directly (no division)
    d_ext['med_use_intensity_score'] = d_ext['med_use_intensity_sum']

    # Clinical coding exactly as used above in notebook
    # PHQ-4: recode 1-4 -> 0-3, then sum only if all 4 items are valid
    phq4_cols = ['PHQ4_1', 'PHQ4_2', 'PHQ4_3', 'PHQ4_4']
    phq4_raw = d_ext[phq4_cols].apply(pd.to_numeric, errors='coerce')
    phq4_recoded = phq4_raw.replace({1: 0, 2: 1, 3: 2, 4: 3})
    phq4_recoded = phq4_recoded.where(phq4_raw.isin([1, 2, 3, 4]), np.nan)
    mask_phq4_complete = phq4_recoded.notna().all(axis=1)
    d_ext['PHQ4_sum'] = phq4_recoded.sum(axis=1)
    d_ext.loc[~mask_phq4_complete, 'PHQ4_sum'] = np.nan

    # Behandlung_dich: a=1, b=0, else NaN
    bh = d_ext['Behandlung_dich'].astype(str).str.strip().str.lower()
    d_ext['behandlung'] = np.where(bh == 'a', 1, np.where(bh == 'b', 0, np.nan))

    # Diagnose_selbst_dich: a=1, b=0, c/else=NaN
    ds = d_ext['Diagnose_selbst_dich'].astype(str).str.strip().str.lower()
    d_ext['diagnose_selbst'] = np.where(ds == 'a', 1, np.where(ds == 'b', 0, np.nan))

    # EQ5D5L_3: a=1 ... e=5, else NaN
    eq_map = {'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5}
    eq_raw = d_ext['EQ5D5L_3'].astype(str).str.strip().str.lower()
    d_ext['eq5d_daily'] = eq_raw.map(eq_map)

    required = [
        'med_use_intensity_score',
        'age_2026', 'male', 'edu_total_years', 'employed',
        'PHQ4_sum', 'behandlung', 'diagnose_selbst', 'eq5d_daily',
    ]

    df_reg_int_ext = d_ext.loc[has_medical_use & known_intensity].copy()

    print()
    print('Extended intensity regression among medical AI users')
    print('  Outcome: med_use_intensity_score = (item1 + item2 + item3)')
    print('  Predictors: age_2026 + male + edu_total_years + employed + PHQ4_sum + behandlung + diagnose_selbst + eq5d_daily')
    print(f'  Starting N (medical users, known intensity on all 3 items): {len(df_reg_int_ext)}')

    # Intensity overview before complete-case filtering
    label_map = {1: 'Selten', 2: 'Manchmal', 3: 'Oft', 4: 'Immer'}

    pooled_start = df_reg_int_ext[int_cols].stack()
    pooled_start = pooled_start[pooled_start.isin([1, 2, 3, 4])]
    counts_start = pooled_start.value_counts().sort_index()

    print('  Intensity frequencies (pooled across items 1-3, starting sample):')
    for k in [1, 2, 3, 4]:
        print(f"    {label_map[k]}: {int(counts_start.get(k, 0))}")

    print('  Per-item intensity frequencies (starting sample):')
    for c in int_cols:
        vc = df_reg_int_ext[c].value_counts().sort_index()
        print(f"    {c}: " + ", ".join([f"{label_map[k]}={int(vc.get(k, 0))}" for k in [1, 2, 3, 4]]))

    # Row-loss diagnostics
    remaining = df_reg_int_ext.copy()
    print('  Row loss by required variable:')
    for c in required:
        before = len(remaining)
        n_valid = int(remaining[c].notna().sum())
        n_miss = int(remaining[c].isna().sum())
        remaining = remaining.dropna(subset=[c])
        lost = before - len(remaining)
        print(f'    {c:<22} valid={n_valid:>6}  missing={n_miss:>6}  lost={lost:>6}')

    # Complete-case sample for model
    df_reg_int_ext = df_reg_int_ext.dropna(subset=required).copy()

    # Numeric type safety
    num_cols = [
        'med_use_intensity_score', 'age_2026', 'male', 'edu_total_years', 'employed',
        'PHQ4_sum', 'behandlung', 'diagnose_selbst', 'eq5d_daily'
    ]
    for c in num_cols:
        df_reg_int_ext[c] = pd.to_numeric(df_reg_int_ext[c], errors='coerce')
    df_reg_int_ext = df_reg_int_ext.dropna(subset=required)

    df_reg_int_ext['male'] = df_reg_int_ext['male'].astype(int)
    df_reg_int_ext['employed'] = df_reg_int_ext['employed'].astype(int)
    df_reg_int_ext['behandlung'] = df_reg_int_ext['behandlung'].astype(int)
    df_reg_int_ext['diagnose_selbst'] = df_reg_int_ext['diagnose_selbst'].astype(int)

    print(f'  Final complete-case N: {len(df_reg_int_ext)}')

    # Intensity overview in final complete-case sample
    pooled_final = df_reg_int_ext[int_cols].stack()
    pooled_final = pooled_final[pooled_final.isin([1, 2, 3, 4])]
    counts_final = pooled_final.value_counts().sort_index()

    print('  Intensity frequencies (pooled across items 1-3, final complete cases):')
    for k in [1, 2, 3, 4]:
        print(f"    {label_map[k]}: {int(counts_final.get(k, 0))}")

    if len(df_reg_int_ext) < 10:
        print('  Too few complete cases to fit model safely.')
    else:
        preds = [
            'age_2026', 'male', 'edu_total_years', 'employed',
            'PHQ4_sum', 'behandlung', 'diagnose_selbst', 'eq5d_daily'
        ]
        keep_preds = [p for p in preds if df_reg_int_ext[p].nunique(dropna=True) > 1]
        dropped = [p for p in preds if p not in keep_preds]
        if dropped:
            print('  Dropped constant predictor(s): ' + ', '.join(dropped))

        if not keep_preds:
            print('  All predictors are constant; model not fit.')
        else:
            formula = 'med_use_intensity_score ~ ' + ' + '.join(keep_preds)
            print(f'  Model: {formula}')
            try:
                result_int_ext = smf.ols(formula, data=df_reg_int_ext).fit(cov_type='HC3')
                print(result_int_ext.summary())
            except Exception as e:
                print(f'  Model fit failed: {e}')
else:
    print('Expected 3 medical-use columns (items 1-3), but they are not all available; extended intensity model not run.')


Extended intensity regression among medical AI users
  Outcome: med_use_intensity_score = (item1 + item2 + item3)
  Predictors: age_2026 + male + edu_total_years + employed + PHQ4_sum + behandlung + diagnose_selbst + eq5d_daily
  Starting N (medical users, known intensity on all 3 items): 8727
  Intensity frequencies (pooled across items 1-3, starting sample):
    Selten: 6516
    Manchmal: 6580
    Oft: 2875
    Immer: 736
  Per-item intensity frequencies (starting sample):
    ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_int: Selten=2680, Manchmal=2450, Oft=998, Immer=240
    ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2_int: Selten=1877, Manchmal=2216, Oft=993, Immer=331
    ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3_int: Selten=1959, Manchmal=1914, Oft=884, Immer=165
  Row loss by required variable:
    med_use_intensity_score valid=  8727  missing=     0  lost=     0
    age_2026               valid=  8727  missing=     0  lost=     0
    mal

### Mental health-related use intensity: sociodemographic predictors


In [22]:
# -----------------------------
# Intensity regression among emotional AI users (with Q4.13),
# combining grief/loss (Q4.9) + life changes (Q4.10) into one variable
# Outcome: emo_use_intensity_score_combined
# -----------------------------

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# Emotional items incl. Q4.13
purpose_cols = [
    'ki_bei_aengsten_und_sorgen',                               # Q4.4
    'ki_bei_selbstzweifeln_oder_niedergeschlagenheit',         # Q4.5
    'ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we', # Q4.6
    'ki_bei_zwischenmenschlichen_konflikten',                  # Q4.7
    'ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan', # Q4.8
    'ki_bei_trauer_oder_verlust',                              # Q4.9
    'ki_wenn_grosse_lebensveraenderungen_anstehen',            # Q4.10
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4', # Q4.11
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5', # Q4.12
    'ki_fuer_sonstige_gesundheits_psychische_oder_emotionale',   # Q4.13
]
purpose_cols = [c for c in purpose_cols if c in df_base_predictors.columns]

df_emo_int = df_base_predictors.loc[df_base_predictors['ai_use'] == 1].copy()

def norm_txt(x):
    if pd.isna(x):
        return ''
    s = str(x).strip().lower()
    if s == 'nan':
        return ''
    return s.replace('ä', 'ae').replace('ö', 'oe').replace('ü', 'ue').replace('ß', 'ss')

intensity_map = {'selten': 1, 'manchmal': 2, 'oft': 3, 'immer': 4}
non_use_answers = {
    'hierfuer nutze ich keine ki',
    '(bisher) kein bedarf, wuerde ki aber dafuer nutzen',
}

def map_intensity(v):
    s = norm_txt(v)
    if s == '' or s in non_use_answers:
        return 0
    if s in intensity_map:
        return intensity_map[s]
    for k, val in intensity_map.items():
        if k in s:
            return val
    return np.nan

# Map all item intensities
for c in purpose_cols:
    df_emo_int[f'{c}_int'] = df_emo_int[c].apply(map_intensity)

# Columns for combined score
col_grief = 'ki_bei_trauer_oder_verlust_int'
col_life = 'ki_wenn_grosse_lebensveraenderungen_anstehen_int'

if col_grief not in df_emo_int.columns or col_life not in df_emo_int.columns:
    raise KeyError("Could not find grief/loss or life-changes intensity columns after mapping.")

# Combine grief/loss + life changes:
# use MAX so pair contributes 0..4 (not double-counted)
df_emo_int['grief_life_combined_int'] = df_emo_int[[col_grief, col_life]].max(axis=1)

# Build final intensity component list:
# all item_int columns except grief/loss and life-changes, then add combined variable
all_int_cols = [f'{c}_int' for c in purpose_cols]
int_cols_combined = [c for c in all_int_cols if c not in [col_grief, col_life]] + ['grief_life_combined_int']

# Emotional users: at least one component > 0
has_emotional_use = df_emo_int[int_cols_combined].apply(
    lambda r: any(pd.notna(v) and v > 0 for v in r), axis=1
)

# Keep rows with known intensities on all used components
known_intensity = df_emo_int[int_cols_combined].notna().all(axis=1)

# Combined outcome score
df_emo_int['emo_use_intensity_score_combined'] = df_emo_int[int_cols_combined].sum(axis=1)

# Regression sample
cols_needed = ['emo_use_intensity_score_combined', 'age_2026', 'male', 'edu_total_years', 'employed']
df_reg_emo_int = df_emo_int.loc[has_emotional_use & known_intensity].copy()
df_reg_emo_int = df_reg_emo_int.dropna(subset=cols_needed)

for c in cols_needed:
    df_reg_emo_int[c] = pd.to_numeric(df_reg_emo_int[c], errors='coerce')
df_reg_emo_int = df_reg_emo_int.dropna(subset=cols_needed)

df_reg_emo_int['male'] = df_reg_emo_int['male'].astype(int)
df_reg_emo_int['employed'] = df_reg_emo_int['employed'].astype(int)

print(f"N (complete cases): {len(df_reg_emo_int):,}")
print(df_reg_emo_int['emo_use_intensity_score_combined'].describe().round(3).to_string())

formula = 'emo_use_intensity_score_combined ~ age_2026 + male + edu_total_years + employed'
result = smf.ols(formula, data=df_reg_emo_int).fit(cov_type='HC3')

print(f"\nModel: {formula}")
print(result.summary())

N (complete cases): 4,581
count    4581.000
mean        4.595
std         5.085
min         1.000
25%         1.000
50%         3.000
75%         6.000
max        36.000

Model: emo_use_intensity_score_combined ~ age_2026 + male + edu_total_years + employed
                                   OLS Regression Results                                   
Dep. Variable:     emo_use_intensity_score_combined   R-squared:                       0.050
Model:                                          OLS   Adj. R-squared:                  0.049
Method:                               Least Squares   F-statistic:                     60.33
Date:                              Wed, 15 Apr 2026   Prob (F-statistic):           9.89e-50
Time:                                      21:29:18   Log-Likelihood:                -13832.
No. Observations:                              4581   AIC:                         2.767e+04
Df Residuals:                                  4576   BIC:                         2.771e+0

### Mental health-related use intensity: sociodemographic + clinical predictors


In [23]:
# -----------------------------
# 9) Extended intensity regression among emotional AI users
# Outcome: emo_use_intensity_score (Q4.4-Q4.13, SUM score)
# Predictors: sociodemographic + clinical
#
# Special handling:
# - Q4.13 (Sonstige) is INCLUDED
# - Q4.9 (grief/loss) + Q4.10 (life changes) are COMBINED into ONE component
#   via max(Q4.9, Q4.10) so they are not overcounted
# -----------------------------

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

purpose_cols_4_13 = [
    'ki_bei_aengsten_und_sorgen',                               # Q4.4
    'ki_bei_selbstzweifeln_oder_niedergeschlagenheit',         # Q4.5
    'ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we', # Q4.6
    'ki_bei_zwischenmenschlichen_konflikten',                  # Q4.7
    'ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan', # Q4.8
    'ki_bei_trauer_oder_verlust',                              # Q4.9
    'ki_wenn_grosse_lebensveraenderungen_anstehen',            # Q4.10
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4', # Q4.11
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5', # Q4.12
    'ki_fuer_sonstige_gesundheits_psychische_oder_emotionale',   # Q4.13 (INCLUDED)
]
purpose_cols_4_13 = [c for c in purpose_cols_4_13 if c in df_base_predictors.columns]

if len(purpose_cols_4_13) != 10:
    print('Warning: expected 10 emotional-use columns (Q4.4-Q4.13), found:', len(purpose_cols_4_13))

# Base sample: AI users only
d_ext_emo = df_base_predictors.loc[df_base_predictors['ai_use'] == 1].copy()

def norm_txt(x):
    if pd.isna(x):
        return ''
    s = str(x).strip().lower()
    if s == 'nan':
        return ''
    s = s.replace('ä', 'ae').replace('ö', 'oe').replace('ü', 'ue').replace('ß', 'ss')
    return s

intensity_map = {
    'selten': 1,
    'manchmal': 2,
    'oft': 3,
    'immer': 4,
}

non_use_answers = {
    'hierfuer nutze ich keine ki',
    '(bisher) kein bedarf, wuerde ki aber dafuer nutzen',
}

def map_intensity(v):
    s = norm_txt(v)
    if s == '':
        return 0
    if s in non_use_answers:
        return 0
    if s in intensity_map:
        return intensity_map[s]
    for k, val in intensity_map.items():
        if k in s:
            return val
    return np.nan

if len(purpose_cols_4_13) > 0:
    # Map item intensities
    for c in purpose_cols_4_13:
        d_ext_emo[f'{c}_int'] = d_ext_emo[c].apply(map_intensity)

    # Build combined grief/life-change component to avoid double weighting
    col_grief = 'ki_bei_trauer_oder_verlust_int'
    col_life = 'ki_wenn_grosse_lebensveraenderungen_anstehen_int'
    combined_col = 'grief_life_combined_int'

    if col_grief in d_ext_emo.columns and col_life in d_ext_emo.columns:
        d_ext_emo[combined_col] = d_ext_emo[[col_grief, col_life]].max(axis=1)
    else:
        raise KeyError('Could not create combined grief/life variable (missing Q4.9 or Q4.10).')

    # Intensity columns for scoring:
    # all mapped columns except Q4.9 & Q4.10, plus combined grief/life
    all_int_cols = [f'{c}_int' for c in purpose_cols_4_13]
    int_cols = [c for c in all_int_cols if c not in [col_grief, col_life]] + [combined_col]

    # Include only emotional users: at least one intensity 1..4
    has_emotional_use = d_ext_emo[int_cols].apply(
        lambda r: any(pd.notna(v) and v > 0 for v in r), axis=1
    )

    # Keep only rows with known mapped intensity values (0..4) on all used components
    known_intensity = d_ext_emo[int_cols].notna().all(axis=1)

    # Intensity score as raw sum across components (Q4.9+Q4.10 counted once)
    d_ext_emo['emo_use_intensity_sum'] = d_ext_emo[int_cols].sum(axis=1)
    d_ext_emo['emo_use_intensity_score'] = d_ext_emo['emo_use_intensity_sum']

    # Clinical coding exactly as prior extended models
    phq4_cols = ['PHQ4_1', 'PHQ4_2', 'PHQ4_3', 'PHQ4_4']
    phq4_raw = d_ext_emo[phq4_cols].apply(pd.to_numeric, errors='coerce')
    phq4_recoded = phq4_raw.replace({1: 0, 2: 1, 3: 2, 4: 3})
    phq4_recoded = phq4_recoded.where(phq4_raw.isin([1, 2, 3, 4]), np.nan)
    mask_phq4_complete = phq4_recoded.notna().all(axis=1)
    d_ext_emo['PHQ4_sum'] = phq4_recoded.sum(axis=1)
    d_ext_emo.loc[~mask_phq4_complete, 'PHQ4_sum'] = np.nan

    bh = d_ext_emo['Behandlung_dich'].astype(str).str.strip().str.lower()
    d_ext_emo['behandlung'] = np.where(bh == 'a', 1, np.where(bh == 'b', 0, np.nan))

    ds = d_ext_emo['Diagnose_selbst_dich'].astype(str).str.strip().str.lower()
    d_ext_emo['diagnose_selbst'] = np.where(ds == 'a', 1, np.where(ds == 'b', 0, np.nan))

    eq_map = {'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5}
    eq_raw = d_ext_emo['EQ5D5L_3'].astype(str).str.strip().str.lower()
    d_ext_emo['eq5d_daily'] = eq_raw.map(eq_map)

    required = [
        'emo_use_intensity_score',
        'age_2026', 'male', 'edu_total_years', 'employed',
        'PHQ4_sum', 'behandlung', 'diagnose_selbst', 'eq5d_daily',
    ]

    df_reg_emo_int_ext = d_ext_emo.loc[has_emotional_use & known_intensity].copy()

    print()
    print('Extended intensity regression among emotional AI users')
    print('  Outcome: emo_use_intensity_score = sum(Q4.4-Q4.13), with Q4.9+Q4.10 combined by MAX (not double-counted)')
    print('  Predictors: age_2026 + male + edu_total_years + employed + PHQ4_sum + behandlung + diagnose_selbst + eq5d_daily')
    print(f'  Starting N (emotional users, known intensity on all components): {len(df_reg_emo_int_ext)}')

    label_map = {1: 'Selten', 2: 'Manchmal', 3: 'Oft', 4: 'Immer'}

    pooled_start = df_reg_emo_int_ext[int_cols].stack()
    pooled_start = pooled_start[pooled_start.isin([1, 2, 3, 4])]
    counts_start = pooled_start.value_counts().sort_index()

    print('  Intensity frequencies (pooled across components, starting sample):')
    for k in [1, 2, 3, 4]:
        print(f"    {label_map[k]}: {int(counts_start.get(k, 0))}")

    print('  Per-component intensity frequencies (starting sample):')
    for c in int_cols:
        vc = df_reg_emo_int_ext[c].value_counts().sort_index()
        print(f"    {c}: " + ', '.join([f"{label_map[k]}={int(vc.get(k, 0))}" for k in [1, 2, 3, 4]]))

    # Row-loss diagnostics
    remaining = df_reg_emo_int_ext.copy()
    print('  Row loss by required variable:')
    for c in required:
        before = len(remaining)
        n_valid = int(remaining[c].notna().sum())
        n_miss = int(remaining[c].isna().sum())
        remaining = remaining.dropna(subset=[c])
        lost = before - len(remaining)
        print(f'    {c:<22} valid={n_valid:>6}  missing={n_miss:>6}  lost={lost:>6}')

    # Complete-case sample
    df_reg_emo_int_ext = df_reg_emo_int_ext.dropna(subset=required).copy()

    num_cols = [
        'emo_use_intensity_score', 'age_2026', 'male', 'edu_total_years', 'employed',
        'PHQ4_sum', 'behandlung', 'diagnose_selbst', 'eq5d_daily',
    ]
    for c in num_cols:
        df_reg_emo_int_ext[c] = pd.to_numeric(df_reg_emo_int_ext[c], errors='coerce')
    df_reg_emo_int_ext = df_reg_emo_int_ext.dropna(subset=required)

    df_reg_emo_int_ext['male'] = df_reg_emo_int_ext['male'].astype(int)
    df_reg_emo_int_ext['employed'] = df_reg_emo_int_ext['employed'].astype(int)
    df_reg_emo_int_ext['behandlung'] = df_reg_emo_int_ext['behandlung'].astype(int)
    df_reg_emo_int_ext['diagnose_selbst'] = df_reg_emo_int_ext['diagnose_selbst'].astype(int)

    print(f'  Final complete-case N: {len(df_reg_emo_int_ext)}')

    pooled_final = df_reg_emo_int_ext[int_cols].stack()
    pooled_final = pooled_final[pooled_final.isin([1, 2, 3, 4])]
    counts_final = pooled_final.value_counts().sort_index()

    print('  Intensity frequencies (pooled across components, final complete cases):')
    for k in [1, 2, 3, 4]:
        print(f"    {label_map[k]}: {int(counts_final.get(k, 0))}")

    if len(df_reg_emo_int_ext) < 10:
        print('  Too few complete cases to fit model safely.')
    else:
        preds = [
            'age_2026', 'male', 'edu_total_years', 'employed',
            'PHQ4_sum', 'behandlung', 'diagnose_selbst', 'eq5d_daily',
        ]
        keep_preds = [p for p in preds if df_reg_emo_int_ext[p].nunique(dropna=True) > 1]
        dropped = [p for p in preds if p not in keep_preds]
        if dropped:
            print('  Dropped constant predictor(s): ' + ', '.join(dropped))

        if not keep_preds:
            print('  All predictors are constant; model not fit.')
        else:
            formula = 'emo_use_intensity_score ~ ' + ' + '.join(keep_preds)
            print(f'  Model: {formula}')
            try:
                result_emo_int_ext = smf.ols(formula, data=df_reg_emo_int_ext).fit(cov_type='HC3')
                print(result_emo_int_ext.summary())
            except Exception as e:
                print(f'  Model fit failed: {e}')
else:
    print('Emotional-use columns not available; extended intensity analysis not run.')


Extended intensity regression among emotional AI users
  Outcome: emo_use_intensity_score = sum(Q4.4-Q4.13), with Q4.9+Q4.10 combined by MAX (not double-counted)
  Predictors: age_2026 + male + edu_total_years + employed + PHQ4_sum + behandlung + diagnose_selbst + eq5d_daily
  Starting N (emotional users, known intensity on all components): 5097
  Intensity frequencies (pooled across components, starting sample):
    Selten: 6521
    Manchmal: 4811
    Oft: 1863
    Immer: 464
  Per-component intensity frequencies (starting sample):
    ki_bei_aengsten_und_sorgen_int: Selten=1009, Manchmal=760, Oft=351, Immer=82
    ki_bei_selbstzweifeln_oder_niedergeschlagenheit_int: Selten=628, Manchmal=444, Oft=230, Immer=60
    ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we_int: Selten=299, Manchmal=220, Oft=92, Immer=37
    ki_bei_zwischenmenschlichen_konflikten_int: Selten=834, Manchmal=505, Oft=196, Immer=46
    ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan_int: Selten=1288, 

### Side effect intensity: sociodemographic predictors


In [24]:
# -----------------------------
# Side-effect intensity regression (sociodemographic predictors only)
# Outcome:
#   side_effect_intensity_mean = (sum of side-effect intensity points) / (number of experienced side effects)
#   -> range 1..3 among included participants
# Inclusion:
#   - AI users only
#   - at least one experienced side effect (any item intensity > 0)
#   - complete predictors: age_2026, male, edu_total_years, employed
# -----------------------------

import re
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# Base sample: AI users only
d_se = df_base_predictors.loc[df_base_predictors['ai_use'] == 1].copy()

# Side-effect columns
col_social = 'ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner'  # G2Q5.1
col_dep = 'ki_ai_dependency'                                            # G2Q6
col_dec = 'ki_decision_difficulty'                                      # G2Q7

needed_cols = [col_social, col_dep, col_dec]
missing_cols = [c for c in needed_cols if c not in d_se.columns]

if missing_cols:
    print("Missing side-effect column(s):", missing_cols)
else:
    def norm_txt(x):
        if pd.isna(x):
            return ''
        s = str(x).strip().lower()
        if s == 'nan':
            return ''
        s = s.replace('ä', 'ae').replace('ö', 'oe').replace('ü', 'ue').replace('ß', 'ss')
        return s

    # --- Intensity coding ---
    # Social item:
    #   -1 -> 1, -2 -> 2, -3 -> 3
    #   0/+1/+2/+3 -> 0
    # Handles strings like "0 (Unverändert)" / "-3 (Viel schlechter)"
    def code_social(v):
        if pd.isna(v):
            return np.nan
        s = str(v).strip().lower()
        if s in ['', 'nan']:
            return np.nan

        m = re.search(r'[-+]?\d+', s)
        if not m:
            return np.nan

        n = int(m.group(0))
        if n in (-1, -2, -3):
            return abs(n)
        if n in (0, 1, 2, 3):
            return 0
        return np.nan

    # Dependency + decision difficulty:
    # gar nicht -> 0, ein wenig -> 1, teilweise -> 2, voellig -> 3
    def code_likert_side_effect(v):
        s = norm_txt(v)
        if s == '':
            return np.nan
        if s == 'trifft gar nicht zu':
            return 0
        if s == 'trifft ein wenig zu':
            return 1
        if s == 'trifft teilweise zu':
            return 2
        if s in ['trifft voellig zu', 'trifft völlig zu']:
            return 3
        return np.nan

    d_se['se_social_int'] = d_se[col_social].apply(code_social)
    d_se['se_dep_int'] = d_se[col_dep].apply(code_likert_side_effect)
    d_se['se_dec_int'] = d_se[col_dec].apply(code_likert_side_effect)

    int_cols = ['se_social_int', 'se_dep_int', 'se_dec_int']

    # ---------- Overview ----------
    # Base for overview: AI users with at least one side-effect item answered
    answered_any = d_se[int_cols].notna().any(axis=1)
    d_overview = d_se.loc[answered_any].copy()

    print()
    print('Side-effect coding overview (AI users with >=1 side-effect item answered)')
    print(f'  N base: {len(d_overview)}')

    side_effect_labels = {
        'se_social_int': 'Social alienation (G2Q5.1)',
        'se_dep_int': 'AI dependency (G2Q6)',
        'se_dec_int': 'Decision difficulty (G2Q7)',
    }

    for c in int_cols:
        lbl = side_effect_labels.get(c, c)
        vals = d_overview[c]
        n_any = int((vals > 0).sum())
        n_none = int((vals == 0).sum())
        n_miss = int(vals.isna().sum())
        lvl1 = int((vals == 1).sum())
        lvl2 = int((vals == 2).sum())
        lvl3 = int((vals == 3).sum())

        print(f'\n  {lbl}:')
        print(f'    Any side effect (>0): {n_any}')
        print(f'    No side effect (=0):  {n_none}')
        print(f'    Missing:              {n_miss}')
        print(f'    Intensity levels:     1={lvl1}, 2={lvl2}, 3={lvl3}')

    d_overview['n_side_effects_experienced'] = (d_overview[int_cols] > 0).sum(axis=1)
    print('\n  Number of side effects experienced per person (0-3):')
    print(d_overview['n_side_effects_experienced'].value_counts().sort_index().to_string())

    # ---------- Regression sample ----------
    # At least one side-effect item answered
    d_reg = d_se.loc[answered_any].copy()

    # Number of experienced side effects (intensity > 0)
    d_reg['n_side_effects_experienced'] = (d_reg[int_cols] > 0).sum(axis=1)

    # Include only participants with at least one experienced side effect
    d_reg = d_reg.loc[d_reg['n_side_effects_experienced'] > 0].copy()

    # Sum score and mean intensity (1..3)
    d_reg['side_effect_intensity_sum'] = d_reg[int_cols].fillna(0).sum(axis=1)
    d_reg['side_effect_intensity_mean'] = (
        d_reg['side_effect_intensity_sum'] / d_reg['n_side_effects_experienced']
    )

    # Predictor complete-case filter
    cols_needed = ['side_effect_intensity_mean', 'age_2026', 'male', 'edu_total_years', 'employed']
    df_reg_se_int = d_reg.dropna(subset=cols_needed).copy()

    # Type safety
    for c in cols_needed:
        df_reg_se_int[c] = pd.to_numeric(df_reg_se_int[c], errors='coerce')
    df_reg_se_int = df_reg_se_int.dropna(subset=cols_needed)

    df_reg_se_int['male'] = df_reg_se_int['male'].astype(int)
    df_reg_se_int['employed'] = df_reg_se_int['employed'].astype(int)

    print()
    print('Side-effect intensity regression among AI users with >=1 experienced side effect')
    print('  Outcome: side_effect_intensity_mean = sum(intensity of 3 side-effect items) / number of experienced side effects')
    print('  Coding:')
    print('    social item: -1->1, -2->2, -3->3; 0/+1/+2/+3 -> 0')
    print('    dependency/decision: gar nicht=0, ein wenig=1, teilweise=2, voellig=3')
    print(f'  N (complete cases): {len(df_reg_se_int)}')

    print('\nOutcome descriptives (side_effect_intensity_mean):')
    print(df_reg_se_int['side_effect_intensity_mean'].describe().round(3).to_string())

    if len(df_reg_se_int) < 10:
        print('  Too few complete cases to fit model safely.')
    else:
        preds = ['age_2026', 'male', 'edu_total_years', 'employed']
        keep_preds = [p for p in preds if df_reg_se_int[p].nunique(dropna=True) > 1]
        dropped = [p for p in preds if p not in keep_preds]

        if dropped:
            print('  Dropped constant predictor(s): ' + ', '.join(dropped))

        if not keep_preds:
            print('  All predictors are constant; model not fit.')
        else:
            formula = 'side_effect_intensity_mean ~ ' + ' + '.join(keep_preds)
            print(f'  Model: {formula}')
            try:
                result_se_int = smf.ols(formula, data=df_reg_se_int).fit(cov_type='HC3')
                print(result_se_int.summary())
            except Exception as e:
                print(f'  Model fit failed: {e}')


Side-effect coding overview (AI users with >=1 side-effect item answered)
  N base: 14909

  Social alienation (G2Q5.1):
    Any side effect (>0): 51
    No side effect (=0):  14537
    Missing:              321
    Intensity levels:     1=35, 2=8, 3=8

  AI dependency (G2Q6):
    Any side effect (>0): 1499
    No side effect (=0):  13391
    Missing:              19
    Intensity levels:     1=1169, 2=274, 3=56

  Decision difficulty (G2Q7):
    Any side effect (>0): 696
    No side effect (=0):  14195
    Missing:              18
    Intensity levels:     1=590, 2=100, 3=6

  Number of side effects experienced per person (0-3):
n_side_effects_experienced
0    13081
1     1422
2      394
3       12

Side-effect intensity regression among AI users with >=1 experienced side effect
  Outcome: side_effect_intensity_mean = sum(intensity of 3 side-effect items) / number of experienced side effects
  Coding:
    social item: -1->1, -2->2, -3->3; 0/+1/+2/+3 -> 0
    dependency/decision: gar 

### Side effect intensity: sociodemographic + clinical predictors


In [25]:
# -----------------------------
# Side-effect intensity regression (sociodemographic + clinical predictors)
# Outcome:
#   side_effect_intensity_mean = (sum of side-effect intensity points) / (number of experienced side effects)
#   -> range 1..3 among included participants
# Inclusion:
#   - AI users only
#   - at least one experienced side effect (any item intensity > 0)
#   - complete predictors and outcome
# -----------------------------

import re
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# Base sample: AI users only
d_se = df_base_predictors.loc[df_base_predictors['ai_use'] == 1].copy()

# Side-effect columns
col_social = 'ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner'  # G2Q5.1
col_dep = 'ki_ai_dependency'                                            # G2Q6
col_dec = 'ki_decision_difficulty'                                      # G2Q7

needed_cols = [col_social, col_dep, col_dec]
missing_cols = [c for c in needed_cols if c not in d_se.columns]

if missing_cols:
    print("Missing side-effect column(s):", missing_cols)
else:
    def norm_txt(x):
        if pd.isna(x):
            return ''
        s = str(x).strip().lower()
        if s == 'nan':
            return ''
        s = s.replace('ä', 'ae').replace('ö', 'oe').replace('ü', 'ue').replace('ß', 'ss')
        return s

    # Social item coding:
    # -1 -> 1, -2 -> 2, -3 -> 3 ; 0/+1/+2/+3 -> 0
    def code_social(v):
        if pd.isna(v):
            return np.nan
        s = str(v).strip().lower()
        if s in ['', 'nan']:
            return np.nan

        m = re.search(r'[-+]?\d+', s)
        if not m:
            return np.nan

        n = int(m.group(0))
        if n in (-1, -2, -3):
            return abs(n)
        if n in (0, 1, 2, 3):
            return 0
        return np.nan

    # Dependency / decision coding:
    # gar nicht -> 0, ein wenig -> 1, teilweise -> 2, voellig -> 3
    def code_likert_side_effect(v):
        s = norm_txt(v)
        if s == '':
            return np.nan
        if s == 'trifft gar nicht zu':
            return 0
        if s == 'trifft ein wenig zu':
            return 1
        if s == 'trifft teilweise zu':
            return 2
        if s in ['trifft voellig zu', 'trifft völlig zu']:
            return 3
        return np.nan

    d_se['se_social_int'] = d_se[col_social].apply(code_social)
    d_se['se_dep_int'] = d_se[col_dep].apply(code_likert_side_effect)
    d_se['se_dec_int'] = d_se[col_dec].apply(code_likert_side_effect)

    int_cols = ['se_social_int', 'se_dep_int', 'se_dec_int']

    # At least one side-effect item answered
    answered_any = d_se[int_cols].notna().any(axis=1)
    d_reg = d_se.loc[answered_any].copy()

    # Number of experienced side effects (intensity > 0)
    d_reg['n_side_effects_experienced'] = (d_reg[int_cols] > 0).sum(axis=1)

    # Keep only people with at least one experienced side effect
    d_reg = d_reg.loc[d_reg['n_side_effects_experienced'] > 0].copy()

    # Build intensity outcome
    d_reg['side_effect_intensity_sum'] = d_reg[int_cols].fillna(0).sum(axis=1)
    d_reg['side_effect_intensity_mean'] = (
        d_reg['side_effect_intensity_sum'] / d_reg['n_side_effects_experienced']
    )

    # -----------------------------
    # Clinical coding (same logic as your prior extended models)
    # -----------------------------
    # PHQ-4: recode 1-4 -> 0-3; sum only if all 4 items valid
    phq4_cols = ['PHQ4_1', 'PHQ4_2', 'PHQ4_3', 'PHQ4_4']
    phq4_raw = d_reg[phq4_cols].apply(pd.to_numeric, errors='coerce')
    phq4_recoded = phq4_raw.replace({1: 0, 2: 1, 3: 2, 4: 3})
    phq4_recoded = phq4_recoded.where(phq4_raw.isin([1, 2, 3, 4]), np.nan)
    mask_phq4_complete = phq4_recoded.notna().all(axis=1)
    d_reg['PHQ4_sum'] = phq4_recoded.sum(axis=1)
    d_reg.loc[~mask_phq4_complete, 'PHQ4_sum'] = np.nan

    # Behandlung_dich: a=1, b=0, else NaN
    bh = d_reg['Behandlung_dich'].astype(str).str.strip().str.lower()
    d_reg['behandlung'] = np.where(bh == 'a', 1, np.where(bh == 'b', 0, np.nan))

    # Diagnose_selbst_dich: a=1, b=0, c/else=NaN
    ds = d_reg['Diagnose_selbst_dich'].astype(str).str.strip().str.lower()
    d_reg['diagnose_selbst'] = np.where(ds == 'a', 1, np.where(ds == 'b', 0, np.nan))

    # EQ5D5L_3: a=1 ... e=5, else NaN
    eq_map = {'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5}
    eq_raw = d_reg['EQ5D5L_3'].astype(str).str.strip().str.lower()
    d_reg['eq5d_daily'] = eq_raw.map(eq_map)

    required = [
        'side_effect_intensity_mean',
        'age_2026', 'male', 'edu_total_years', 'employed',
        'PHQ4_sum', 'behandlung', 'diagnose_selbst', 'eq5d_daily'
    ]

    print()
    print('Extended side-effect intensity regression among AI users with >=1 experienced side effect')
    print('  Outcome: side_effect_intensity_mean = side_effect_intensity_sum / number of experienced side effects')
    print('  Predictors: age_2026 + male + edu_total_years + employed + PHQ4_sum + behandlung + diagnose_selbst + eq5d_daily')
    print(f'  Starting N (AI users, >=1 side-effect item answered): {len(d_reg)}')

    # Row-loss diagnostics
    remaining = d_reg.copy()
    print('  Row loss by required variable:')
    for c in required:
        before = len(remaining)
        n_valid = int(remaining[c].notna().sum())
        n_miss = int(remaining[c].isna().sum())
        remaining = remaining.dropna(subset=[c])
        lost = before - len(remaining)
        print(f'    {c:<26} valid={n_valid:>6}  missing={n_miss:>6}  lost={lost:>6}')

    # Complete-case sample
    df_reg_se_int_ext = d_reg.dropna(subset=required).copy()

    # Numeric type safety
    num_cols = required
    for c in num_cols:
        df_reg_se_int_ext[c] = pd.to_numeric(df_reg_se_int_ext[c], errors='coerce')
    df_reg_se_int_ext = df_reg_se_int_ext.dropna(subset=required)

    df_reg_se_int_ext['male'] = df_reg_se_int_ext['male'].astype(int)
    df_reg_se_int_ext['employed'] = df_reg_se_int_ext['employed'].astype(int)
    df_reg_se_int_ext['behandlung'] = df_reg_se_int_ext['behandlung'].astype(int)
    df_reg_se_int_ext['diagnose_selbst'] = df_reg_se_int_ext['diagnose_selbst'].astype(int)

    print(f'  Final complete-case N: {len(df_reg_se_int_ext)}')

    print('\nOutcome descriptives (side_effect_intensity_mean):')
    print(df_reg_se_int_ext['side_effect_intensity_mean'].describe().round(3).to_string())

    if len(df_reg_se_int_ext) < 10:
        print('  Too few complete cases to fit model safely.')
    else:
        preds = [
            'age_2026', 'male', 'edu_total_years', 'employed',
            'PHQ4_sum', 'behandlung', 'diagnose_selbst', 'eq5d_daily'
        ]
        keep_preds = [p for p in preds if df_reg_se_int_ext[p].nunique(dropna=True) > 1]
        dropped = [p for p in preds if p not in keep_preds]

        if dropped:
            print('  Dropped constant predictor(s): ' + ', '.join(dropped))

        if not keep_preds:
            print('  All predictors are constant; model not fit.')
        else:
            formula = 'side_effect_intensity_mean ~ ' + ' + '.join(keep_preds)
            print(f'  Model: {formula}')
            try:
                result_se_int_ext = smf.ols(formula, data=df_reg_se_int_ext).fit(cov_type='HC3')
                print(result_se_int_ext.summary())
            except Exception as e:
                print(f'  Model fit failed: {e}')


Extended side-effect intensity regression among AI users with >=1 experienced side effect
  Outcome: side_effect_intensity_mean = side_effect_intensity_sum / number of experienced side effects
  Predictors: age_2026 + male + edu_total_years + employed + PHQ4_sum + behandlung + diagnose_selbst + eq5d_daily
  Starting N (AI users, >=1 side-effect item answered): 1828
  Row loss by required variable:
    side_effect_intensity_mean valid=  1828  missing=     0  lost=     0
    age_2026                   valid=  1828  missing=     0  lost=     0
    male                       valid=  1826  missing=     2  lost=     2
    edu_total_years            valid=  1677  missing=   149  lost=   149
    employed                   valid=  1674  missing=     3  lost=     3
    PHQ4_sum                   valid=   541  missing=  1133  lost=  1133
    behandlung                 valid=   538  missing=     3  lost=     3
    diagnose_selbst            valid=   505  missing=    33  lost=    33
    eq5d_daily

## Trust in AI answers

### Sociodemographic predictors


In [26]:
# UPDATED OLS regression:
# Keep ONLY participants who (a) use AI in general AND (b) use AI for medical OR emotional purposes.
# Predictors: age_2026 + male + edu_total_years + employed
# Outcome: ki_ich_halte_die_antworten_die_ich_von_einem_ki_program* (1-100)

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# ---------- helpers ----------
def norm_txt(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    if s in {"", "nan", "none"}:
        return ""
    s = (s.replace("ä", "ae")
           .replace("ö", "oe")
           .replace("ü", "ue")
           .replace("ß", "ss"))
    return s

def classify_purpose_use(row, cols, domain="medical"):
    """
    Returns:
      1 = uses for this purpose (at least one substantive answer)
      0 = only explicit non-use answers
      NaN = all missing across the purpose columns
    """
    vals = [norm_txt(row[c]) for c in cols if c in row.index]
    vals = [v for v in vals if v != ""]  # answered values only

    if len(vals) == 0:
        return np.nan  # all missing

    if domain == "medical":
        # non-use labels used in your regression logic
        def is_non_use(v):
            return (
                ("trifft nicht zu" in v and "medizin" in v)
                or v in {"kein bedarf", "keine nutzung", "nutze ich nicht", "nicht genutzt"}
            )
    else:  # emotional
        def is_non_use(v):
            return (
                ("trifft nicht zu" in v and ("psycholog" in v or "gesundheits" in v or "emotional" in v))
                or v in {"kein bedarf", "keine nutzung", "nutze ich nicht", "nicht genutzt"}
            )

    # if any answered value is NOT a non-use answer -> use = 1
    if any(not is_non_use(v) for v in vals):
        return 1
    return 0

# ---------- base dataframe ----------
if "df_base_predictors" in globals():
    d0 = df_base_predictors.copy()
elif "df_reg" in globals():
    d0 = df_reg.copy()
elif "df" in globals():
    d0 = df.copy()
else:
    raise NameError("No dataframe found. Expected one of: df_base_predictors, df_reg, df")

# ---------- required columns ----------
preds = ["age_2026", "male", "edu_total_years", "employed"]

medical_cols = [
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3",
]

emotional_cols = [
    "ki_bei_aengsten_und_sorgen",
    "ki_bei_selbstzweifeln_oder_niedergeschlagenheit",
    "ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we",
    "ki_bei_zwischenmenschlichen_konflikten",
    "ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan",
    "ki_bei_trauer_oder_verlust",
    "ki_wenn_grosse_lebensveraenderungen_anstehen",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5",
    "ki_fuer_sonstige_gesundheits_psychische_oder_emotionale",
    'ki_fuer_sonstige_gesundheits_psychische_oder_emotionale',
]

for c in preds:
    if c not in d0.columns:
        raise KeyError(f"Missing predictor column: {c}")

if "ai_use" not in d0.columns:
    raise KeyError("Column 'ai_use' is missing. Run the earlier prep cells first.")

missing_purpose = [c for c in (medical_cols + emotional_cols) if c not in d0.columns]
if missing_purpose:
    raise KeyError("Missing purpose column(s): " + ", ".join(missing_purpose[:10]))

# outcome column by prefix (robust to long final name)
outcome_prefix = "ki_ich_halte_die_antworten_die_ich_von_einem_ki_program"
matches = [c for c in d0.columns if str(c).lower().startswith(outcome_prefix.lower())]
if len(matches) == 0:
    raise KeyError(f"No column starts with '{outcome_prefix}'")
if len(matches) > 1:
    raise ValueError("Multiple possible outcome columns:\n  - " + "\n  - ".join(matches))
outcome_col = matches[0]

# ---------- filter: AI users ----------
d = d0.loc[d0["ai_use"] == 1].copy()

# classify medical/emotional use exactly in this subset
d["med_use"] = d.apply(lambda r: classify_purpose_use(r, medical_cols, domain="medical"), axis=1)
d["emo_use"] = d.apply(lambda r: classify_purpose_use(r, emotional_cols, domain="emotional"), axis=1)

# keep medical OR emotional users
d = d.loc[(d["med_use"] == 1) | (d["emo_use"] == 1)].copy()

# ---------- numeric conversion ----------
def to_num(s):
    return pd.to_numeric(
        s.astype(str).str.replace(",", ".", regex=False).str.strip(),
        errors="coerce"
    )

for c in preds + [outcome_col]:
    d[c] = to_num(d[c])

# ---------- complete-case on predictors + outcome ----------
n_after_use_filter = len(d)
d_cc = d.dropna(subset=preds + [outcome_col]).copy()

d_cc["male"] = d_cc["male"].astype(int)
d_cc["employed"] = d_cc["employed"].astype(int)

print(f"N after AI+medical/emotional-use filter: {n_after_use_filter}")
print(f"Complete-case N (4 predictors + outcome): {len(d_cc)}")
print("\nOutcome descriptives:")
print(d_cc[outcome_col].describe().round(3).to_string())

# ---------- OLS ----------
formula = f'Q("{outcome_col}") ~ age_2026 + male + edu_total_years + employed'
print("\nModel:", formula)

model = smf.ols(formula, data=d_cc).fit(cov_type="HC3")
print(model.summary())

N after AI+medical/emotional-use filter: 15046
Complete-case N (4 predictors + outcome): 7682

Outcome descriptives:
count    7682.000
mean       50.466
std        23.825
min         0.000
25%        30.000
50%        50.000
75%        70.000
max       100.000

Model: Q("ki_ich_halte_die_antworten_die_ich_von_einem_ki_program") ~ age_2026 + male + edu_total_years + employed
                                                 OLS Regression Results                                                 
Dep. Variable:     Q("ki_ich_halte_die_antworten_die_ich_von_einem_ki_program")   R-squared:                       0.004
Model:                                                                      OLS   Adj. R-squared:                  0.004
Method:                                                           Least Squares   F-statistic:                     7.955
Date:                                                          Wed, 15 Apr 2026   Prob (F-statistic):           2.14e-06
Time:             

### Extended model: sociodemographic + clinical predictors


In [27]:
# OLS regression (extended):
# Keep ONLY participants who (a) use AI in general AND (b) use AI for medical OR emotional purposes.
# Predictors: age_2026 + male + edu_total_years + employed + PHQ4_sum + behandlung + diagnose_selbst + eq5d_daily
# Outcome: ki_ich_halte_die_antworten_die_ich_von_einem_ki_program* (1-100)

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# ---------- helpers ----------
def norm_txt(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    if s in {"", "nan", "none"}:
        return ""
    s = (s.replace("ä", "ae")
           .replace("ö", "oe")
           .replace("ü", "ue")
           .replace("ß", "ss"))
    return s

def classify_purpose_use(row, cols, domain="medical"):
    """
    Returns:
      1 = uses for this purpose (at least one substantive answer)
      0 = only explicit non-use answers
      NaN = all missing across the purpose columns
    """
    vals = [norm_txt(row[c]) for c in cols if c in row.index]
    vals = [v for v in vals if v != ""]  # answered only

    if len(vals) == 0:
        return np.nan

    if domain == "medical":
        def is_non_use(v):
            return (
                ("trifft nicht zu" in v and "medizin" in v)
                or v in {"kein bedarf", "keine nutzung", "nutze ich nicht", "nicht genutzt"}
            )
    else:  # emotional
        def is_non_use(v):
            return (
                ("trifft nicht zu" in v and ("psycholog" in v or "gesundheits" in v or "emotional" in v))
                or v in {"kein bedarf", "keine nutzung", "nutze ich nicht", "nicht genutzt"}
            )

    return 1 if any(not is_non_use(v) for v in vals) else 0

def to_num(s):
    return pd.to_numeric(
        s.astype(str).str.replace(",", ".", regex=False).str.strip(),
        errors="coerce"
    )

# ---------- base dataframe ----------
if "df_base_predictors" in globals():
    d0 = df_base_predictors.copy()
elif "df_reg" in globals():
    d0 = df_reg.copy()
elif "df" in globals():
    d0 = df.copy()
else:
    raise NameError("No dataframe found. Expected one of: df_base_predictors, df_reg, df")

if "ai_use" not in d0.columns:
    raise KeyError("Column 'ai_use' is missing. Run the prep cells first.")

# ---------- outcome ----------
outcome_prefix = "ki_ich_halte_die_antworten_die_ich_von_einem_ki_program"
matches = [c for c in d0.columns if str(c).lower().startswith(outcome_prefix.lower())]
if len(matches) == 0:
    raise KeyError(f"No column starts with '{outcome_prefix}'")
if len(matches) > 1:
    raise ValueError("Multiple possible outcome columns:\n  - " + "\n  - ".join(matches))
outcome_col = matches[0]
print("Outcome column:", outcome_col)

# ---------- purpose columns ----------
medical_cols = [
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3",
]

emotional_cols = [
    "ki_bei_aengsten_und_sorgen",
    "ki_bei_selbstzweifeln_oder_niedergeschlagenheit",
    "ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we",
    "ki_bei_zwischenmenschlichen_konflikten",
    "ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan",
    "ki_bei_trauer_oder_verlust",
    "ki_wenn_grosse_lebensveraenderungen_anstehen",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5",
    "ki_fuer_sonstige_gesundheits_psychische_oder_emotionale",
    'ki_fuer_sonstige_gesundheits_psychische_oder_emotionale',
]

# ---------- filter: AI users + (medical OR emotional users) ----------
d = d0.loc[d0["ai_use"] == 1].copy()
d["med_use"] = d.apply(lambda r: classify_purpose_use(r, medical_cols, "medical"), axis=1)
d["emo_use"] = d.apply(lambda r: classify_purpose_use(r, emotional_cols, "emotional"), axis=1)
d = d.loc[(d["med_use"] == 1) | (d["emo_use"] == 1)].copy()

# ---------- clinical coding ----------
# PHQ-4: recode 1-4 -> 0-3; sum only if all 4 items valid
phq4_cols = ["PHQ4_1", "PHQ4_2", "PHQ4_3", "PHQ4_4"]
for c in phq4_cols:
    if c not in d.columns:
        raise KeyError(f"Missing PHQ-4 column: {c}")

phq4_raw = d[phq4_cols].apply(pd.to_numeric, errors="coerce")
phq4_recoded = phq4_raw.replace({1: 0, 2: 1, 3: 2, 4: 3})
phq4_recoded = phq4_recoded.where(phq4_raw.isin([1, 2, 3, 4]), np.nan)
mask_phq4_complete = phq4_recoded.notna().all(axis=1)
d["PHQ4_sum"] = phq4_recoded.sum(axis=1)
d.loc[~mask_phq4_complete, "PHQ4_sum"] = np.nan

# Behandlung_dich: a=1, b=0, else NaN
if "Behandlung_dich" not in d.columns:
    raise KeyError("Missing column: Behandlung_dich")
bh = d["Behandlung_dich"].astype(str).str.strip().str.lower()
d["behandlung"] = np.where(bh == "a", 1, np.where(bh == "b", 0, np.nan))

# Diagnose_selbst_dich: a=1, b=0, c/else=NaN
if "Diagnose_selbst_dich" not in d.columns:
    raise KeyError("Missing column: Diagnose_selbst_dich")
ds = d["Diagnose_selbst_dich"].astype(str).str.strip().str.lower()
d["diagnose_selbst"] = np.where(ds == "a", 1, np.where(ds == "b", 0, np.nan))

# EQ5D5L_3: a=1 ... e=5, else NaN
if "EQ5D5L_3" not in d.columns:
    raise KeyError("Missing column: EQ5D5L_3")
eq_map = {"a": 1, "b": 2, "c": 3, "d": 4, "e": 5}
d["eq5d_daily"] = d["EQ5D5L_3"].astype(str).str.strip().str.lower().map(eq_map)

# ---------- predictors ----------
preds_soc = ["age_2026", "male", "edu_total_years", "employed"]
preds_cli = ["PHQ4_sum", "behandlung", "diagnose_selbst", "eq5d_daily"]
preds_all = preds_soc + preds_cli

for c in preds_soc + [outcome_col]:
    if c not in d.columns:
        raise KeyError(f"Missing required column: {c}")

for c in preds_all + [outcome_col]:
    d[c] = to_num(d[c])

# complete-case on all predictors + outcome
n_after_use_filter = len(d)
d_cc = d.dropna(subset=preds_all + [outcome_col]).copy()

# int casting for binary predictors
for c in ["male", "employed", "behandlung", "diagnose_selbst"]:
    d_cc[c] = d_cc[c].astype(int)

print(f"N after AI + (medical/emotional use) filter: {n_after_use_filter}")
print(f"Complete-case N (soc + clinical + outcome): {len(d_cc)}")

print("\nOutcome descriptives:")
print(d_cc[outcome_col].describe().round(3).to_string())

# ---------- OLS ----------
formula = (
    f'Q("{outcome_col}") ~ age_2026 + male + edu_total_years + employed'
    f' + PHQ4_sum + behandlung + diagnose_selbst + eq5d_daily'
)
print("\nModel:", formula)

model = smf.ols(formula, data=d_cc).fit(cov_type="HC3")
print(model.summary())

Outcome column: ki_ich_halte_die_antworten_die_ich_von_einem_ki_program
N after AI + (medical/emotional use) filter: 15046
Complete-case N (soc + clinical + outcome): 2781

Outcome descriptives:
count    2781.000
mean       49.982
std        24.210
min         0.000
25%        30.000
50%        50.000
75%        70.000
max       100.000

Model: Q("ki_ich_halte_die_antworten_die_ich_von_einem_ki_program") ~ age_2026 + male + edu_total_years + employed + PHQ4_sum + behandlung + diagnose_selbst + eq5d_daily
                                                 OLS Regression Results                                                 
Dep. Variable:     Q("ki_ich_halte_die_antworten_die_ich_von_einem_ki_program")   R-squared:                       0.007
Model:                                                                      OLS   Adj. R-squared:                  0.004
Method:                                                           Least Squares   F-statistic:                     2.371
Date: 

## Mediation analyses: side effects

### Trust in AI (mediator), sociodemographic model


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# =========================================================
# MEDIATION ANALYSIS (with bootstrap) - CORRECTED
# Sociodemographic-only model
# X = age_2026, male, edu_total_years, employed
# M = trust_ai_answers (continuous, 0-100)
# Y = side_effect_any (binary)
# Base: AI users only
# =========================================================

# -----------------------------
# 1) Base sample
# -----------------------------
d = df_base_predictors.loc[df_base_predictors["ai_use"] == 1].copy()

# -----------------------------
# 2) Build mediator: trust variable
# -----------------------------
mediator_prefix = "ki_ich_halte_die_antworten_die_ich_von_einem_ki_program"
m_matches = [c for c in d.columns if str(c).lower().startswith(mediator_prefix.lower())]

if len(m_matches) == 0:
    raise KeyError(f"No column starts with '{mediator_prefix}'")
if len(m_matches) > 1:
    raise ValueError("Multiple possible mediator columns:\n  - " + "\n  - ".join(m_matches))

mediator_col = m_matches[0]
d["trust_ai_answers"] = pd.to_numeric(
    d[mediator_col].astype(str).str.replace(",", ".", regex=False).str.strip(),
    errors="coerce"
)

print(f"Mediator column used: {mediator_col}")
print("Mediator descriptives:")
print(d["trust_ai_answers"].describe().round(3).to_string())

# -----------------------------
# 3) Build outcome: side_effect_any (exact prior coding)
# -----------------------------
side_effect_cols = {
    "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner": "G2Q5.1 Social alienation",
    "ki_ai_dependency": "G2Q6 AI dependency",
    "ki_decision_difficulty": "G2Q7 Decision difficulty",
}

SOCIAL_YES_NUM = {-1, -2, -3}
YES_VALUES_TEXT = {
    "trifft ein wenig zu",
    "trifft teilweise zu",
    "trifft völlig zu",
    "trifft voellig zu",
}

def to_yes_no(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    if col == "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner":
        num = pd.to_numeric(s, errors="coerce")
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0

    if s.lower() in YES_VALUES_TEXT:
        return 1
    return 0

for col in side_effect_cols:
    ycol = f"{col}_yes"
    if col in d.columns:
        d[ycol] = d[col].apply(lambda x: to_yes_no(x, col))
    else:
        d[ycol] = np.nan

ycols = [f"{c}_yes" for c in side_effect_cols]
answered_any = d[ycols].notna().any(axis=1)
any_yes = d[ycols].eq(1).any(axis=1)
d["side_effect_any"] = np.where(~answered_any, np.nan, np.where(any_yes, 1, 0))

print("\nPerson-level outcome side_effect_any:")
print(d["side_effect_any"].value_counts(dropna=False).to_string())

# -----------------------------
# 4) Common complete-case sample
# -----------------------------
X = ["age_2026", "male", "edu_total_years", "employed"]
needed = ["trust_ai_answers", "side_effect_any"] + X

dfm = d.dropna(subset=needed).copy()
dfm["side_effect_any"] = dfm["side_effect_any"].astype(int)
dfm["male"] = dfm["male"].astype(int)
dfm["employed"] = dfm["employed"].astype(int)
dfm["age_2026"] = pd.to_numeric(dfm["age_2026"], errors="coerce")
dfm["edu_total_years"] = pd.to_numeric(dfm["edu_total_years"], errors="coerce")
dfm["trust_ai_answers"] = pd.to_numeric(dfm["trust_ai_answers"], errors="coerce")
dfm = dfm.dropna(subset=needed)

print(f"\nCommon complete-case N: {len(dfm):,}")

# -----------------------------
# 5) Main mediation models
# -----------------------------
# Corrected alignment: robust HC3 for OLS path a
model_a = smf.ols("trust_ai_answers ~ age_2026 + male + edu_total_years + employed", data=dfm).fit(cov_type="HC3")
model_c = smf.logit("side_effect_any ~ age_2026 + male + edu_total_years + employed", data=dfm).fit(disp=0)
model_cp = smf.logit("side_effect_any ~ age_2026 + male + edu_total_years + employed + trust_ai_answers", data=dfm).fit(disp=0)

print("\n=== Path a: trust_ai_answers ~ X (OLS, HC3 robust SE) ===")
print(model_a.summary())
print("\n=== Total effect c: side_effect_any ~ X (logit) ===")
print(model_c.summary())
print("\n=== Direct effect c' (+ b): side_effect_any ~ X + trust_ai_answers (logit) ===")
print(model_cp.summary())

# c vs c' comparison
rows = []
for p in X:
    c = model_c.params.get(p, np.nan)
    cp = model_cp.params.get(p, np.nan)
    rows.append({
        "predictor": p,
        "coef_c_total": c,
        "OR_c_total": np.exp(c),
        "p_c_total": model_c.pvalues.get(p, np.nan),
        "coef_cprime_direct": cp,
        "OR_cprime_direct": np.exp(cp),
        "p_cprime_direct": model_cp.pvalues.get(p, np.nan),
        "delta_coef_c_minus_cprime": c - cp
    })

rows.append({
    "predictor": "trust_ai_answers (b path)",
    "coef_c_total": np.nan,
    "OR_c_total": np.nan,
    "p_c_total": np.nan,
    "coef_cprime_direct": model_cp.params.get("trust_ai_answers", np.nan),
    "OR_cprime_direct": np.exp(model_cp.params.get("trust_ai_answers", np.nan)),
    "p_cprime_direct": model_cp.pvalues.get("trust_ai_answers", np.nan),
    "delta_coef_c_minus_cprime": np.nan
})

compare = pd.DataFrame(rows)
print("\n=== Mediation-style comparison ===")
print(compare.round(4).to_string(index=False))

# -----------------------------
# 6) Bootstrap indirect effects (a*b)
# -----------------------------
B = 2000
SEED = 42
rng = np.random.default_rng(SEED)

ab_store = {p: [] for p in X}
ok = 0
fail = 0

for _ in range(B):
    idx = rng.integers(0, len(dfm), len(dfm))
    bs = dfm.iloc[idx]

    try:
        a_fit = smf.ols("trust_ai_answers ~ age_2026 + male + edu_total_years + employed", data=bs).fit()
        cp_fit = smf.logit("side_effect_any ~ age_2026 + male + edu_total_years + employed + trust_ai_answers", data=bs).fit(disp=0)

        b_coef = cp_fit.params.get("trust_ai_answers", np.nan)

        for p in X:
            a_coef = a_fit.params.get(p, np.nan)
            ab_store[p].append(a_coef * b_coef)

        ok += 1
    except Exception:
        for p in X:
            ab_store[p].append(np.nan)
        fail += 1

boot_rows = []
for p in X:
    vals = np.array(ab_store[p], dtype=float)
    vals = vals[~np.isnan(vals)]
    if len(vals) == 0:
        boot_rows.append({
            "predictor": p,
            "ab_mean": np.nan,
            "ab_median": np.nan,
            "ab_ci_2.5": np.nan,
            "ab_ci_97.5": np.nan,
            "n_boot_ok_for_predictor": 0
        })
    else:
        boot_rows.append({
            "predictor": p,
            "ab_mean": vals.mean(),
            "ab_median": np.median(vals),
            "ab_ci_2.5": np.quantile(vals, 0.025),
            "ab_ci_97.5": np.quantile(vals, 0.975),
            "n_boot_ok_for_predictor": len(vals)
        })

boot_table = pd.DataFrame(boot_rows)

print("\n=== Bootstrap summary ===")
print(f"Requested draws: {B}")
print(f"Successful draws: {ok}")
print(f"Failed draws: {fail}")

print("\n=== Indirect effects (a*b) with bootstrap 95% CI ===")
print(boot_table.round(4).to_string(index=False))
print("\nInterpretation hint: CI excluding 0 suggests an indirect effect for that predictor.")

### Trust in AI (mediator), extended model with clinical predictors


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# =========================================================
# MEDIATION ANALYSIS (with bootstrap) - MATCHED STYLE
# X = age_2026, male, edu_total_years, employed, PHQ4_sum, behandlung, diagnose_selbst, eq5d_daily
# M = trust_ai_answers (continuous, 0-100)
# Y = side_effect_any (binary)
# Base: AI users only
# =========================================================

# -----------------------------
# 1) Base sample
# -----------------------------
d = df_base_predictors.loc[df_base_predictors["ai_use"] == 1].copy()

# -----------------------------
# 2) Build mediator: trust variable (0-100)
# -----------------------------
mediator_prefix = "ki_ich_halte_die_antworten_die_ich_von_einem_ki_program"
m_matches = [c for c in d.columns if str(c).lower().startswith(mediator_prefix.lower())]

if len(m_matches) == 0:
    raise KeyError(f"No column starts with '{mediator_prefix}'")
if len(m_matches) > 1:
    raise ValueError("Multiple possible mediator columns:\n  - " + "\n  - ".join(m_matches))

mediator_col = m_matches[0]
d["trust_ai_answers"] = pd.to_numeric(
    d[mediator_col].astype(str).str.replace(",", ".", regex=False).str.strip(),
    errors="coerce"
)

print(f"Mediator column used: {mediator_col}")
print("Mediator descriptives:")
print(d["trust_ai_answers"].describe().round(3).to_string())

# -----------------------------
# 3) Build outcome: side_effect_any (exact prior coding)
# -----------------------------
side_effect_cols = {
    "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner": "G2Q5.1 Social alienation",
    "ki_ai_dependency": "G2Q6 AI dependency",
    "ki_decision_difficulty": "G2Q7 Decision difficulty",
}

SOCIAL_YES_NUM = {-1, -2, -3}
YES_VALUES_TEXT = {
    "trifft ein wenig zu",
    "trifft teilweise zu",
    "trifft völlig zu",
    "trifft voellig zu",
}

def to_yes_no(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    if col == "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner":
        num = pd.to_numeric(s, errors="coerce")
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0

    if s.lower() in YES_VALUES_TEXT:
        return 1
    return 0

for col in side_effect_cols:
    ycol = f"{col}_yes"
    if col in d.columns:
        d[ycol] = d[col].apply(lambda x: to_yes_no(x, col))
    else:
        d[ycol] = np.nan

ycols = [f"{c}_yes" for c in side_effect_cols]
answered_any = d[ycols].notna().any(axis=1)
any_yes = d[ycols].eq(1).any(axis=1)
d["side_effect_any"] = np.where(~answered_any, np.nan, np.where(any_yes, 1, 0))

print("\nPerson-level outcome side_effect_any:")
print(d["side_effect_any"].value_counts(dropna=False).to_string())

# -----------------------------
# 4) Build clinical variables (same coding as prior models)
# -----------------------------
phq4_cols = ["PHQ4_1", "PHQ4_2", "PHQ4_3", "PHQ4_4"]
for c in phq4_cols:
    if c not in d.columns:
        raise KeyError(f"Missing PHQ-4 column: {c}")

phq4_raw = d[phq4_cols].apply(pd.to_numeric, errors="coerce")
phq4_recoded = phq4_raw.replace({1: 0, 2: 1, 3: 2, 4: 3})
phq4_recoded = phq4_recoded.where(phq4_raw.isin([1, 2, 3, 4]), np.nan)
mask_phq4_complete = phq4_recoded.notna().all(axis=1)
d["PHQ4_sum"] = phq4_recoded.sum(axis=1)
d.loc[~mask_phq4_complete, "PHQ4_sum"] = np.nan

if "Behandlung_dich" not in d.columns:
    raise KeyError("Missing column: Behandlung_dich")
bh = d["Behandlung_dich"].astype(str).str.strip().str.lower()
d["behandlung"] = np.where(bh == "a", 1, np.where(bh == "b", 0, np.nan))

if "Diagnose_selbst_dich" not in d.columns:
    raise KeyError("Missing column: Diagnose_selbst_dich")
ds = d["Diagnose_selbst_dich"].astype(str).str.strip().str.lower()
d["diagnose_selbst"] = np.where(ds == "a", 1, np.where(ds == "b", 0, np.nan))

if "EQ5D5L_3" not in d.columns:
    raise KeyError("Missing column: EQ5D5L_3")
eq_map = {"a": 1, "b": 2, "c": 3, "d": 4, "e": 5}
d["eq5d_daily"] = d["EQ5D5L_3"].astype(str).str.strip().str.lower().map(eq_map)

# -----------------------------
# 5) Common complete-case sample
# -----------------------------
X = [
    "age_2026", "male", "edu_total_years", "employed",
    "PHQ4_sum", "behandlung", "diagnose_selbst", "eq5d_daily"
]
needed = ["trust_ai_answers", "side_effect_any"] + X

for c in ["age_2026", "edu_total_years", "trust_ai_answers", "PHQ4_sum", "eq5d_daily"]:
    d[c] = pd.to_numeric(d[c], errors="coerce")

dfm = d.dropna(subset=needed).copy()

for c in ["male", "employed", "behandlung", "diagnose_selbst", "side_effect_any"]:
    dfm[c] = pd.to_numeric(dfm[c], errors="coerce").astype(int)

print(f"\nCommon complete-case N: {len(dfm):,}")

# -----------------------------
# 6) Main mediation models
# -----------------------------
x_formula = " + ".join(X)

# MATCHED CHANGE: robust HC3 for path a
model_a = smf.ols(f"trust_ai_answers ~ {x_formula}", data=dfm).fit(cov_type="HC3")
model_c = smf.logit(f"side_effect_any ~ {x_formula}", data=dfm).fit(disp=0)
model_cp = smf.logit(f"side_effect_any ~ {x_formula} + trust_ai_answers", data=dfm).fit(disp=0)

print("\n=== Path a: trust_ai_answers ~ X (OLS, HC3 robust SE) ===")
print(model_a.summary())
print("\n=== Total effect c: side_effect_any ~ X (logit) ===")
print(model_c.summary())
print("\n=== Direct effect c' (+ b): side_effect_any ~ X + trust_ai_answers (logit) ===")
print(model_cp.summary())

# c vs c' comparison
rows = []
for p in X:
    c = model_c.params.get(p, np.nan)
    cp = model_cp.params.get(p, np.nan)
    rows.append({
        "predictor": p,
        "coef_c_total": c,
        "OR_c_total": np.exp(c),
        "p_c_total": model_c.pvalues.get(p, np.nan),
        "coef_cprime_direct": cp,
        "OR_cprime_direct": np.exp(cp),
        "p_cprime_direct": model_cp.pvalues.get(p, np.nan),
        "delta_coef_c_minus_cprime": c - cp
    })

rows.append({
    "predictor": "trust_ai_answers (b path)",
    "coef_c_total": np.nan,
    "OR_c_total": np.nan,
    "p_c_total": np.nan,
    "coef_cprime_direct": model_cp.params.get("trust_ai_answers", np.nan),
    "OR_cprime_direct": np.exp(model_cp.params.get("trust_ai_answers", np.nan)),
    "p_cprime_direct": model_cp.pvalues.get("trust_ai_answers", np.nan),
    "delta_coef_c_minus_cprime": np.nan
})

compare = pd.DataFrame(rows)
print("\n=== Mediation-style comparison ===")
print(compare.round(4).to_string(index=False))

# -----------------------------
# 7) Bootstrap indirect effects (a*b)
# -----------------------------
B = 2000
SEED = 42
rng = np.random.default_rng(SEED)

ab_store = {p: [] for p in X}
ok = 0
fail = 0

for _ in range(B):
    idx = rng.integers(0, len(dfm), len(dfm))
    bs = dfm.iloc[idx]

    try:
        # OLS for a-path in each bootstrap sample
        a_fit = smf.ols(f"trust_ai_answers ~ {x_formula}", data=bs).fit()
        cp_fit = smf.logit(f"side_effect_any ~ {x_formula} + trust_ai_answers", data=bs).fit(disp=0)

        b_coef = cp_fit.params.get("trust_ai_answers", np.nan)

        for p in X:
            a_coef = a_fit.params.get(p, np.nan)
            ab_store[p].append(a_coef * b_coef)

        ok += 1
    except Exception:
        for p in X:
            ab_store[p].append(np.nan)
        fail += 1

boot_rows = []
for p in X:
    vals = np.array(ab_store[p], dtype=float)
    vals = vals[~np.isnan(vals)]
    if len(vals) == 0:
        boot_rows.append({
            "predictor": p,
            "ab_mean": np.nan,
            "ab_median": np.nan,
            "ab_ci_2.5": np.nan,
            "ab_ci_97.5": np.nan,
            "n_boot_ok_for_predictor": 0
        })
    else:
        boot_rows.append({
            "predictor": p,
            "ab_mean": vals.mean(),
            "ab_median": np.median(vals),
            "ab_ci_2.5": np.quantile(vals, 0.025),
            "ab_ci_97.5": np.quantile(vals, 0.975),
            "n_boot_ok_for_predictor": len(vals)
        })

boot_table = pd.DataFrame(boot_rows)

print("\n=== Bootstrap summary ===")
print(f"Requested draws: {B}")
print(f"Successful draws: {ok}")
print(f"Failed draws: {fail}")

print("\n=== Indirect effects (a*b) with bootstrap 95% CI ===")
print(boot_table.round(4).to_string(index=False))
print("\nInterpretation hint: CI excluding 0 suggests an indirect effect for that predictor.")

## Exploratory models: loneliness

These analyses explore associations between loneliness and (a) mental health-related support use and (b) side effects.


### Side effects (binary): sociodemographics + loneliness


In [30]:
# =========================================================
# Logistic regression:
# side_effect_any ~ sociodemographics + loneliness_sum_2_7
#
# Predictors:
#   age_2026, male, edu_total_years, employed, loneliness_sum_2_7
# Outcome:
#   side_effect_any (1 = any side effect, 0 = none)
#
# Base:
#   AI users only + complete sociodemographics + complete loneliness items
# =========================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# -----------------------------
# 1) Base dataframe
# -----------------------------
if "df_base_predictors" in globals():
    d0 = df_base_predictors.copy()
elif "df_reg" in globals():
    d0 = df_reg.copy()
elif "df" in globals():
    d0 = df.copy()
else:
    raise NameError("No dataframe found. Expected one of: df_base_predictors, df_reg, df")

# -----------------------------
# 2) Required columns
# -----------------------------
soc_cols = ["age_2026", "male", "edu_total_years", "employed"]
lon_cols = ["loneliness_2", "loneliness_3", "Loneliness_4", "loneliness_5", "loneliness_6", "loneliness_7"]

side_effect_cols = {
    "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner": "G2Q5.1 Social alienation",
    "ki_ai_dependency": "G2Q6 AI dependency",
    "ki_decision_difficulty": "G2Q7 Decision difficulty",
}

needed_cols = ["ai_use"] + soc_cols + lon_cols + list(side_effect_cols.keys())
missing = [c for c in needed_cols if c not in d0.columns]
if missing:
    raise KeyError("Missing required columns:\n  - " + "\n  - ".join(missing))

# -----------------------------
# 3) Restrict to AI users + complete sociodemographics
# -----------------------------
d = d0.loc[d0["ai_use"] == 1].copy()
d = d.dropna(subset=soc_cols).copy()

print(f"AI users + complete sociodemographics: {len(d):,}")

# -----------------------------
# 4) Build loneliness scale (numeric coding 1..4)
#   1 = Trifft ganz genau zu
#   2 = Trifft eher zu
#   3 = Trifft eher nicht zu
#   4 = Trifft gar nicht zu
#
#   Items 2/4/6: 1/2 -> 1 ; 3/4 -> 0
#   Items 3/5/7: 3/4 -> 1 ; 1/2 -> 0
# -----------------------------
def to_int_1_4(v):
    x = pd.to_numeric(v, errors="coerce")
    if pd.isna(x):
        return np.nan
    x = int(x)
    return x if x in {1, 2, 3, 4} else np.nan

def code_forward_num(v):  # for 2,4,6
    x = to_int_1_4(v)
    if pd.isna(x):
        return np.nan
    return 1 if x in {1, 2} else 0

def code_reverse_num(v):  # for 3,5,7
    x = to_int_1_4(v)
    if pd.isna(x):
        return np.nan
    return 1 if x in {3, 4} else 0

d["loneliness_2_bin"] = d["loneliness_2"].apply(code_forward_num)
d["loneliness_3_bin"] = d["loneliness_3"].apply(code_reverse_num)
d["loneliness_4_bin"] = d["Loneliness_4"].apply(code_forward_num)  # capital L
d["loneliness_5_bin"] = d["loneliness_5"].apply(code_reverse_num)
d["loneliness_6_bin"] = d["loneliness_6"].apply(code_forward_num)
d["loneliness_7_bin"] = d["loneliness_7"].apply(code_reverse_num)

lon_bin_cols = [
    "loneliness_2_bin", "loneliness_3_bin", "loneliness_4_bin",
    "loneliness_5_bin", "loneliness_6_bin", "loneliness_7_bin"
]

# Keep complete loneliness only
d = d.dropna(subset=lon_bin_cols).copy()
d["loneliness_sum_2_7"] = d[lon_bin_cols].sum(axis=1)   # 0..6
d["loneliness_mean_2_7"] = d[lon_bin_cols].mean(axis=1) # 0..1

print(f"After complete loneliness filter: {len(d):,}")

# -----------------------------
# 5) Build side_effect_any (same prior coding)
# -----------------------------
SOCIAL_YES_NUM = {-1, -2, -3}
YES_VALUES_TEXT = {"trifft ein wenig zu", "trifft teilweise zu", "trifft völlig zu", "trifft voellig zu"}

def to_yes_no(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    if col == "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner":
        num = pd.to_numeric(s, errors="coerce")
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0

    return 1 if s.lower() in YES_VALUES_TEXT else 0

for col in side_effect_cols:
    d[f"{col}_yes"] = d[col].apply(lambda x: to_yes_no(x, col))

ycols = [f"{c}_yes" for c in side_effect_cols]
answered_any = d[ycols].notna().any(axis=1)
any_yes = d[ycols].eq(1).any(axis=1)
d["side_effect_any"] = np.where(~answered_any, np.nan, np.where(any_yes, 1, 0))

print("\nside_effect_any distribution (before complete-case regression filter):")
print(d["side_effect_any"].value_counts(dropna=False).to_string())

# -----------------------------
# 6) Regression sample (complete cases)
# -----------------------------
preds = ["age_2026", "male", "edu_total_years", "employed", "loneliness_sum_2_7"]
required = ["side_effect_any"] + preds

for c in ["age_2026", "edu_total_years", "loneliness_sum_2_7"]:
    d[c] = pd.to_numeric(d[c], errors="coerce")

dfm = d.dropna(subset=required).copy()
dfm["side_effect_any"] = dfm["side_effect_any"].astype(int)
dfm["male"] = pd.to_numeric(dfm["male"], errors="coerce").astype(int)
dfm["employed"] = pd.to_numeric(dfm["employed"], errors="coerce").astype(int)

print(f"\nFinal complete-case N for model: {len(dfm):,}")

# -----------------------------
# 7) Logistic regression
# -----------------------------
formula = "side_effect_any ~ age_2026 + male + edu_total_years + employed + loneliness_sum_2_7"
model = smf.logit(formula, data=dfm).fit(disp=0)

print("\n=== Logistic regression ===")
print(f"Model: {formula}")
print(model.summary())

# Odds-ratio table
or_table = pd.DataFrame({
    "predictor": model.params.index,
    "coef": model.params.values,
    "OR": np.exp(model.params.values),
    "CI_2.5": np.exp(model.conf_int()[0].values),
    "CI_97.5": np.exp(model.conf_int()[1].values),
    "p_value": model.pvalues.values
})

print("\n=== Odds ratios (95% CI) ===")
print(or_table.round(4).to_string(index=False))

AI users + complete sociodemographics: 13,664
After complete loneliness filter: 1,788

side_effect_any distribution (before complete-case regression filter):
side_effect_any
0.0    1611
1.0     162
NaN      15

Final complete-case N for model: 1,773

=== Logistic regression ===
Model: side_effect_any ~ age_2026 + male + edu_total_years + employed + loneliness_sum_2_7
                           Logit Regression Results                           
Dep. Variable:        side_effect_any   No. Observations:                 1773
Model:                          Logit   Df Residuals:                     1767
Method:                           MLE   Df Model:                            5
Date:                Wed, 15 Apr 2026   Pseudo R-squ.:                 0.05363
Time:                        21:29:28   Log-Likelihood:                -512.93
converged:                       True   LL-Null:                       -542.00
Covariance Type:            nonrobust   LLR p-value:                 2.947e-1

### Mediation: side effects (loneliness-related pathway)


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# =========================================================
# MEDIATION ANALYSIS (with bootstrap)
# X = age_2026, male, edu_total_years, employed
# M = loneliness_sum_2_7 (0..6)
# Y = side_effect_any (0/1)
# Base: AI users only
# =========================================================

# -----------------------------
# 1) Base sample
# -----------------------------
if "df_base_predictors" in globals():
    d0 = df_base_predictors.copy()
elif "df_reg" in globals():
    d0 = df_reg.copy()
elif "df" in globals():
    d0 = df.copy()
else:
    raise NameError("No dataframe found. Expected one of: df_base_predictors, df_reg, df")

soc_cols = ["age_2026", "male", "edu_total_years", "employed"]
lon_cols = ["loneliness_2", "loneliness_3", "Loneliness_4", "loneliness_5", "loneliness_6", "loneliness_7"]

side_effect_cols = {
    "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner": "G2Q5.1 Social alienation",
    "ki_ai_dependency": "G2Q6 AI dependency",
    "ki_decision_difficulty": "G2Q7 Decision difficulty",
}

needed_cols = ["ai_use"] + soc_cols + lon_cols + list(side_effect_cols.keys())
missing = [c for c in needed_cols if c not in d0.columns]
if missing:
    raise KeyError("Missing required columns:\n  - " + "\n  - ".join(missing))

d = d0.loc[d0["ai_use"] == 1].copy()
d = d.dropna(subset=soc_cols).copy()
print(f"AI users + complete sociodemographics: {len(d):,}")

# -----------------------------
# 2) Build mediator: loneliness_sum_2_7
# Numeric coding in data:
# 1=Trifft ganz genau zu, 2=Trifft eher zu, 3=Trifft eher nicht zu, 4=Trifft gar nicht zu
# Items 2/4/6: 1/2 -> 1 ; 3/4 -> 0
# Items 3/5/7: 3/4 -> 1 ; 1/2 -> 0
# -----------------------------
def to_int_1_4(v):
    x = pd.to_numeric(v, errors="coerce")
    if pd.isna(x):
        return np.nan
    x = int(x)
    return x if x in {1, 2, 3, 4} else np.nan

def code_forward_num(v):
    x = to_int_1_4(v)
    if pd.isna(x):
        return np.nan
    return 1 if x in {1, 2} else 0

def code_reverse_num(v):
    x = to_int_1_4(v)
    if pd.isna(x):
        return np.nan
    return 1 if x in {3, 4} else 0

d["loneliness_2_bin"] = d["loneliness_2"].apply(code_forward_num)
d["loneliness_3_bin"] = d["loneliness_3"].apply(code_reverse_num)
d["loneliness_4_bin"] = d["Loneliness_4"].apply(code_forward_num)  # capital L
d["loneliness_5_bin"] = d["loneliness_5"].apply(code_reverse_num)
d["loneliness_6_bin"] = d["loneliness_6"].apply(code_forward_num)
d["loneliness_7_bin"] = d["loneliness_7"].apply(code_reverse_num)

lon_bin_cols = [
    "loneliness_2_bin", "loneliness_3_bin", "loneliness_4_bin",
    "loneliness_5_bin", "loneliness_6_bin", "loneliness_7_bin"
]

# keep complete loneliness info for mediator
d = d.dropna(subset=lon_bin_cols).copy()
d["loneliness_sum_2_7"] = d[lon_bin_cols].sum(axis=1)   # 0..6
d["loneliness_mean_2_7"] = d[lon_bin_cols].mean(axis=1) # 0..1 (optional)

print(f"After complete loneliness filter: {len(d):,}")
print("\nMediator descriptives (loneliness_sum_2_7):")
print(d["loneliness_sum_2_7"].describe().round(3).to_string())

# -----------------------------
# 3) Build outcome: side_effect_any
# -----------------------------
SOCIAL_YES_NUM = {-1, -2, -3}
YES_VALUES_TEXT = {"trifft ein wenig zu", "trifft teilweise zu", "trifft völlig zu", "trifft voellig zu"}

def to_yes_no(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    if col == "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner":
        num = pd.to_numeric(s, errors="coerce")
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0

    return 1 if s.lower() in YES_VALUES_TEXT else 0

for col in side_effect_cols:
    d[f"{col}_yes"] = d[col].apply(lambda x: to_yes_no(x, col))

ycols = [f"{c}_yes" for c in side_effect_cols]
answered_any = d[ycols].notna().any(axis=1)
any_yes = d[ycols].eq(1).any(axis=1)
d["side_effect_any"] = np.where(~answered_any, np.nan, np.where(any_yes, 1, 0))

print("\nPerson-level side_effect_any:")
print(d["side_effect_any"].value_counts(dropna=False).to_string())

# -----------------------------
# 4) Common complete-case sample
# -----------------------------
X = ["age_2026", "male", "edu_total_years", "employed"]
needed = ["loneliness_sum_2_7", "side_effect_any"] + X

for c in ["age_2026", "edu_total_years", "loneliness_sum_2_7"]:
    d[c] = pd.to_numeric(d[c], errors="coerce")

dfm = d.dropna(subset=needed).copy()
dfm["side_effect_any"] = dfm["side_effect_any"].astype(int)
dfm["male"] = pd.to_numeric(dfm["male"], errors="coerce").astype(int)
dfm["employed"] = pd.to_numeric(dfm["employed"], errors="coerce").astype(int)

print(f"\nCommon complete-case N: {len(dfm):,}")

# -----------------------------
# 5) Mediation models
# -----------------------------
# Path a: M ~ X (OLS, HC3)
model_a = smf.ols(
    "loneliness_sum_2_7 ~ age_2026 + male + edu_total_years + employed",
    data=dfm
).fit(cov_type="HC3")

# Total effect c: Y ~ X (logit)
model_c = smf.logit(
    "side_effect_any ~ age_2026 + male + edu_total_years + employed",
    data=dfm
).fit(disp=0)

# Direct effect c' (+ b): Y ~ X + M (logit)
model_cp = smf.logit(
    "side_effect_any ~ age_2026 + male + edu_total_years + employed + loneliness_sum_2_7",
    data=dfm
).fit(disp=0)

print("\n=== Path a: loneliness_sum_2_7 ~ X (OLS, HC3) ===")
print(model_a.summary())

print("\n=== Total effect c: side_effect_any ~ X (logit) ===")
print(model_c.summary())

print("\n=== Direct effect c' (+ b): side_effect_any ~ X + loneliness_sum_2_7 (logit) ===")
print(model_cp.summary())

# c vs c' comparison table
rows = []
for p in X:
    c = model_c.params.get(p, np.nan)
    cp = model_cp.params.get(p, np.nan)
    rows.append({
        "predictor": p,
        "coef_c_total": c,
        "OR_c_total": np.exp(c),
        "p_c_total": model_c.pvalues.get(p, np.nan),
        "coef_cprime_direct": cp,
        "OR_cprime_direct": np.exp(cp),
        "p_cprime_direct": model_cp.pvalues.get(p, np.nan),
        "delta_coef_c_minus_cprime": c - cp
    })

rows.append({
    "predictor": "loneliness_sum_2_7 (b path)",
    "coef_c_total": np.nan,
    "OR_c_total": np.nan,
    "p_c_total": np.nan,
    "coef_cprime_direct": model_cp.params.get("loneliness_sum_2_7", np.nan),
    "OR_cprime_direct": np.exp(model_cp.params.get("loneliness_sum_2_7", np.nan)),
    "p_cprime_direct": model_cp.pvalues.get("loneliness_sum_2_7", np.nan),
    "delta_coef_c_minus_cprime": np.nan
})

compare = pd.DataFrame(rows)
print("\n=== Mediation-style comparison ===")

### Mental health-related use (binary): sociodemographics + loneliness


In [32]:
# =========================================================
# Logistic regression:
# emo_use ~ sociodemographics + loneliness_sum_2_7
#
# Predictors:
#   age_2026, male, edu_total_years, employed, loneliness_sum_2_7
# Outcome:
#   emo_use (1 = emotional AI use, 0 = no emotional AI use)
#
# Base:
#   AI users only + complete sociodemographics + complete loneliness items
#
# NOTE: emo_use includes Q4.13 (Sonstiges) IF available.
# =========================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# -----------------------------
# 1) Base dataframe
# -----------------------------
if "df_base_predictors" in globals():
    d0 = df_base_predictors.copy()
elif "df_reg" in globals():
    d0 = df_reg.copy()
elif "df" in globals():
    d0 = df.copy()
else:
    raise NameError("No dataframe found. Expected one of: df_base_predictors, df_reg, df")

# -----------------------------
# 2) Required columns
# -----------------------------
soc_cols = ["age_2026", "male", "edu_total_years", "employed"]
lon_cols = ["loneliness_2", "loneliness_3", "Loneliness_4", "loneliness_5", "loneliness_6", "loneliness_7"]

emo_cols_4_12 = [
    "ki_bei_aengsten_und_sorgen",
    "ki_bei_selbstzweifeln_oder_niedergeschlagenheit",
    "ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we",
    "ki_bei_zwischenmenschlichen_konflikten",
    "ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan",
    "ki_bei_trauer_oder_verlust",
    "ki_wenn_grosse_lebensveraenderungen_anstehen",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5",
]
emo_col_13 = "ki_fuer_sonstige_gesundheits_psychische_oder_emotionale"

emo_cols = emo_cols_4_12 + ([emo_col_13] if emo_col_13 in d0.columns else [])
if emo_col_13 not in d0.columns:
    print("Warning: Q4.13 (Sonstiges) column not found; emo_use will be based on Q4.4–Q4.12 only.")

needed_cols = ["ai_use"] + soc_cols + lon_cols + emo_cols
missing = [c for c in needed_cols if c not in d0.columns]
if missing:
    raise KeyError("Missing required columns:\n  - " + "\n  - ".join(missing))

# -----------------------------
# 3) Restrict to AI users + complete sociodemographics
# -----------------------------
d = d0.loc[d0["ai_use"] == 1].copy()
d = d.dropna(subset=soc_cols).copy()
print(f"AI users + complete sociodemographics: {len(d):,}")

# -----------------------------
# 4) Build loneliness scale (same coding as your previous model)
# -----------------------------
def to_int_1_4(v):
    x = pd.to_numeric(v, errors="coerce")
    if pd.isna(x):
        return np.nan
    x = int(x)
    return x if x in {1, 2, 3, 4} else np.nan

def code_forward_num(v):  # for items 2,4,6
    x = to_int_1_4(v)
    if pd.isna(x):
        return np.nan
    return 1 if x in {1, 2} else 0

def code_reverse_num(v):  # for items 3,5,7
    x = to_int_1_4(v)
    if pd.isna(x):
        return np.nan
    return 1 if x in {3, 4} else 0

d["loneliness_2_bin"] = d["loneliness_2"].apply(code_forward_num)
d["loneliness_3_bin"] = d["loneliness_3"].apply(code_reverse_num)
d["loneliness_4_bin"] = d["Loneliness_4"].apply(code_forward_num)  # capital L
d["loneliness_5_bin"] = d["loneliness_5"].apply(code_reverse_num)
d["loneliness_6_bin"] = d["loneliness_6"].apply(code_forward_num)
d["loneliness_7_bin"] = d["loneliness_7"].apply(code_reverse_num)

lon_bin_cols = [
    "loneliness_2_bin", "loneliness_3_bin", "loneliness_4_bin",
    "loneliness_5_bin", "loneliness_6_bin", "loneliness_7_bin"
]

d = d.dropna(subset=lon_bin_cols).copy()
d["loneliness_sum_2_7"] = d[lon_bin_cols].sum(axis=1)   # 0..6
print(f"After complete loneliness filter: {len(d):,}")

# -----------------------------
# 5) Build outcome: emo_use (now incl. Q4.13 if present)
# -----------------------------
non_use_answers = {
    "hierfür nutze ich keine ki",
    "(bisher) kein bedarf, würde ki aber dafür nutzen",
}

def norm_str(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    return "" if s in {"", "nan"} else s

emo_vals = []
for _, row in d[emo_cols].iterrows():
    vals = [norm_str(row[c]) for c in emo_cols]
    answered = [v for v in vals if v != ""]
    if len(answered) == 0:
        emo_vals.append(np.nan)   # all missing -> missing outcome
    elif any(v not in non_use_answers for v in answered):
        emo_vals.append(1)        # any substantive emotional use
    else:
        emo_vals.append(0)        # only non-use responses

d["emo_use"] = emo_vals

print("\nemo_use distribution (before complete-case regression filter):")
print(d["emo_use"].value_counts(dropna=False).to_string())

# -----------------------------
# 6) Regression sample (complete cases)
# -----------------------------
preds = ["age_2026", "male", "edu_total_years", "employed", "loneliness_sum_2_7"]
required = ["emo_use"] + preds

for c in ["age_2026", "edu_total_years", "loneliness_sum_2_7"]:
    d[c] = pd.to_numeric(d[c], errors="coerce")

dfm = d.dropna(subset=required).copy()
dfm["emo_use"] = dfm["emo_use"].astype(int)
dfm["male"] = pd.to_numeric(dfm["male"], errors="coerce").astype(int)
dfm["employed"] = pd.to_numeric(dfm["employed"], errors="coerce").astype(int)

print(f"\nFinal complete-case N for model: {len(dfm):,}")

# -----------------------------
# 7) Logistic regression
# -----------------------------
formula = "emo_use ~ age_2026 + male + edu_total_years + employed + loneliness_sum_2_7"
model = smf.logit(formula, data=dfm).fit(disp=0)

print("\n=== Logistic regression ===")
print(f"Model: {formula}")
print(model.summary())

# Odds-ratio table
or_table = pd.DataFrame({
    "predictor": model.params.index,
    "coef": model.params.values,
    "OR": np.exp(model.params.values),
    "CI_2.5": np.exp(model.conf_int()[0].values),
    "CI_97.5": np.exp(model.conf_int()[1].values),
    "p_value": model.pvalues.values
})

print("\n=== Odds ratios (95% CI) ===")
print(or_table.round(4).to_string(index=False))

AI users + complete sociodemographics: 13,664
After complete loneliness filter: 1,788

emo_use distribution (before complete-case regression filter):
emo_use
0    1238
1     550

Final complete-case N for model: 1,788

=== Logistic regression ===
Model: emo_use ~ age_2026 + male + edu_total_years + employed + loneliness_sum_2_7
                           Logit Regression Results                           
Dep. Variable:                emo_use   No. Observations:                 1788
Model:                          Logit   Df Residuals:                     1782
Method:                           MLE   Df Model:                            5
Date:                Wed, 15 Apr 2026   Pseudo R-squ.:                 0.03358
Time:                        21:29:29   Log-Likelihood:                -1066.4
converged:                       True   LL-Null:                       -1103.5
Covariance Type:            nonrobust   LLR p-value:                 1.427e-14
                         coef    std e

### Predictor: intensity of mental health use (full eligible sample)


In [34]:
# =========================================================
# Side-effect logistic — ALIGNED with extended emotional intensity (Q4.4–Q4.13)
# - Same map_intensity, non_use, grief/life max, int_cols
# - Sample: AI users with has_emotional_use & known_intensity (all int_cols non-NaN)
# - Predictor: emo_use_intensity_sum = sum(int_cols) — same as extended block
# =========================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

if "df_base_predictors" in globals():
    d0 = df_base_predictors.copy()
elif "df_reg" in globals():
    d0 = df_reg.copy()
elif "df" in globals():
    d0 = df.copy()
else:
    raise NameError("Need df_base_predictors, df_reg, or df")

d = d0.loc[d0["ai_use"] == 1].copy()

# --- Q4.4 – Q4.13 (same list as extended emotional block) ---
purpose_cols = [
    "ki_bei_aengsten_und_sorgen",
    "ki_bei_selbstzweifeln_oder_niedergeschlagenheit",
    "ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we",
    "ki_bei_zwischenmenschlichen_konflikten",
    "ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan",
    "ki_bei_trauer_oder_verlust",
    "ki_wenn_grosse_lebensveraenderungen_anstehen",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5",
    "ki_fuer_sonstige_gesundheits_psychische_oder_emotionale",
]
purpose_cols = [c for c in purpose_cols if c in d.columns]

col_grief = "ki_bei_trauer_oder_verlust"
col_life = "ki_wenn_grosse_lebensveraenderungen_anstehen"

# --- IDENTICAL to extended emotional block ---
def norm_txt(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    if s == "nan":
        return ""
    s = s.replace("ä", "ae").replace("ö", "oe").replace("ü", "ue").replace("ß", "ss")
    return s

intensity_map = {
    "selten": 1,
    "manchmal": 2,
    "oft": 3,
    "immer": 4,
}

non_use_answers = {
    "hierfuer nutze ich keine ki",
    "(bisher) kein bedarf, wuerde ki aber dafuer nutzen",
}

def map_intensity(v):
    s = norm_txt(v)
    if s == "":
        return 0
    if s in non_use_answers:
        return 0
    if s in intensity_map:
        return intensity_map[s]
    for k, val in intensity_map.items():
        if k in s:
            return val
    return np.nan

for c in purpose_cols:
    d[f"{c}_int"] = d[c].apply(map_intensity)

col_grief_int = f"{col_grief}_int"
col_life_int = f"{col_life}_int"
combined_col = "grief_life_combined_int"

if col_grief in d.columns and col_life in d.columns:
    d[combined_col] = d[[col_grief_int, col_life_int]].max(axis=1)
else:
    raise KeyError("Need Q4.9 and Q4.10 for grief/life combine.")

all_int_cols = [f"{c}_int" for c in purpose_cols]
int_cols = [c for c in all_int_cols if c not in [col_grief_int, col_life_int]] + [combined_col]

# Same inclusion rules as extended emotional block
has_emotional_use = d[int_cols].apply(
    lambda r: any(pd.notna(v) and v > 0 for v in r), axis=1
)
known_intensity = d[int_cols].notna().all(axis=1)

d["emo_use_intensity_sum"] = d[int_cols].sum(axis=1)
d["mh_intensity_mean_4_13"] = d["emo_use_intensity_sum"] / len(int_cols)

d_align = d.loc[has_emotional_use & known_intensity].copy()

# --- Side effects (unchanged logic) ---
side_effect_cols = {
    "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner": "G2Q5.1",
    "ki_ai_dependency": "G2Q6",
    "ki_decision_difficulty": "G2Q7",
}
SOCIAL_YES_NUM = {-1, -2, -3}
YES_VALUES_TEXT = {"trifft ein wenig zu", "trifft teilweise zu", "trifft völlig zu", "trifft voellig zu"}

def to_yes_no(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return np.nan
    if col == "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner":
        num = pd.to_numeric(s, errors="coerce")
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0
    return 1 if s.lower() in YES_VALUES_TEXT else 0

for col in side_effect_cols:
    if col in d_align.columns:
        d_align[f"{col}_yes"] = d_align[col].apply(lambda x: to_yes_no(x, col))
    else:
        d_align[f"{col}_yes"] = np.nan

ycols = [f"{c}_yes" for c in side_effect_cols]
answered_any = d_align[ycols].notna().any(axis=1)
any_yes = d_align[ycols].eq(1).any(axis=1)
d_align["side_effect_any"] = np.where(~answered_any, np.nan, np.where(any_yes, 1, 0))

# Primary model: SUM predictor (matches extended emotional OLS outcome)
required = [
    "side_effect_any",
    "emo_use_intensity_sum",
    "age_2026",
    "male",
    "edu_total_years",
    "employed",
]
df_reg = d_align.dropna(subset=required).copy()

for c in ["age_2026", "edu_total_years", "emo_use_intensity_sum"]:
    df_reg[c] = pd.to_numeric(df_reg[c], errors="coerce")
df_reg["side_effect_any"] = df_reg["side_effect_any"].astype(int)
df_reg["male"] = pd.to_numeric(df_reg["male"], errors="coerce").astype(int)
df_reg["employed"] = pd.to_numeric(df_reg["employed"], errors="coerce").astype(int)
df_reg = df_reg.dropna(subset=required)

print(f"AI users, emotional use + known intensity on all components (aligned): {len(d_align):,}")
print(f"Complete-case N (side-effect model): {len(df_reg):,}")

formula = (
    "side_effect_any ~ emo_use_intensity_sum + age_2026 + male + edu_total_years + employed"
)
model = smf.logit(formula, data=df_reg).fit(disp=0)
print("\n", formula)
print(model.summary())

coef = model.params
conf = model.conf_int()
conf.columns = ["lo", "hi"]
report = pd.DataFrame({
    "OR": np.exp(coef),
    "CI_lo": np.exp(conf["lo"]),
    "CI_hi": np.exp(conf["hi"]),
    "p": model.pvalues,
})
print("\nOdds ratios:\n", report.round(4).to_string())

# Optional: same model on MEAN scale (OR = per +1 on 0–4 average item); coef ≈ 9× sum coef
formula_mean = (
    "side_effect_any ~ mh_intensity_mean_4_13 + age_2026 + male + edu_total_years + employed"
)
df_reg_m = d_align.dropna(
    subset=["side_effect_any", "mh_intensity_mean_4_13", "age_2026", "male", "edu_total_years", "employed"]
).copy()
for c in ["age_2026", "edu_total_years", "mh_intensity_mean_4_13"]:
    df_reg_m[c] = pd.to_numeric(df_reg_m[c], errors="coerce")
df_reg_m["side_effect_any"] = df_reg_m["side_effect_any"].astype(int)
df_reg_m["male"] = pd.to_numeric(df_reg_m["male"], errors="coerce").astype(int)
df_reg_m["employed"] = pd.to_numeric(df_reg_m["employed"], errors="coerce").astype(int)
df_reg_m = df_reg_m.dropna(
    subset=["side_effect_any", "mh_intensity_mean_4_13", "age_2026", "male", "edu_total_years", "employed"]
)
mod_mean = smf.logit(formula_mean, data=df_reg_m).fit(disp=0)
print("\n(Equivalent scaling) Mean-based model:\n", formula_mean)
print(mod_mean.summary())

AI users, emotional use + known intensity on all components (aligned): 5,097
Complete-case N (side-effect model): 4,525

 side_effect_any ~ emo_use_intensity_sum + age_2026 + male + edu_total_years + employed
                           Logit Regression Results                           
Dep. Variable:        side_effect_any   No. Observations:                 4525
Model:                          Logit   Df Residuals:                     4519
Method:                           MLE   Df Model:                            5
Date:                Wed, 15 Apr 2026   Pseudo R-squ.:                 0.08962
Time:                        21:29:34   Log-Likelihood:                -2218.5
converged:                       True   LL-Null:                       -2436.9
Covariance Type:            nonrobust   LLR p-value:                 3.452e-92
                            coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------

### Predictor: intensity of mental health use 
### Outcome: intensity of side effects


In [35]:
import re
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# ---------- Choose input df ----------
if "df_base_predictors" in globals():
    d0 = df_base_predictors.copy()
elif "df" in globals():
    d0 = df.copy()
else:
    raise NameError("Need df_base_predictors or df")

# AI users only
d = d0.loc[d0["ai_use"] == 1].copy()

# =========================================================
# (A) Emotional use intensity (Q4.4–Q4.13) -> emo_use_intensity_sum
# =========================================================
purpose_cols = [
    "ki_bei_aengsten_und_sorgen",
    "ki_bei_selbstzweifeln_oder_niedergeschlagenheit",
    "ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we",
    "ki_bei_zwischenmenschlichen_konflikten",
    "ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan",
    "ki_bei_trauer_oder_verlust",
    "ki_wenn_grosse_lebensveraenderungen_anstehen",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5",
    "ki_fuer_sonstige_gesundheits_psychische_oder_emotionale",
]
purpose_cols = [c for c in purpose_cols if c in d.columns]

col_grief = "ki_bei_trauer_oder_verlust"
col_life = "ki_wenn_grosse_lebensveraenderungen_anstehen"

def norm_txt(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    if s == "nan":
        return ""
    return s.replace("ä", "ae").replace("ö", "oe").replace("ü", "ue").replace("ß", "ss")

intensity_map = {"selten": 1, "manchmal": 2, "oft": 3, "immer": 4}
non_use_answers = {
    "hierfuer nutze ich keine ki",
    "(bisher) kein bedarf, wuerde ki aber dafuer nutzen",
}

def map_intensity(v):
    s = norm_txt(v)
    if s == "" or s in non_use_answers:
        return 0
    if s in intensity_map:
        return intensity_map[s]
    for k, val in intensity_map.items():
        if k in s:
            return val
    return np.nan  # unknown/unmapped

for c in purpose_cols:
    d[f"{c}_int"] = d[c].apply(map_intensity)

if col_grief not in d.columns or col_life not in d.columns:
    raise KeyError("Need both grief (Q4.9) and life change (Q4.10) columns present.")

col_grief_int = f"{col_grief}_int"
col_life_int = f"{col_life}_int"
combined_col = "grief_life_combined_int"
d[combined_col] = d[[col_grief_int, col_life_int]].max(axis=1)

all_int_cols = [f"{c}_int" for c in purpose_cols]
int_cols = [c for c in all_int_cols if c not in [col_grief_int, col_life_int]] + [combined_col]

has_emotional_use = d[int_cols].apply(lambda r: any(pd.notna(v) and v > 0 for v in r), axis=1)
known_intensity = d[int_cols].notna().all(axis=1)

d["emo_use_intensity_sum"] = d[int_cols].sum(axis=1)

d_emo = d.loc[has_emotional_use & known_intensity, ["emo_use_intensity_sum"]].copy()
print(f"(A) Emotional intensity sample (AI users, has emotional use + known intensity): {len(d_emo):,}")

# =========================================================
# (B) Side-effect intensity (3 items) -> side_effect_intensity_mean
# =========================================================
col_social = "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner"  # G2Q5.1
col_dep = "ki_ai_dependency"                                         # G2Q6
col_dec = "ki_decision_difficulty"                                   # G2Q7

needed_se = [col_social, col_dep, col_dec]
missing_se = [c for c in needed_se if c not in d.columns]
if missing_se:
    raise KeyError(f"Missing side-effect column(s): {missing_se}")

def code_social(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip().lower()
    if s in ["", "nan"]:
        return np.nan
    m = re.search(r"[-+]?\d+", s)
    if not m:
        return np.nan
    n = int(m.group(0))
    if n in (-1, -2, -3):
        return abs(n)  # 1..3
    if n in (0, 1, 2, 3):
        return 0
    return np.nan

def code_likert_se(v):
    s = norm_txt(v)
    if s == "":
        return np.nan
    if s == "trifft gar nicht zu":
        return 0
    if s == "trifft ein wenig zu":
        return 1
    if s == "trifft teilweise zu":
        return 2
    if s == "trifft voellig zu":  # "völlig" becomes "voellig" via norm_txt
        return 3
    return np.nan

d["se_social_int"] = d[col_social].apply(code_social)
d["se_dep_int"] = d[col_dep].apply(code_likert_se)
d["se_dec_int"] = d[col_dec].apply(code_likert_se)

se_int_cols = ["se_social_int", "se_dep_int", "se_dec_int"]

answered_any_se = d[se_int_cols].notna().any(axis=1)
d_se = d.loc[answered_any_se].copy()

d_se["n_side_effects_experienced"] = (d_se[se_int_cols] > 0).sum(axis=1)
d_se = d_se.loc[d_se["n_side_effects_experienced"] > 0].copy()

d_se["side_effect_intensity_sum"] = d_se[se_int_cols].fillna(0).sum(axis=1)
d_se["side_effect_intensity_mean"] = d_se["side_effect_intensity_sum"] / d_se["n_side_effects_experienced"]

d_se_int = d_se[["side_effect_intensity_mean"]].copy()
print(f"(B) Side-effect intensity sample (AI users, answered any SE + >=1 experienced): {len(d_se_int):,}")

# =========================================================
# (C) Intersect + regression (avoid overlapping columns)
# =========================================================
work = (
    d.drop(columns=["emo_use_intensity_sum"])
     .join(d_emo, how="inner")
     .join(d_se_int, how="inner")
)

required = [
    "side_effect_intensity_mean",
    "emo_use_intensity_sum",
    "age_2026",
    "male",
    "edu_total_years",
    "employed",
]
df_reg = work.dropna(subset=required).copy()

# Type safety
for c in ["side_effect_intensity_mean", "emo_use_intensity_sum", "age_2026", "edu_total_years"]:
    df_reg[c] = pd.to_numeric(df_reg[c], errors="coerce")
df_reg["male"] = pd.to_numeric(df_reg["male"], errors="coerce")
df_reg["employed"] = pd.to_numeric(df_reg["employed"], errors="coerce")
df_reg = df_reg.dropna(subset=required)

df_reg["male"] = df_reg["male"].astype(int)
df_reg["employed"] = df_reg["employed"].astype(int)

print(f"(C) Final regression sample (intersection + complete-case): {len(df_reg):,}")
print("\nOutcome descriptives (side_effect_intensity_mean):")
print(df_reg["side_effect_intensity_mean"].describe().round(3).to_string())

if len(df_reg) < 10:
    print("\nToo few complete cases to fit model safely.")
else:
    preds = ["emo_use_intensity_sum", "age_2026", "male", "edu_total_years", "employed"]
    keep = [p for p in preds if df_reg[p].nunique(dropna=True) > 1]
    dropped = [p for p in preds if p not in keep]
    if dropped:
        print("\nDropped constant predictor(s): " + ", ".join(dropped))

    if not keep:
        print("\nAll predictors are constant; model not fit.")
    else:
        formula = "side_effect_intensity_mean ~ " + " + ".join(keep)
        print("\nModel:", formula)
        res = smf.ols(formula, data=df_reg).fit(cov_type="HC3")
        print(res.summary())

(A) Emotional intensity sample (AI users, has emotional use + known intensity): 5,097
(B) Side-effect intensity sample (AI users, answered any SE + >=1 experienced): 1,828
(C) Final regression sample (intersection + complete-case): 1,040

Outcome descriptives (side_effect_intensity_mean):
count    1040.000
mean        1.232
std         0.446
min         1.000
25%         1.000
50%         1.000
75%         1.333
max         3.000

Model: side_effect_intensity_mean ~ emo_use_intensity_sum + age_2026 + male + edu_total_years + employed
                                OLS Regression Results                                
Dep. Variable:     side_effect_intensity_mean   R-squared:                       0.024
Model:                                    OLS   Adj. R-squared:                  0.019
Method:                         Least Squares   F-statistic:                     4.787
Date:                        Wed, 15 Apr 2026   Prob (F-statistic):           0.000248
Time:                    